# Cross-Asset Event Analogue Dashboard

A fully configurable multi-event, multi-asset historical analogue framework.  
**Run All Cells** → tabbed dashboard appears in the last cell output (~60s on first run).

---

### How it works
1. **CONFIG section** defines your event universe, asset universe, and analogue scoring weights  
2. **DATA section** pulls and aligns all price history once  
3. **CHARTS section** builds reusable chart functions (pure, no side effects)  
4. **LIVE section** powers the real-time analogue matching engine  
5. **DASHBOARD section** wires everything into a tabbed ipywidgets interface  

### Making it your own
- Swap the event list for any shock type (equity crashes, rate hikes, pandemics, geopolitical)  
- Add/remove assets by editing `ASSETS` — the rest auto-adapts  
- Change analogue scoring weights in `ANALOGUE_WEIGHTS`  
- Pre-1985 or data-scarce events: use the `MANUAL_DATA` injection pattern (see §1.3)


---
## § 1 · CONFIG
> **The only section you need to edit** to repurpose the dashboard for any event universe or asset class.  
> Each cell is independent — re-run any one cell without re-running the others.  
> All downstream sections (§2–§6) read from the variables defined here.
>
> | Cell | Variables defined | Purpose |
> |------|------------------|---------|
> | §1.1 | `DASHBOARD_TITLE`, `FRED_API_KEY` | Identity & credentials |
> | §1.2 | `EVENTS`, `EVENT_TAGS`, `MACRO_CONTEXT`, `ALL_TAGS` | Event universe |
> | §1.3 | `MANUAL_DATA` | Pre-1985 / data-scarce events |
> | §1.4 | `ASSETS`, `CUSTOM_GROUPS`, `INVERT_OVERRIDE`, `FRED_PRICE_SERIES`, `USDCNH_FALLBACKS` | Asset universe |
> | §1.5 | `PRE_WINDOW_TD`, `POST_WINDOW_TD`, `FETCH_BUFFER`, `POIS`, `DAY0_ROLL`, `STEP_IN_*`, `VIX_TICKER` | Analysis windows |
> | §1.6 | `TRIGGER_ASSET`, `ANALOGUE_WEIGHTS`, `SIMILARITY_ASSET_POOL`, `LIVE_FRED_MAP` | Analogue engine |


In [ ]:
# §1.1 — DASHBOARD IDENTITY & API KEY
# ─────────────────────────────────────────────────────────────
# DASHBOARD_TITLE : displayed in the notebook header markdown (cell [00])
# FRED_API_KEY    : free key from https://fred.stlouisfed.org/docs/api/api_key.html
#                   Required for rates, credit spreads, and breakeven series.
#                   Set to None to skip FRED pulls (those columns will be absent).

DASHBOARD_TITLE = "Middle East & Global Shock — Cross-Asset Tracker"
FRED_API_KEY_NOTEBOOK = "your_key_here"  # optional fallback; prefer env var
_fred_env_key = (__import__("os").getenv("FRED_API_KEY") or "").strip()
_fred_nb_key = str(FRED_API_KEY_NOTEBOOK or "").strip()
if _fred_env_key:
    FRED_API_KEY = _fred_env_key
elif _fred_nb_key and _fred_nb_key.lower() not in {"none", "your_key_here"}:
    FRED_API_KEY = _fred_nb_key
else:
    FRED_API_KEY = None

def _fred_key_configured(key=None):
    _v = FRED_API_KEY if key is None else key
    _v = str(_v or "").strip()
    return bool(_v) and _v.lower() not in {"none", "your_key_here"}

_FRED_KEY_SOURCE = (
    "environment variable" if _fred_env_key else
    "notebook fallback" if FRED_API_KEY else
    "disabled"
)
print(f"FRED key source: {_FRED_KEY_SOURCE}")

# ── FRED pull settings ────────────────────────────────────────
# FRED is only used for series with no yfinance equivalent:
# breakevens (T10YIE, T5YIE, DFII10) and OAS spreads (BAML*).
# Single attempt per series — treated as optional enrichment.
# A warning banner is shown in §3.1 if any series fail.
FRED_TIMEOUT = 60



In [ ]:
# ?1.2 ? EVENT UNIVERSE
# ?????????????????????????????????????????????????????????????
# Universal preset loader (keeps original dashboard flow/features).
# Change ACTIVE_PRESET to switch contexts quickly.

ACTIVE_PRESET = "Middle East Conflict"  # Options: Middle East Conflict, US CPI, US NFP, US PCE, US Elections, Custom
EVENT_LIBRARY_PATH = "events.yaml"

import sys; sys.dont_write_bytecode = True
from universal_preset_loader import load_preset_pack

_pack = load_preset_pack(EVENT_LIBRARY_PATH, ACTIVE_PRESET)

EVENTS = sorted(_pack.events, key=lambda x: x[1])
EVENT_TAGS = dict(_pack.event_tags)
MACRO_CONTEXT = dict(_pack.macro_context)
ALL_TAGS = list(_pack.all_tags)

# Manual reconstructed event support (used in ?1.3)
if ACTIVE_PRESET == "Middle East Conflict":
    EVENT_TAGS.setdefault("1973 Oil Embargo†", {"energy_shock", "sanctions"})
    MACRO_CONTEXT.setdefault("1973 Oil Embargo†", {"trigger": 4, "cpi": "high", "fed": "hiking"})
    for _t in ["energy_shock", "military_conflict", "shipping_disruption", "sanctions", "pandemic"]:
        if _t not in ALL_TAGS:
            ALL_TAGS.append(_t)

print(f"Preset loaded: {ACTIVE_PRESET} | Events: {len(EVENTS)} | Tags: {len(ALL_TAGS)}")




In [ ]:
# §1.3 — MANUAL DATA INJECTION
# ─────────────────────────────────────────────────────────────
# yfinance only carries data back to ~1985. For older events, supply
# hand-sourced price levels at each POI offset, then let the engine
# inject them as a standard pd.Series just like any live pull.
#
# Returns are expressed the same way as live data:
#   - Price assets  : % return from t-1W close (offset -5)
#   - Rate assets   : Δbps from t-1W level
#
# Source notes for 1973 Oil Embargo:
#   Gold     — SD Bullion AM+PM fix averages (London)
#   DXY      — FRED DTWEXM Nominal Major Currencies USD Index
#   US 10Y   — FRED DGS10 daily yield (bp change)
#   WTI      — EIA / academic sources; pre-embargo anchor ~$3/bbl
#   Brent    — Brent futures did not trade until 1988; returns proxied
#              from WTI (both tracked the same physical crude market
#              pre-1985, spread was negligible). Flagged below.
#   S&P 500  — pulled from Yahoo Finance (^GSPC available from 1970)
#              and injected in §3.1 after the main data pull.

_POI_MAP = {"t-1M": -21, "t-1W": -5, "t0": 0, "t+1W": 5, "t+1M": 21, "t+3M": 63}

def _px_to_ret(px_dict, is_rates=False):
    """Convert raw price/yield levels to returns anchored at t-1W (offset -5)."""
    anchor = px_dict["t-1W"]
    if is_rates:
        return {_POI_MAP[k]: (px_dict[k] - anchor) * 100 for k in px_dict}
    return {_POI_MAP[k]: (px_dict[k] / anchor - 1) * 100 for k in px_dict}

# Shared WTI 1973 returns — also used as Brent proxy (see note above)
_wti_1973_rets = {-21: 0.0, -5: 0.0, 0: 0.0, 5: 17.0, 21: 33.0, 63: 130.0}

MANUAL_DATA = {
    "1973 Oil Embargo†": {
        "date": "1973-10-17",
        "assets": {
            "Gold": _px_to_ret({
                "t-1M": 101.5625, "t-1W": 102.625, "t0": 103.125,
                "t+1W": 99.5875,  "t+1M": 91.25,   "t+3M": 128.4125,
            }),
            "DXY": _px_to_ret({
                "t-1M": 97.8183, "t-1W": 97.5388, "t0": 97.1477,
                "t+1W": 97.0417, "t+1M": 100.6644, "t+3M": 106.3856,
            }),
            "US 10Y Yield": _px_to_ret({
                "t-1M": 7.08, "t-1W": 6.77, "t0": 6.82,
                "t+1W": 6.75, "t+1M": 6.76, "t+3M": 6.97,
            }, is_rates=True),
            # WTI: direct % returns (EIA/academic; pre-embargo anchor ~$3/bbl)
            "WTI Crude (spot)": _wti_1973_rets,
            # Brent proxy: Brent futures didn't trade until 1988.
            # Pre-1985 WTI and Brent tracked the same physical crude market
            # (negligible spread). Values identical to WTI above.
            "Brent Futures": _wti_1973_rets,
            # S&P 500 pulled live from Yahoo (^GSPC available from 1970)
        },
    },
    # Add more pre-1985 events here following the same pattern
}


In [ ]:
# §1.4 — ASSET UNIVERSE & GROUPS
# ─────────────────────────────────────────────────────────────
# ASSETS: list of (label, yfinance/FRED ticker, asset_class, source)
#   source "yf"   → yfinance daily close
#   source "fred" → FRED via direct JSON API (requires FRED_API_KEY)
#
# FX convention: positive return = USD stronger throughout.
#   USDXXX tickers (e.g. USDJPY) → kept as-is
#   XXXUSD tickers (e.g. EURUSD) → automatically inverted and relabelled USDEUR
#   Override specific labels in INVERT_OVERRIDE if needed.
#
# CUSTOM_GROUPS: named subsets used in chart group selectors.
#   Keys appear in every group/class dropdown in the dashboard.
#   Values must reference labels that exist in ASSETS.
#
# FRED_PRICE_SERIES: FRED tickers whose values are price levels
#   (not rates). These get % return treatment instead of Δbps.
#
# USDCNH_FALLBACKS: ticker priority list for CNH — offshore yuan
#   availability varies by data provider.
ASSETS = [
    # ("Label", "ticker", "asset_class", "source")
    # source: "yf" = yfinance | "fred" = FRED via direct JSON API
    # FX inversion auto-derived: USDXXX tickers flipped, XXXUSD kept as-is

    # ── Equities (US) ─────────────────────────────────────────
    ("S&P 500",             "^GSPC",         "Equities",        "yf"),
    ("MSCI World ex-US",    "EFA",            "Equities",        "yf"),
    ("MSCI EM",             "EEM",            "Equities",        "yf"),
    ("Energy Equities",     "XLE",            "Equities",        "yf"),
    # ── World Indices ─────────────────────────────────────────
    ("Nikkei 225",          "^N225",          "World Indices",   "yf"),
    ("KOSPI",               "^KS11",          "World Indices",   "yf"),
    ("Hang Seng",           "^HSI",           "World Indices",   "yf"),
    ("Shanghai Comp",       "000001.SS",      "World Indices",   "yf"),
    ("Taiwan Weighted",     "^TWII",          "World Indices",   "yf"),
    ("ASX 200",             "^AXJO",          "World Indices",   "yf"),
    ("Nifty 50",            "^NSEI",          "World Indices",   "yf"),
    ("FTSE 100",            "^FTSE",          "World Indices",   "yf"),
    ("DAX",                 "^GDAXI",         "World Indices",   "yf"),
    ("CAC 40",              "^FCHI",          "World Indices",   "yf"),
    ("Euro Stoxx 50",       "^STOXX50E",      "World Indices",   "yf"),
    ("STI",                 "^STI",           "World Indices",   "yf"),
    ("Sensex",              "^BSESN",         "World Indices",   "yf"),
    ("Jakarta Comp",        "^JKSE",          "World Indices",   "yf"),
    # ── Sector ETFs ───────────────────────────────────────────
    ("Comm Services",       "XLC",            "Sector ETFs",     "yf"),
    ("Materials",           "XLB",            "Sector ETFs",     "yf"),
    ("Financials",          "XLF",            "Sector ETFs",     "yf"),
    ("Industrials",         "XLI",            "Sector ETFs",     "yf"),
    ("Technology",          "XLK",            "Sector ETFs",     "yf"),
    ("Cons Staples",        "XLP",            "Sector ETFs",     "yf"),
    ("Real Estate",         "XLRE",           "Sector ETFs",     "yf"),
    ("Utilities",           "XLU",            "Sector ETFs",     "yf"),
    ("Healthcare",          "XLV",            "Sector ETFs",     "yf"),
    ("Cons Discret",        "XLY",            "Sector ETFs",     "yf"),
    ("Gold Miners",         "GDX",            "Sector ETFs",     "yf"),
    ("Oil Services",        "OIH",            "Sector ETFs",     "yf"),
    ("Regional Banks",      "KRE",            "Sector ETFs",     "yf"),
    ("Homebuilders",        "ITB",            "Sector ETFs",     "yf"),
    ("Innovation (ARKK)",   "ARKK",           "Sector ETFs",     "yf"),
    # ── Bond ETFs ─────────────────────────────────────────────
    ("20Y Treasury (TLT)",  "TLT",            "Bond ETFs",       "yf"),
    ("7-10Y Treasury (IEF)","IEF",             "Bond ETFs",       "yf"),
    ("1-3Y Treasury (SHY)", "SHY",            "Bond ETFs",       "yf"),
    ("TIPS ETF (TIP)",      "TIP",            "Bond ETFs",       "yf"),
    ("Gold ETF (IAU)",      "IAU",            "Bond ETFs",       "yf"),
    # ── Thematic ETFs ─────────────────────────────────────────
    ("Airlines (JETS)",     "JETS",           "Thematic ETFs",   "yf"),
    ("Agri (MOO)",          "MOO",            "Thematic ETFs",   "yf"),
    ("Defense (ITA)",       "ITA",            "Thematic ETFs",   "yf"),
    ("Cyber Security (BUG)","BUG",            "Thematic ETFs",   "yf"),
    ("Clean Energy (ICLN)", "ICLN",           "Thematic ETFs",   "yf"),
    ("Nuclear (NLR)",       "NLR",            "Thematic ETFs",   "yf"),
    # ── Country ETFs ──────────────────────────────────────────
    ("ETF Japan",           "EWJ",            "Country ETFs",    "yf"),
    ("ETF Brazil",          "EWZ",            "Country ETFs",    "yf"),
    ("ETF Korea",           "EWY",            "Country ETFs",    "yf"),
    ("ETF Taiwan",          "EWT",            "Country ETFs",    "yf"),
    ("ETF China",           "MCHI",           "Country ETFs",    "yf"),
    ("ETF Germany",         "EWG",            "Country ETFs",    "yf"),
    ("ETF UK",              "EWU",            "Country ETFs",    "yf"),
    ("ETF Australia",       "EWA",            "Country ETFs",    "yf"),
    ("ETF India",           "INDA",           "Country ETFs",    "yf"),
    ("ETF Mexico",          "EWW",            "Country ETFs",    "yf"),
    ("ETF South Africa",    "EZA",            "Country ETFs",    "yf"),
    ("ETF Turkey",          "TUR",            "Country ETFs",    "yf"),
    # ── Oil & Energy ──────────────────────────────────────────
    ("WTI Crude (spot)",    "CL=F",           "Oil & Energy",    "yf"),
    ("Brent Futures",       "BZ=F",           "Oil & Energy",    "yf"),
    # Natural Gas spot (DHHNGSP) removed — NG=F futures covers it
    ("Natural Gas Fut",     "NG=F",           "Oil & Energy",    "yf"),
    ("Oil Vol (OVX)",       "OVX",            "Oil & Energy",    "yf"),
    ("Gasoline (UGA)",      "UGA",            "Oil & Energy",    "yf"),
    # ── Precious Metals ───────────────────────────────────────
    ("Gold",                "GC=F",           "Precious Metals", "yf"),
    ("Silver",              "SI=F",           "Precious Metals", "yf"),
    ("Platinum",            "PL=F",           "Precious Metals", "yf"),
    ("Palladium",           "PA=F",           "Precious Metals", "yf"),
    ("Gold Vol (GVZ)",      "GVZ",            "Precious Metals", "yf"),
    # ── FX G10 ────────────────────────────────────────────────
    ("DXY",                 "DX-Y.NYB",       "FX",              "yf"),
    ("USDCHF",              "USDCHF=X",       "FX",              "yf"),
    ("EURUSD",              "EURUSD=X",       "FX",              "yf"),
    ("GBPUSD",              "GBPUSD=X",       "FX",              "yf"),
    ("AUDUSD",              "AUDUSD=X",       "FX",              "yf"),
    ("NZDUSD",              "NZDUSD=X",       "FX",              "yf"),
    ("USDJPY",              "USDJPY=X",       "FX",              "yf"),
    ("USDCAD",              "USDCAD=X",       "FX",              "yf"),
    ("USDNOK",              "USDNOK=X",       "FX",              "yf"),
    ("USDSEK",              "USDSEK=X",       "FX",              "yf"),
    # ── FX EM LATAM ───────────────────────────────────────────
    ("USDMXN",              "USDMXN=X",       "FX",              "yf"),
    ("USDBRL",              "USDBRL=X",       "FX",              "yf"),
    ("USDCLP",              "USDCLP=X",       "FX",              "yf"),
    ("USDCOP",              "USDCOP=X",       "FX",              "yf"),
    # ── FX EM EMEA ────────────────────────────────────────────
    ("USDZAR",              "USDZAR=X",       "FX",              "yf"),
    ("USDTRY",              "USDTRY=X",       "FX",              "yf"),
    ("USDPLN",              "USDPLN=X",       "FX",              "yf"),
    ("USDHUF",              "USDHUF=X",       "FX",              "yf"),
    ("USDCZK",              "USDCZK=X",       "FX",              "yf"),
    ("USDILS",              "USDILS=X",       "FX",              "yf"),
    # ── FX EM Asia ────────────────────────────────────────────
    # USDCNH: try primary ticker, fall back to CNH=X
    ("USDCNH",              "USDCNH=X",       "FX",              "yf"),
    ("USDTWD",              "TWD=X",          "FX",              "yf"),
    ("USDINR",              "USDINR=X",       "FX",              "yf"),
    ("USDIDR",              "USDIDR=X",       "FX",              "yf"),
    ("USDKRW",              "USDKRW=X",       "FX",              "yf"),
    ("USDMYR",              "USDMYR=X",       "FX",              "yf"),
    ("USDTHB",              "USDTHB=X",       "FX",              "yf"),
    ("USDPHP",              "USDPHP=X",       "FX",              "yf"),
    ("USDSGD",              "USDSGD=X",       "FX",              "yf"),
    # ── Rates (yfinance) ─────────────────────────────────────
    # ^IRX=13W T-bill, ^FVX=5Y, ^TNX=10Y, ^TYX=30Y — yield indices (bp-change)
    ("US 3M Yield",         "^IRX",           "Rates",           "yf"),
    ("US 5Y Yield",         "^FVX",           "Rates",           "yf"),
    ("US 10Y Yield",        "^TNX",           "Rates",           "yf"),
    ("US 30Y Yield",        "^TYX",           "Rates",           "yf"),
    # ── Rates (FRED-only — breakevens & real yield) ───────────
    # No yfinance equivalent. Single attempt — skipped if FRED is down.
    ("US 10Y Breakeven",    "T10YIE",         "Rates",           "fred"),
    ("US 5Y Breakeven",     "T5YIE",          "Rates",           "fred"),
    ("US 10Y Real Yield",   "DFII10",         "Rates",           "fred"),
    # ── Rates Futures ─────────────────────────────────────────
    ("2Y UST Fut",          "ZT=F",           "Rates Futures",   "yf"),
    ("5Y UST Fut",          "ZF=F",           "Rates Futures",   "yf"),
    ("10Y UST Fut",         "ZN=F",           "Rates Futures",   "yf"),
    ("30Y UST Fut",         "ZB=F",           "Rates Futures",   "yf"),
    # ── Credit (FRED) ─────────────────────────────────────────
    ("US IG OAS",           "BAMLC0A0CM",     "Credit",          "fred"),
    ("US BBB OAS",          "BAMLC0A4CBBB",   "Credit",          "fred"),
    ("US HY OAS",           "BAMLH0A0HYM2",   "Credit",          "fred"),
    ("Euro HY OAS",         "BAMLHE00EHY0EY", "Credit",          "fred"),
    ("HY Bond ETF (HYG)",   "HYG",            "Credit",          "yf"),
    ("IG Bond ETF (LQD)",   "LQD",            "Credit",          "yf"),
    ("EM Sov Debt (EMB)",   "EMB",            "Credit",          "yf"),
    # ── Volatility ────────────────────────────────────────────
    ("VIX",                 "^VIX",           "Volatility",      "yf"),
    ("VXN (Nasdaq Vol)",    "^VXN",           "Volatility",      "yf"),
    # ── Commodities ───────────────────────────────────────────
    ("Copper",              "HG=F",           "Commodities",     "yf"),
    ("Broad Commod",        "DJP",            "Commodities",     "yf"),
    ("Broad Commod (PDBC)", "PDBC",           "Commodities",     "yf"),
    ("Agriculture",         "DBA",            "Commodities",     "yf"),
    ("Rice",                "ZR=F",           "Commodities",     "yf"),
    ("Corn",                "ZC=F",           "Commodities",     "yf"),
    ("Wheat",               "ZW=F",           "Commodities",     "yf"),
    ("Soybeans",            "ZS=F",           "Commodities",     "yf"),
    ("Lumber",              "LBS=F",          "Commodities",     "yf"),
    ("Cocoa",               "CC=F",           "Commodities",     "yf"),
    ("Coffee",              "KC=F",           "Commodities",     "yf"),
    ("Cotton",              "CT=F",           "Commodities",     "yf"),
    ("Sugar",               "SB=F",           "Commodities",     "yf"),
    # ── Crypto ────────────────────────────────────────────────
    ("Bitcoin",             "BTC-USD",        "Crypto",          "yf"),
    ("Ethereum",            "ETH-USD",        "Crypto",          "yf"),
    # ── Shipping ──────────────────────────────────────────────
    ("Shipping (BDRY)",     "BDRY",           "Shipping",        "yf"),
]

# USDCNH fallback tickers — tried in order until one works
USDCNH_FALLBACKS = ["USDCNH=X", "CNH=X", "CNYUSD=X"]

# FX inversion override
INVERT_OVERRIDE = {"DXY": True}

# YF_RATES_TICKERS: yfinance tickers returning yield levels (not prices).
# bp-change treatment: Δbps = (current_yield - prior_yield) * 100
YF_RATES_TICKERS = {"^IRX", "^FVX", "^TNX", "^TYX"}

# FRED series treated as prices (% return), not levels (bp change)
FRED_PRICE_SERIES = set()  # OAS/breakevens are bp-change; no FRED price-level series

CUSTOM_GROUPS = {
    # ── Equities ──────────────────────────────────────────────
    "Equities":             ["S&P 500","MSCI World ex-US","MSCI EM","Energy Equities"],
    "Sector ETFs":          ["Comm Services","Materials","Financials","Industrials",
                             "Technology","Cons Staples","Real Estate","Utilities","Healthcare","Cons Discret",
                             "Gold Miners","Oil Services","Regional Banks","Homebuilders","Innovation (ARKK)"],
    "World Indices":        ["Nikkei 225","KOSPI","Hang Seng","Shanghai Comp","Taiwan Weighted",
                             "ASX 200","Nifty 50","FTSE 100","DAX","CAC 40","Euro Stoxx 50",
                             "STI","Sensex","Jakarta Comp"],
    "Country ETFs":         ["ETF Japan","ETF Brazil","ETF Korea","ETF Taiwan","ETF China",
                             "ETF Germany","ETF UK","ETF Australia","ETF India",
                             "ETF Mexico","ETF South Africa","ETF Turkey"],
    "Country ETFs Asia":    ["ETF Japan","ETF Korea","ETF Taiwan","ETF China","ETF Australia","ETF India"],
    "Country ETFs EM":      ["ETF Brazil","ETF Korea","ETF Taiwan","ETF China",
                             "ETF India","ETF Mexico","ETF South Africa","ETF Turkey"],
    # ── Commodities ───────────────────────────────────────────
    "Oil & Energy":         ["WTI Crude (spot)","Brent Futures","Natural Gas Fut",
                             "Oil Vol (OVX)","Gasoline (UGA)","Energy Equities","Oil Services"],
    "Precious Metals":      ["Gold","Silver","Platinum","Palladium","Gold Vol (GVZ)"],
    "Commodities":          ["Copper","Broad Commod","Broad Commod (PDBC)","Agriculture",
                             "Rice","Corn","Wheat","Soybeans","Lumber","Cocoa","Coffee","Cotton","Sugar"],
    "Soft Commodities":     ["Agriculture","Rice","Corn","Wheat","Soybeans","Cocoa","Coffee","Cotton","Sugar"],
    # ── FX ────────────────────────────────────────────────────
    "FX All":               ["DXY","USDCHF","EURUSD","GBPUSD","AUDUSD","NZDUSD","USDJPY",
                             "USDCAD","USDNOK","USDSEK",
                             "USDMXN","USDBRL","USDCLP","USDCOP",
                             "USDZAR","USDTRY","USDPLN","USDHUF","USDCZK","USDILS",
                             "USDCNH","USDTWD","USDINR","USDIDR","USDKRW","USDMYR","USDTHB","USDPHP","USDSGD"],
    "FX G10":               ["DXY","USDCAD","USDNOK","AUDUSD","NZDUSD","USDJPY","EURUSD","GBPUSD","USDCHF","USDSEK"],
    "FX EM":                ["USDMXN","USDBRL","USDCLP","USDCOP",
                             "USDZAR","USDTRY","USDPLN","USDHUF","USDCZK","USDILS",
                             "USDCNH","USDTWD","USDINR","USDIDR","USDKRW","USDMYR","USDTHB","USDPHP"],
    "FX EM Asia":           ["USDCNH","USDTWD","USDINR","USDIDR","USDKRW","USDMYR","USDTHB","USDPHP","USDSGD"],
    "FX ASEAN":             ["USDSGD","USDMYR","USDTHB","USDPHP","USDIDR"],
    "FX EM EMEA":           ["USDZAR","USDTRY","USDPLN","USDHUF","USDCZK","USDILS"],
    "FX EM LATAM":          ["USDMXN","USDBRL","USDCLP","USDCOP"],
    "FX EM High Carry":     ["USDMXN","USDBRL","USDCLP","USDCOP","USDZAR","USDTRY"],
    "FX EM Low Carry":      ["USDCNH","USDTWD","USDKRW","USDINR","USDMYR","USDTHB","USDPHP","USDSGD"],
    "FX Oil Exporters":     ["USDCAD","USDNOK","AUDUSD","NZDUSD","USDMXN","USDBRL","USDCOP"],
    "FX Oil Importers":     ["USDJPY","EURUSD","GBPUSD","USDCHF","USDZAR","USDTRY",
                             "USDPLN","USDCNH","USDTWD","USDINR","USDIDR","USDKRW","USDMYR","USDTHB","USDPHP"],
    "Dollar Bloc":          ["USDCAD","AUDUSD","NZDUSD"],
    "Commodity FX":         ["USDCAD","USDNOK","AUDUSD","USDMXN","USDBRL","USDCOP","USDCLP","USDZAR"],
    # ── Rates ─────────────────────────────────────────────────
    "DM Rates":             ["US 1Y Yield","US 2Y Yield","US 5Y Yield","US 10Y Yield","US 30Y Yield",
                             "US 10Y Breakeven","US 5Y Breakeven","US 10Y Real Yield"],
    "Yield Curve":          ["US 2Y Yield","US 10Y Yield","US 30Y Yield"],
    "Breakevens":           ["US 5Y Breakeven","US 10Y Breakeven","US 10Y Real Yield"],
    "Rates Futures":        ["2Y UST Fut","5Y UST Fut","10Y UST Fut","30Y UST Fut"],
    # ── Credit ────────────────────────────────────────────────
    "Credit":               ["US IG OAS","US BBB OAS","US HY OAS","Euro HY OAS",
                             "HY Bond ETF (HYG)","IG Bond ETF (LQD)","EM Sov Debt (EMB)"],
    # ── Volatility ────────────────────────────────────────────
    "Volatility":           ["VIX","VXN (Nasdaq Vol)","Oil Vol (OVX)","Gold Vol (GVZ)"],
    # ── Crypto ────────────────────────────────────────────────
    "Crypto":               ["Bitcoin","Ethereum"],
    # ── Shipping ──────────────────────────────────────────────
    "Shipping":             ["Shipping (BDRY)"],
    # ── Bond ETFs ─────────────────────────────────────────────
    "Bond ETFs":            ["20Y Treasury (TLT)","7-10Y Treasury (IEF)",
                             "1-3Y Treasury (SHY)","TIPS ETF (TIP)","Gold ETF (IAU)"],
    "Yield Curve ETFs":     ["1-3Y Treasury (SHY)","7-10Y Treasury (IEF)","20Y Treasury (TLT)"],
    # ── Thematic ETFs ─────────────────────────────────────────
    "Thematic ETFs":        ["Airlines (JETS)","Agri (MOO)","Defense (ITA)",
                             "Cyber Security (BUG)","Clean Energy (ICLN)","Nuclear (NLR)"],
    "Defense & Security":   ["Defense (ITA)","Cyber Security (BUG)","Nuclear (NLR)"],
    "Oil Sensitive":        ["Airlines (JETS)","Agri (MOO)","WTI Crude (spot)",
                             "Brent Futures","Oil Services","Energy Equities"],
    # ── Cross-asset themes ────────────────────────────────────
    "Risk Barometer":       ["S&P 500","VIX","US HY OAS","WTI Crude (spot)","DXY"],
    "Safe Havens":          ["Gold","Silver","USDJPY","USDCHF","US 10Y Yield","US 10Y Real Yield"],
    "Risk-On Basket":       ["S&P 500","MSCI EM","Copper","WTI Crude (spot)","US HY OAS","Bitcoin"],
    "Inflation Hedge":      ["Gold","WTI Crude (spot)","Copper","US 10Y Breakeven","US 5Y Breakeven","Agriculture"],
    "EM Stress":            ["MSCI EM","EM Sov Debt (EMB)","USDCNH","USDKRW","USDINR","US HY OAS","Copper"],
    "Middle East Risk":     ["WTI Crude (spot)","Brent Futures","Gold","Oil Vol (OVX)",
                             "USDILS","Energy Equities","Oil Services"],
}


In [ ]:
# §1.5 — ANALYSIS WINDOWS & HORIZONS
# ─────────────────────────────────────────────────────────────
# PRE_WINDOW_TD / POST_WINDOW_TD : trading days of history shown around Day 0.
# FETCH_BUFFER                   : extra calendar days added to each end of the
#                                  data pull to ensure clean price denominators.
# POIS                           : Points of Interest — the named offsets used
#                                  in heatmaps, summary tables, and step-in analysis.
#                                  Format: (label, trading_day_offset_from_day0)
# DAY0_ROLL                      : "ceil" → first trading day >= event date (default)
#                                  "floor" → last trading day <= event date
# STEP_IN_PRIMARY                : default day shown in Step-In Analysis tab.
# VIX_TICKER / VIX_LABEL        : VIX series identifiers.

PRE_WINDOW_TD   = 21    # trading days before Day 0
POST_WINDOW_TD  = 63    # trading days after Day 0 (≈ t+3M)
FETCH_BUFFER    = 45    # extra calendar days for clean price denominators

POIS = [
    ("t-1M",  -21),
    ("t-1W",   -5),
    ("t0",      0),
    ("t+1W",    5),
    ("t+1M",   21),
    ("t+3M",   63),
]

DAY0_ROLL       = "ceil"   # first trading day >= event date
STEP_IN_PRIMARY = 7        # default step-in day for Step-In Analysis tab
VIX_TICKER      = "^VIX"
VIX_LABEL       = "VIX"


In [ ]:
# §1.6 — ANALOGUE ENGINE CONFIG
# ─────────────────────────────────────────────────────────────
# TRIGGER_ASSET
#   Names the primary market asset for this event class.
#   Changing this single variable propagates to three places automatically:
#     1. SIMILARITY_ASSET_POOL — trigger asset is always slot 0 (highest priority)
#     2. Live Config widget label — shows the trigger asset's current price
#     3. _macro_sim() — trigger level comparison uses percentile rank within the
#        TRIGGER_ASSET's own rolling 5-year price history (scale-invariant).
#        See TRIGGER_PERCENTILE_MAP built in §3.1.
#
#   Examples:
#     Brent Futures   → oil/geopolitical shock dashboard (current)
#     US 10Y Yield    → rates shock dashboard (e.g. central bank shocks, taper tantrums)
#     S&P 500         → equity shock dashboard (e.g. crashes, corrections)
#     DXY             → dollar shock dashboard (e.g. EM crises, Plaza Accord analogues)

TRIGGER_ASSET = "Brent Futures"

# ANALOGUE_WEIGHTS
#   Composite score = quant*q + tag*t + macro*m  (must sum to 1.0)
#   quant  : cosine similarity of live vs historical asset returns at Day+N
#   tag    : Jaccard similarity of event tags (EVENT_TAGS §1.2)
#   macro  : macro context match (trigger level, CPI regime, Fed stance)
#
#   For a pure price-pattern study:  {"quant": 1.0, "tag": 0.0, "macro": 0.0}
#   For a pure structural study:     {"quant": 0.0, "tag": 0.5, "macro": 0.5}

ANALOGUE_WEIGHTS = {
    "quant": 0.50,
    "tag":   0.30,
    "macro": 0.20,
}
assert abs(sum(ANALOGUE_WEIGHTS.values()) - 1.0) < 1e-9, "ANALOGUE_WEIGHTS must sum to 1.0"

# SIMILARITY_ASSET_POOL
#   Assets whose Day+N returns are used in the cosine similarity calculation.
#   TRIGGER_ASSET is always prepended automatically — do not add it again here.
#   Recommendation: prefer yfinance-only assets (no FRED publication lag).
#   These also populate the sim-asset multi-select widget in the dashboard.

_SIM_POOL_BASE = [
    "VIX", "Gold", "DXY", "S&P 500",
    "US 10Y Yield", "US HY OAS",
    "EURUSD", "USDJPY", "Copper", "Shipping (BDRY)",
]
# Prepend TRIGGER_ASSET (deduplicated)
SIMILARITY_ASSET_POOL = (
    [TRIGGER_ASSET] + [a for a in _SIM_POOL_BASE if a != TRIGGER_ASSET]
)

# TRIGGER_ZSCORE_SIGMA
#   Controls the width of the Gaussian similarity kernel used in the
#   macro trigger comparison (§5.2 _macro_sim). Similarity is:
#       trigger_score = exp(-0.5 * (z_live - z_hist)^2 / sigma^2)
#   sigma = 1.0 → tight: only events within ±1 std dev score well (>60%)
#   sigma = 1.5 → moderate (default): ±1.5 std dev events score ~72%
#   sigma = 2.0 → loose: events up to ±2 std devs still score ~61%
#   Increase sigma to cast a wider net when your event universe is small.

TRIGGER_ZSCORE_SIGMA = 1.5

# ANALOGUE_MIN_COUNT
#   Minimum number of analogue events to keep after applying the score cutoff.
#   If the cutoff is too strict, the selector widens to the top-N scored events.
#   Keeps sparse presets actionable without fully discarding the ranking.
ANALOGUE_MIN_COUNT = 3

# LIVE_FRED_MAP
#   FRED series to pull for the live event window (label → FRED ticker).
#   Only used if FRED_API_KEY is set. Add/remove series here.

LIVE_FRED_MAP = {
    # Yields → yfinance. Only genuine FRED-only series here.
    "US 10Y Breakeven": "T10YIE",
    "US 5Y Breakeven":  "T5YIE",
    "US 10Y Real Yield":"DFII10",
    "US IG OAS":        "BAMLC0A0CM",
    "US BBB OAS":       "BAMLC0A4CBBB",
    "US HY OAS":        "BAMLH0A0HYM2",
    "Euro HY OAS":      "BAMLHE00EHY0EY",
}

# PORTFOLIO_SCENARIOS
#   3 sample portfolios for E2 Portfolio Stress Tester (L11).
#   Each portfolio is a dict of { asset_label: notional_usd }
#   Positive notional = long, negative = short.
#   Labels must match ASSETS labels exactly.
#   Add/copy/modify portfolios freely — each dict is one scenario.
#   The stress tester runs all scenarios side by side.

PORTFOLIO_SCENARIOS = {
    "Geopolitical Long": {
        # Long energy + safe havens, short risk equities
        "Brent Futures":        500_000,
        "WTI Crude (spot)":     300_000,
        "Gold":                 400_000,
        "20Y Treasury (TLT)":   300_000,
        "Defense (ITA)":        200_000,
        "S&P 500":             -300_000,
        "MSCI EM":             -200_000,
        "Airlines (JETS)":     -150_000,
    },
    "Risk-Off Flight": {
        # Flight to quality — long bonds/gold/JPY, short credit/EM
        "20Y Treasury (TLT)":   600_000,
        "Gold":                 400_000,
        "USDJPY":              -300_000,   # short USDJPY = long JPY
        "USDCHF":              -200_000,   # short USDCHF = long CHF
        "HY Bond ETF (HYG)":   -400_000,
        "EM Sov Debt (EMB)":   -300_000,
        "MSCI EM":             -300_000,
    },
    "Oil Shock Arb": {
        # Long oil complex + commodity FX, short oil consumers
        "Brent Futures":        600_000,
        "Natural Gas Fut":      200_000,
        "Oil Services":         300_000,
        "Energy Equities":      300_000,
        "USDNOK":              -200_000,   # short USDNOK = long NOK (oil exporter)
        "USDCAD":              -200_000,   # short USDCAD = long CAD
        "Airlines (JETS)":     -400_000,
        "Technology":          -200_000,
    },
}

# ?? Preset overrides (universal mode) ???????????????????????
from universal_preset_loader import PRESET_CONFIGS
_pcfg = PRESET_CONFIGS.get(globals().get("ACTIVE_PRESET", "Middle East Conflict"), PRESET_CONFIGS["Middle East Conflict"])

TRIGGER_ASSET = _pcfg["trigger_asset"]
ANALOGUE_WEIGHTS = dict(_pcfg["weights"])
assert abs(sum(ANALOGUE_WEIGHTS.values()) - 1.0) < 1e-9, "ANALOGUE_WEIGHTS must sum to 1.0"

_SIM_POOL_BASE = list(_pcfg["sim_pool_base"])
SIMILARITY_ASSET_POOL = [TRIGGER_ASSET] + [a for a in _SIM_POOL_BASE if a != TRIGGER_ASSET]
print(f"Preset trigger: {TRIGGER_ASSET} | Similarity assets: {len(SIMILARITY_ASSET_POOL)}")



In [ ]:
# §1.7 — CONFIG VALIDATION
# ─────────────────────────────────────────────────────────────
# Cross-checks §1.1–§1.6 for internal consistency.
# Run after editing any config cell to catch issues before the data pull.
# Prints a ✅ summary on pass; lists every problem found on fail.
#
# Checks:
#   • Every event in EVENTS has an entry in EVENT_TAGS and MACRO_CONTEXT
#   • MANUAL_DATA event names end in "†" and are NOT duplicated in EVENTS
#   • TRIGGER_ASSET exists in ASSETS (has a ticker in ASSET_META — checked post §3.1)
#   • ANALOGUE_WEIGHTS sum to 1.0
#   • SIMILARITY_ASSET_POOL items are all in ASSETS
#   • POIS offsets are sorted ascending
#   • All POI labels in MANUAL_DATA entries match _POI_MAP keys

_errs = []

# Check EVENTS ↔ EVENT_TAGS ↔ MACRO_CONTEXT consistency
all_event_names = [n for n, _ in EVENTS] + list(MANUAL_DATA.keys())
for name in all_event_names:
    if name not in EVENT_TAGS:
        _errs.append(f"  ⚠️  EVENT_TAGS missing: '{name}'")
    if name not in MACRO_CONTEXT:
        _errs.append(f"  ⚠️  MACRO_CONTEXT missing: '{name}'")

# Check MANUAL_DATA events have † suffix and are not also in EVENTS
for name in MANUAL_DATA:
    if not name.endswith("†"):
        _errs.append(f"  ⚠️  MANUAL_DATA key should end in '†': '{name}'")
    if name in [n for n, _ in EVENTS]:
        _errs.append(f"  ⚠️  MANUAL_DATA event '{name}' also appears in EVENTS — remove from EVENTS")

# Check ANALOGUE_WEIGHTS
if abs(sum(ANALOGUE_WEIGHTS.values()) - 1.0) > 1e-9:
    _errs.append(f"  ⚠️  ANALOGUE_WEIGHTS sum = {sum(ANALOGUE_WEIGHTS.values()):.4f} (must be 1.0)")

# Check POIS are sorted ascending by offset
poi_offsets = [o for _, o in POIS]
if poi_offsets != sorted(poi_offsets):
    _errs.append(f"  ⚠️  POIS offsets not sorted ascending: {poi_offsets}")

# Check MANUAL_DATA POI keys are valid
for name, cfg in MANUAL_DATA.items():
    for lbl, ret_dict in cfg.get("assets", {}).items():
        bad_keys = [k for k in ret_dict if k not in [o for _, o in POIS]]
        if bad_keys:
            _errs.append(f"  ⚠️  MANUAL_DATA '{name}' / '{lbl}': unrecognised offset keys {bad_keys}")

# Check SIMILARITY_ASSET_POOL entries exist in ASSETS
asset_labels = {lbl for lbl, *_ in ASSETS}
valid_similarity_labels = set(asset_labels)
for lbl, ticker, cls, _source in ASSETS:
    if cls != "FX":
        continue
    tkr = ticker.replace("=X", "")
    invert = INVERT_OVERRIDE.get(lbl, tkr.startswith("USD"))
    if not invert and len(tkr) >= 6:
        valid_similarity_labels.add(f"USD{tkr[:3]}")
for a in SIMILARITY_ASSET_POOL:
    if a not in valid_similarity_labels and a != "VIX":
        _errs.append(f"  ⚠️  SIMILARITY_ASSET_POOL: '{a}' not found in ASSETS")

# Check TRIGGER_ASSET exists in ASSETS
if TRIGGER_ASSET not in asset_labels:
    _errs.append(f"  ⚠️  TRIGGER_ASSET '{TRIGGER_ASSET}' not found in ASSETS")

# Report
if _errs:
    print(f"❌ Config validation: {len(_errs)} issue(s) found\n")
    for e in _errs:
        print(e)
else:
    n_events = len(all_event_names)
    n_assets = len(asset_labels)
    print(f"✅ Config validation passed")
    print(f"   {n_events} events  |  {n_assets} assets  |  "
          f"TRIGGER_ASSET = '{TRIGGER_ASSET}'  |  "
          f"Weights = {ANALOGUE_WEIGHTS}")


---
## § 2 · SETUP
> Install dependencies and import libraries. Run once per environment.  
> Do not edit unless adding a new dependency.


In [ ]:
# §2.1 — INSTALL DEPENDENCIES
# ─────────────────────────────────────────────────────────────
# Run once per new environment. Safe to skip if packages already installed.
!pip install yfinance pandas numpy matplotlib seaborn python-dateutil plotly ipywidgets pyyaml requests

In [ ]:
# §2.2 — IMPORTS & SHARED THEME
# ─────────────────────────────────────────────────────────────
# Standard imports + Plotly dark theme constants used across all chart functions.
# EVENT_COLORS / EVENT_DASHES : 13-colour palette, visually distinct on dark bg.
# CLASS_COLORS                : per asset-class colour for group charts.
# dark_layout()               : helper that merges the shared dark theme into
#                               any fig.update_layout() call.
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import yfinance as yf
from datetime import timedelta


# ── SHARED DARK THEME ─────────────────────────────────────────
BG        = "#0d1117"
BG_PANEL  = "#161b22"
BG_CELL   = "#21262d"
COL_GRID  = "#30363d"
COL_TEXT  = "#c9d1d9"
COL_HEAD  = "#e6edf3"
COL_MUTED = "#8b949e"
COL_ZERO  = "#58a6ff"

DARK_LAYOUT = dict(
    paper_bgcolor=BG,
    plot_bgcolor=BG_PANEL,
    font=dict(color=COL_TEXT, family="monospace"),
    title_font=dict(color=COL_HEAD, size=14),
    legend=dict(
        bgcolor=BG_PANEL,
        bordercolor=COL_GRID,
        borderwidth=1,
        font=dict(color=COL_TEXT, size=10),
    ),
    xaxis=dict(gridcolor=COL_GRID, zerolinecolor=COL_GRID,
               linecolor=COL_GRID, tickcolor=COL_MUTED,
               tickfont=dict(color=COL_MUTED)),
    yaxis=dict(gridcolor=COL_GRID, zerolinecolor=COL_GRID,
               linecolor=COL_GRID, tickcolor=COL_MUTED,
               tickfont=dict(color=COL_MUTED)),
)

def dark_layout(**kwargs):
    d = dict(DARK_LAYOUT)
    # merge axes overrides
    for k, v in kwargs.items():
        if k in ("xaxis", "yaxis") and k in d:
            merged = dict(d[k]); merged.update(v); d[k] = merged
        else:
            d[k] = v
    return d

# 13-event colour palette — visually distinct, dark-mode friendly
# Pairs that could be confused get different dash styles
EVENT_COLORS = [
    "#58a6ff",  # 0  blue
    "#f78166",  # 1  salmon
    "#3fb950",  # 2  green
    "#d2a8ff",  # 3  lavender
    "#ffa657",  # 4  orange
    "#ff7b72",  # 5  red-orange
    "#79c0ff",  # 6  light blue
    "#56d364",  # 7  lime
    "#e3b341",  # 8  gold
    "#bc8cff",  # 9  purple
    "#ff9bce",  # 10 pink
    "#89ddff",  # 11 cyan
    "#a5d6ff",  # 12 pale blue
]
EVENT_DASHES = [
    "solid", "solid", "solid", "solid",
    "dash",  "dash",  "dash",  "dash",
    "dot",   "dot",   "dot",   "dot",
    "dashdot",
]

CLASS_COLORS = {
    "Equities":        "#58a6ff",
    "World Indices":   "#79c0ff",
    "Sector ETFs":     "#a5d6ff",
    "Oil & Energy":    "#ffa657",
    "Precious Metals": "#e3b341",
    "FX":              "#3fb950",
    "Rates":           "#f78166",
    "Rates Futures":   "#ff9bce",
    "Credit":          "#d2a8ff",
    "Volatility":      "#bc8cff",
    "Commodities":     "#56d364",
    "Crypto":          "#89ddff",
    "Shipping":        "#8b949e",
}

CHART_W = 1100
CHART_H = 540
print("✅ Imports OK.")

# Lightweight refresh helpers used by chart/widget cells before §6.0 runs.
def make_refresh_state():
    return {"busy": False, "last": None, "ready": False}

def run_guarded_refresh(state, key, render_fn, label="refresh"):
    if not state.get("ready") or state.get("busy"):
        return None
    if key == state.get("last"):
        return None
    state["busy"] = True
    try:
        result = render_fn()
        state["last"] = key
        return result
    except Exception as _ui_e:
        print(f"⚠️  {label} failed: {_ui_e}")
        return None
    finally:
        state["busy"] = False

def out_fig(fig, out):
    with out:
        clear_output(wait=True)
        if not fig:
            return
        try:
            fig.show()
        except Exception as _fig_e:
            print(f"⚠️  Figure render failed: {_fig_e}")

if 'LIVE' not in dir():
    LIVE = {"name": None, "day0": None, "tags": set(), "trigger": 0.0,
            "cpi": "low", "fed": "hold", "trigger_pctile": None, "sim_assets": [],
            "prices": None, "returns": None, "day_n": 0, "bday_index": []}


---
## § 3 · DATA
> Pulls and aligns all price history. Runs once on kernel start (~30-60s).  
> Re-run §3.1 only if you change `EVENTS` or `ASSETS` in the CONFIG section.  
> Custom events added via the dashboard Events tab do a targeted incremental pull.


In [ ]:
# §3.1 — DATA PULL & PREPROCESSING
# ─────────────────────────────────────────────────────────────
# Builds the unified `prices` DataFrame (label-keyed columns, date index).
# All chart functions and the analogue engine read from `prices` and
# `event_returns` — do not modify these directly.
#
# Outputs:
#   prices        : pd.DataFrame — aligned daily closes, all assets
#   event_returns : dict[label → dict[event_name → pd.Series(offset→return)]]
#   ASSET_META    : dict[label → {ticker, class, source, invert, ...}]
#   ASSET_ORDER   : list[label] in definition order
#   ALL_LABELS    : list[label] actually present in prices (post-pull)
#   ALL_CLASSES   : sorted list of unique asset classes

# ── Build metadata ────────────────────────────────────────────
ASSET_META  = {}
ASSET_ORDER = []
# ── Display label helper ─────────────────────────────────────
def _make_display_label(label, ticker, cls, invert):
    """
    USD-first convention: positive return always means USD stronger.
    - USDXXX tickers: kept as-is (invert=True), label unchanged.
    - XXXUSD tickers (e.g. EURUSD): flipped (invert=False),
      displayed as USDEUR so positive = USD stronger.
    - DXY and overrides: kept as-is.
    """
    if cls != "FX":
        return label
    tkr = ticker.replace("=X", "")
    if not tkr.startswith("USD") and not INVERT_OVERRIDE.get(label, False):
        # XXXUSD pair — flipped to USD-first, rename
        foreign = tkr[:3]
        return f"USD{foreign}"
    return label  # USDXXX, DXY — already USD-first

seen = set()
for label, ticker, cls, source in ASSETS:
    if label in seen: continue
    seen.add(label)
    ASSET_ORDER.append(label)
    if label in INVERT_OVERRIDE:
        invert = INVERT_OVERRIDE[label]
    elif cls == "FX":
        invert = ticker.replace("=X","").startswith("USD")  # True=keep, False=flip to USD-first
    else:
        invert = True
    ASSET_META[label] = {
        "ticker":        ticker,
        "class":         cls,
        "source":        source,
        "invert":        invert,
        "is_fred_price": source=="fred" and ticker in FRED_PRICE_SERIES,
        "is_rates_bp":   (source=="fred" and ticker not in FRED_PRICE_SERIES)
                                       or (source=="yf" and ticker in YF_RATES_TICKERS),
        "display_label": _make_display_label(label, ticker, cls, invert),
    }

# ── Date range ────────────────────────────────────────────────
event_dates  = [pd.Timestamp(d) for _,d in EVENTS]
global_start = min(event_dates) - timedelta(days=PRE_WINDOW_TD*2 + FETCH_BUFFER)
global_end   = max(event_dates) + timedelta(days=POST_WINDOW_TD*2 + FETCH_BUFFER)
print(f"Fetching {global_start.date()} → {global_end.date()} …")

# ── Guardrails + cache-first execution ──────────────────────────
import os, pickle, hashlib, json, tempfile, time as _time

CACHE_VERSION = "v7"
CACHE_DIR = "_runtime/cache_prices"
CACHE_ENABLED = True
CACHE_FULL_SNAPSHOT = True  # includes event_returns + trigger map
MAX_STAGE_SECONDS = 900

_stage_started = _time.monotonic()
def _stage_check(stage_name):
    elapsed = _time.monotonic() - _stage_started
    if elapsed > MAX_STAGE_SECONDS:
        raise TimeoutError(f"Stage timeout in §3.1 ({stage_name}) after {elapsed:.1f}s")

os.makedirs(CACHE_DIR, exist_ok=True)

_cache_payload_key = {
    "version": CACHE_VERSION,
    "preset": globals().get("ACTIVE_PRESET", "unknown"),
    "events": sorted([(n, str(d)) for n, d in EVENTS]),
    "assets": sorted([(l, ASSET_META[l]["ticker"], ASSET_META[l]["source"]) for l in ASSET_ORDER]),
    "fred_enabled": bool(globals().get("FRED_API_KEY")),
    "start": str(pd.Timestamp(global_start).date()),
    "end": str(pd.Timestamp(global_end).date()),
}
_cache_hash = hashlib.md5(
    json.dumps(_cache_payload_key, sort_keys=True, default=str).encode("utf-8")
).hexdigest()[:16]
_CACHE_PATH = os.path.join(CACHE_DIR, f"prices_cache_{_cache_hash}.pkl")

def _atomic_pickle_save(path, obj):
    tmp_fd, tmp_path = tempfile.mkstemp(prefix="nb_cache_", suffix=".tmp", dir=os.path.dirname(path) or ".")
    os.close(tmp_fd)
    try:
        with open(tmp_path, "wb") as f:
            pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp_path, path)
    finally:
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except Exception:
                pass

_CACHE_HIT = False
_CACHE_FULL_HIT = False
_cached_snapshot = None
_FRED_STATUS = {
    "requested": [],
    "loaded": [],
    "skipped_no_key": [],
    "failed_provider": [],
}

def _fred_fetch(tkr, start, end):
    """
    Pull a FRED series via the direct JSON API — no pandas_datareader required.
    Returns a pd.Series indexed by date, or raises on failure.
    FRED API docs: https://fred.stlouisfed.org/docs/api/fred/series_observations.html
    """
    import requests
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id":         tkr,
        "api_key":           FRED_API_KEY,
        "file_type":         "json",
        "observation_start": pd.Timestamp(start).strftime("%Y-%m-%d"),
        "observation_end":   pd.Timestamp(end).strftime("%Y-%m-%d"),
    }
    resp = requests.get(url, params=params, timeout=FRED_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    if "error_message" in data:
        raise ValueError(f"FRED API error for '{tkr}': {data['error_message']}")
    obs = data.get("observations", [])
    if not obs:
        raise ValueError(f"FRED returned no observations for '{tkr}'")
    dates  = pd.to_datetime([o["date"] for o in obs])
    values = pd.to_numeric([o["value"] for o in obs], errors="coerce")
    s = pd.Series(values, index=dates, name=tkr)
    return s.dropna()

if CACHE_ENABLED and os.path.exists(_CACHE_PATH):
    try:
        with open(_CACHE_PATH, "rb") as f:
            _cached_snapshot = pickle.load(f)
        _min_keys = {"prices", "ASSET_META", "ASSET_ORDER"}
        _miss = _min_keys - set(_cached_snapshot.keys())
        if _miss:
            raise ValueError(f"Cache missing keys: {_miss}")
        prices = _cached_snapshot["prices"].copy()
        ASSET_META = dict(_cached_snapshot["ASSET_META"])
        ASSET_ORDER = list(_cached_snapshot["ASSET_ORDER"])
        _CACHE_HIT = True
        _CACHE_FULL_HIT = all(
            k in _cached_snapshot for k in [
                "event_returns", "ALL_LABELS", "ALL_CLASSES",
                "CLASS_TO_ASSETS", "TRIGGER_PERCENTILE_MAP", "FRED_STATUS"
            ]
        )
        mode = "FULL" if _CACHE_FULL_HIT else "PRICES-ONLY"
        print(f"  [cache] HIT ({mode}) → {_CACHE_PATH}")
    except Exception as _ce:
        print(f"  [cache] MISS (invalid cache: {_ce})")
        _CACHE_HIT = False
        _CACHE_FULL_HIT = False
else:
    print(f"  [cache] MISS (no cache file)")


# ── yfinance ─────────────────────────────────────────────────
yf_labels  = [l for l in ASSET_ORDER if ASSET_META[l]["source"]=="yf"]
yf_tickers = list(dict.fromkeys([ASSET_META[l]["ticker"] for l in yf_labels]))
if VIX_TICKER not in yf_tickers: yf_tickers.append(VIX_TICKER)

if _CACHE_HIT:
    print("  [cache] Using cached prices; skipping yfinance/FRED pulls")
    raw_yf = pd.DataFrame(index=prices.index)
    raw_fred = pd.DataFrame(index=prices.index)
    fred_labels = [l for l in ASSET_ORDER if ASSET_META.get(l,{}).get("source")=="fred"]
    fred_tickers = [ASSET_META[l]["ticker"] for l in fred_labels]
    _FRED_STATUS = dict(_cached_snapshot.get("FRED_STATUS", {
        "requested": list(fred_labels),
        "loaded": [],
        "skipped_no_key": [],
        "failed_provider": [],
    }))
else:
    # USDCNH fallback — test against a recent fixed window, not global_start.
    # global_start can go back to the 1970s (due to MANUAL_DATA events), and CNH
    # didn't trade freely until 2010, causing yfinance to hang on old date ranges.
    _cnh_test_end   = pd.Timestamp.today().normalize()
    _cnh_test_start = _cnh_test_end - timedelta(days=10)
    usdcnh_ticker_used = None
    for fb in USDCNH_FALLBACKS:
        test = yf.download(fb, start=_cnh_test_start, end=_cnh_test_end,
                           auto_adjust=True, progress=False)
        if not test.empty and len(test) > 0:
            usdcnh_ticker_used = fb
            break
    if usdcnh_ticker_used and usdcnh_ticker_used != "USDCNH=X":
        print(f"  ⚠️  USDCNH=X unavailable — using fallback: {usdcnh_ticker_used}")
        ASSET_META["USDCNH"]["ticker"] = usdcnh_ticker_used
        yf_tickers = [usdcnh_ticker_used if t=="USDCNH=X" else t for t in yf_tickers]
    elif not usdcnh_ticker_used:
        print("  ⚠️  All USDCNH fallbacks failed — USDCNH will be skipped")

    print(f"  yfinance: {len(yf_tickers)} tickers …")
    raw_yf = pd.DataFrame()
    for _yf_attempt in range(3):
        try:
            _dl = yf.download(yf_tickers, start=global_start, end=global_end,
                              auto_adjust=True, progress=False)
            if "Close" in _dl.columns or (hasattr(_dl.columns, "levels")
                                           and "Close" in _dl.columns.get_level_values(0)):
                raw_yf = _dl["Close"]
            else:
                raw_yf = _dl   # single-ticker returns flat frame
            if isinstance(raw_yf, pd.Series):
                raw_yf = raw_yf.to_frame(yf_tickers[0])
            raw_yf.index = pd.to_datetime(raw_yf.index)
            if not raw_yf.empty and len(raw_yf) > 5:
                break
            raise ValueError(f"yfinance returned only {len(raw_yf)} rows — retrying")
        except Exception as _ye:
            if _yf_attempt < 2:
                print(f"  ↻  yfinance attempt {_yf_attempt+1} failed: {_ye} — retrying …")
                time.sleep(4)
            else:
                print(f"  ⚠️  yfinance failed after 3 attempts: {_ye}")
                if raw_yf.empty:
                    raw_yf = pd.DataFrame()
    print(f"  yfinance: {raw_yf.shape[1] if not raw_yf.empty else 0} columns, "
          f"{len(raw_yf)} rows")

    # ── FRED ─────────────────────────────────────────────────────
    # Only FRED-only series remain: breakevens (T10YIE, T5YIE, DFII10)
    # and OAS spreads (BAML*). Single attempt per series.
    # A warning banner is shown below if any series fail.

    # _fred_fetch: single pull with real HTTP timeout via requests.Session.
    # pandas_datareader uses requests internally — socket.setdefaulttimeout()
    # has no effect on it. We mount a TimeoutHTTPAdapter instead.
    def _fred_fetch(tkr, start, end):
        """
        Pull a FRED series via the direct JSON API — no pandas_datareader required.
        Returns a pd.Series indexed by date, or raises on failure.
        FRED API docs: https://fred.stlouisfed.org/docs/api/fred/series_observations.html
        """
        import requests
        url = "https://api.stlouisfed.org/fred/series/observations"
        params = {
            "series_id":         tkr,
            "api_key":           FRED_API_KEY,
            "file_type":         "json",
            "observation_start": pd.Timestamp(start).strftime("%Y-%m-%d"),
            "observation_end":   pd.Timestamp(end).strftime("%Y-%m-%d"),
        }
        resp = requests.get(url, params=params, timeout=FRED_TIMEOUT)
        resp.raise_for_status()
        data = resp.json()
        if "error_message" in data:
            raise ValueError(f"FRED API error for '{tkr}': {data['error_message']}")
        obs = data.get("observations", [])
        if not obs:
            raise ValueError(f"FRED returned no observations for '{tkr}'")
        dates  = pd.to_datetime([o["date"] for o in obs])
        values = pd.to_numeric([o["value"] for o in obs], errors="coerce")
        s = pd.Series(values, index=dates, name=tkr)
        return s.dropna()

    fred_labels  = [l for l in ASSET_ORDER if ASSET_META[l]["source"]=="fred"]
    fred_tickers = list(dict.fromkeys([ASSET_META[l]["ticker"] for l in fred_labels]))
    fred_label_by_ticker = {ASSET_META[l]["ticker"]: l for l in fred_labels}
    raw_fred = pd.DataFrame()
    _FRED_STATUS = {
        "requested": [fred_label_by_ticker.get(t, t) for t in fred_tickers],
        "loaded": [],
        "skipped_no_key": [],
        "failed_provider": [],
    }
    if fred_tickers:
        print(f"  FRED: {len(fred_tickers)} series …")
        if not globals().get("_fred_key_configured", lambda *_: False)():
            _FRED_STATUS["skipped_no_key"] = list(_FRED_STATUS["requested"])
        else:
            for tkr in fred_tickers:
                _lbl = fred_label_by_ticker.get(tkr, tkr)
                try:
                    raw_fred[tkr] = _fred_fetch(tkr, global_start, global_end)
                    _FRED_STATUS["loaded"].append(_lbl)
                except Exception:
                    _FRED_STATUS["failed_provider"].append(_lbl)
            if _FRED_STATUS["loaded"]:
                raw_fred.index = pd.to_datetime(raw_fred.index)
                raw_fred = raw_fred.ffill()
        print(
            f"  FRED summary: requested={len(_FRED_STATUS['requested'])}"
            f" | loaded={len(_FRED_STATUS['loaded'])}"
            f" | skipped_no_key={len(_FRED_STATUS['skipped_no_key'])}"
            f" | failed_provider={len(_FRED_STATUS['failed_provider'])}"
        )
        if _FRED_STATUS["skipped_no_key"]:
            print(f"    skipped_no_key: {_FRED_STATUS['skipped_no_key']}")
        if _FRED_STATUS["failed_provider"]:
            print(f"    failed_provider: {_FRED_STATUS['failed_provider']}")

if _CACHE_HIT:
    prices = prices.sort_index().replace([np.inf, -np.inf], np.nan)
    print(f"\nPrice frame (cache): {prices.shape[1]} assets × {prices.shape[0]} rows")
else:
    # ── Unified price frame ───────────────────────────────────────
    full_idx = raw_yf.index
    if not raw_fred.empty:
        full_idx = full_idx.union(raw_fred.index)
    full_idx = full_idx.sort_values()

    prices = pd.DataFrame(index=full_idx)
    for label in ASSET_ORDER:
        meta = ASSET_META[label]
        tkr  = meta["ticker"]
        if meta["source"]=="yf":
            if tkr in raw_yf.columns:
                prices[label] = raw_yf[tkr].reindex(full_idx).ffill()
            else:
                print(f"  ⚠️  Missing yf: {tkr} ({label})")
        else:
            if tkr in raw_fred.columns:
                prices[label] = raw_fred[tkr].reindex(full_idx).ffill()
            else:
                print(f"  ⚠️  Missing FRED: {tkr} ({label})")

    if VIX_TICKER in raw_yf.columns:
        prices[VIX_LABEL] = raw_yf[VIX_TICKER].reindex(full_idx).ffill()
        if VIX_LABEL not in ASSET_META:
            ASSET_META[VIX_LABEL] = {"ticker":VIX_TICKER,"class":"Volatility","source":"yf",
                                      "invert":True,"is_fred_price":False,"is_rates_bp":False}

    prices = prices.sort_index().dropna(how="all")
    print(f"\nPrice frame: {prices.shape[1]} assets × {prices.shape[0]} rows")

# ── Helpers ───────────────────────────────────────────────────
def resolve_day0(evt_date_str, idx):
    dt = pd.Timestamp(evt_date_str)
    c  = idx[idx >= dt]
    return c[0] if len(c) else None

def unit_label(label):
    return "Δbps" if ASSET_META.get(label,{}).get("is_rates_bp") else "%"

def display_label(label):
    """Return chart-display name (e.g. USDEUR instead of EURUSD after flip)."""
    return ASSET_META.get(label, {}).get("display_label", label)


def compute_event_series(label, evt_name, evt_date_str):
    if label not in prices.columns: return None
    meta = ASSET_META.get(label, {})
    d0   = resolve_day0(evt_date_str, prices.index)
    if d0 is None: return None
    col  = prices[label].dropna()
    if d0 not in col.index: return None
    idx0 = col.index.get_loc(d0)
    denom = col.iloc[idx0-1] if idx0 > 0 else col.iloc[idx0]
    if denom == 0 or np.isnan(denom): return None
    ws = max(0, idx0 - PRE_WINDOW_TD - 5)
    we = min(len(col), idx0 + POST_WINDOW_TD + 5)
    window = col.iloc[ws:we]
    if meta.get("is_rates_bp"):
        ret = (window - denom) * 100
    else:
        ret = (window / denom - 1) * 100
        if not meta.get("invert", True): ret = -ret
    offsets = list(range(ws - idx0, we - idx0))
    n = min(len(offsets), len(ret.values))
    s = pd.Series(ret.values[:n], index=offsets[:n])
    return s[~s.index.duplicated()]

if _CACHE_FULL_HIT:
    event_returns = _cached_snapshot["event_returns"]
    ALL_LABELS = list(_cached_snapshot["ALL_LABELS"])
    ALL_CLASSES = list(_cached_snapshot["ALL_CLASSES"])
    CLASS_TO_ASSETS = dict(_cached_snapshot["CLASS_TO_ASSETS"])
    TRIGGER_PERCENTILE_MAP = dict(_cached_snapshot["TRIGGER_PERCENTILE_MAP"])
    print(f"  [cache] Reused event_returns + trigger map for {len(EVENTS)} events")
else:
    # ── Build event_returns ───────────────────────────────────────
    ALL_LABELS    = [l for l in ASSET_ORDER if l in prices.columns] or list(ASSET_ORDER)
    event_returns = {l: {} for l in ALL_LABELS}
    for _ei, (en, ed) in enumerate(EVENTS):
        if _ei == 0 or (_ei + 1) % 5 == 0 or (_ei + 1) == len(EVENTS):
            print(f"  Building event_returns {_ei+1}/{len(EVENTS)}: {en}")
        _stage_check(f"event_returns:{en}")
        for l in ALL_LABELS:
            s = compute_event_series(l, en, ed)
            if s is not None: event_returns[l][en] = s

    def poi_ret(label, evt_name, offset, tol=2):
        s = event_returns.get(label,{}).get(evt_name)
        if s is None: return np.nan
        if offset in s.index: return float(s.loc[offset])
        nb = s.index[np.abs(np.array(s.index)-offset)<=tol]
        return float(s.loc[nb[0]]) if len(nb) else np.nan

    def get_subset(mode="all", cls=None, group=None, manual=None):
        if mode=="class"  and cls:    return [l for l in ALL_LABELS if ASSET_META.get(l,{}).get("class")==cls]
        if mode=="group"  and group:  return [l for l in CUSTOM_GROUPS.get(group,[]) if l in prices.columns]
        if mode=="manual" and manual: return [l for l in manual if l in prices.columns]
        return [l for l in ALL_LABELS if l in ASSET_META]

    ALL_CLASSES = sorted(set(ASSET_META[l]["class"] for l in ALL_LABELS))
    CLASS_TO_ASSETS = {c: [l for l in ALL_LABELS if ASSET_META[l]["class"]==c] for c in ALL_CLASSES}


# ── Trigger fallback guardrail ───────────────────────────────
if TRIGGER_ASSET not in ALL_LABELS:
    _fallback_candidates = [TRIGGER_ASSET] + globals().get("SIMILARITY_ASSET_POOL", []) + [
        "WTI Crude (spot)", "Brent Futures", "S&P 500", "US 10Y Yield", "DXY"
    ]
    _fallback = next((a for a in _fallback_candidates if a in ALL_LABELS), None)
    if _fallback:
        _old_trigger = TRIGGER_ASSET
        TRIGGER_ASSET = _fallback
        if "SIMILARITY_ASSET_POOL" in globals():
            SIMILARITY_ASSET_POOL = [TRIGGER_ASSET] + [a for a in SIMILARITY_ASSET_POOL if a != TRIGGER_ASSET]
        print(f"  ⚠️  TRIGGER_ASSET '{_old_trigger}' unavailable; fallback → '{TRIGGER_ASSET}'")
    else:
        print("  ⚠️  No usable fallback trigger asset found in ALL_LABELS")


# Continue with event reconstruction regardless of fallback outcome
# The block below stays indented to minimize notebook JSON churn.
if True:
    # ── INJECT MANUAL_DATA events (generalised reconstruction pattern) ──
    # Handles any event whose price history is hand-sourced rather than pulled
    # from yfinance/FRED (e.g. pre-1985 events). Defined in §1.3.
    # Print a clean line so the  progress counter doesn't obscure what runs next
    print(f"  Building event_returns: done ({len(EVENTS)} events)")
    print("  Injecting MANUAL_DATA events …")
    for _evt_name, _evt_cfg in MANUAL_DATA.items():
        _evt_date = _evt_cfg["date"]
        _assets   = _evt_cfg["assets"].copy()

        # S&P 500 special case: Yahoo carries ^GSPC back to 1970, pull it live.
        # Wrapped in a thread timeout — historical yf.download can hang on old dates.
        if "S&P 500" not in _assets:
            try:
                import threading
                _d0s    = pd.Timestamp(_evt_date)
                _sr_box = [None]
                def _gspc_pull():
                    try:
                        _sr_box[0] = yf.download(
                            "^GSPC",
                            start=_d0s - timedelta(days=60),
                            end=_d0s + timedelta(days=POST_WINDOW_TD*2),
                            auto_adjust=True, progress=False)["Close"]
                    except Exception:
                        pass
                _t = threading.Thread(target=_gspc_pull, daemon=True)
                _t.start(); _t.join(timeout=20)
                _sr = _sr_box[0]
                if _sr is None:
                    print(f"  ⚠️  ^GSPC pull for '{_evt_name}' timed out — skipping S&P 500")
                    raise ValueError("timeout")
                _sr.index = pd.to_datetime(_sr.index)
                _bds = _sr.index[_sr.index >= _d0s]
                if len(_bds):
                    _ad0  = _bds[0]
                    _prd  = _sr.index[_sr.index < _ad0]
                    if len(_prd):
                        _den  = float(_sr.loc[_prd[-1]])
                        _idx  = list(_sr.index)
                        _d0p  = _idx.index(_ad0)
                        _spx  = {}
                        for _pl, _po in _POI_MAP.items():
                            _tp = _d0p + _po
                            if 0 <= _tp < len(_idx):
                                _spx[_po] = (float(_sr.iloc[_tp]) / _den - 1) * 100
                        if _spx:
                            _assets["S&P 500"] = _spx
                            print(f"  ✅  S&P 500 for '{_evt_name}': {len(_spx)} POI values")
            except Exception as _e:
                print(f"  ⚠️  S&P 500 pull for '{_evt_name}' failed: {_e}")

        # Inject into event_returns
        for _lbl, _ret_dict in _assets.items():
            if _lbl not in event_returns:
                event_returns[_lbl] = {}
            event_returns[_lbl][_evt_name] = pd.Series(_ret_dict, dtype=float)

        # Register in EVENTS if not already present
        if (_evt_name, _evt_date) not in EVENTS:
            EVENTS.append((_evt_name, _evt_date))

        print(f"  ✅  '{_evt_name}' injected ({len(_assets)} assets)")

    EVENTS.sort(key=lambda x: pd.Timestamp(x[1]))

    # ── TRIGGER_PERCENTILE_MAP ────────────────────────────────────
    # For each historical event, record the percentile rank of TRIGGER_ASSET's
    # Day-0 price within a rolling 5-year window of the TRIGGER_ASSET's own
    # price history ending at Day 0.
    #
    # WHY: raw nominal price ratios break down across very different monetary
    # regimes. Brent at $11 in 1998 and $91 in 2022 have a ratio of 0.12 —
    # implying extreme dissimilarity. But in percentile terms, both may be
    # at moderate levels relative to their own era's price distribution.
    # Percentile rank is scale-invariant and avoids this distortion.
    #
    # METHOD (5-year rolling window, ~1260 trading days):
    #   - Uses prices *ending at Day 0* to avoid look-ahead bias.
    #   - score = 1 - |live_pctile - hist_pctile| / 100
    #   - Perfect match (same regime position) → 1.0
    #   - Opposite extremes (0th vs 100th pctile) → 0.0
    #
    # FALLBACK for pre-1985 events (MANUAL_DATA with †):
    #   Uses the MACRO_CONTEXT trigger value (nominal price) cross-ranked
    #   against the full available price history of TRIGGER_ASSET.
    #   The 1973 Oil Embargo at ~$4/bbl maps to ~2nd pctile of all Brent
    #   history — correctly flagging it as "ultra-cheap oil era".
    #
    # LIVE EVENT: LIVE["trigger_pctile"] is set in pull_live_data() using
    #   the same rolling-window method.

    TRIGGER_PERCENTILE_MAP = {}  # event_name → float percentile (0–100)

    def _compute_trigger_zscore(evt_date_str, window_td=1260):
        """Z-score of TRIGGER_ASSET price at Day 0 within a rolling
        window_td-trading-day window (default 5 years) ending at Day 0.

        Why z-score over percentile rank:
          Percentile rank compresses the tails — the 95th and 99th pctile
          look nearly identical (distance 4/100 = 0.04) even though they may
          represent genuinely different regimes (e.g. "elevated" vs "blow-off").
          Z-score preserves tail sensitivity: an event 3 standard deviations
          above mean is correctly identified as more extreme than 2 std devs.

        The similarity kernel in _macro_sim() uses a Gaussian:
            trigger_score = exp(-0.5 * (z_live - z_hist)^2 / SIGMA^2)
        with SIGMA = TRIGGER_ZSCORE_SIGMA (config in §1.6, default 1.5).
        This gives a smooth 0–1 score that peaks at 1.0 when z-scores match
        exactly and decays naturally as regimes diverge.

        Returns: float z-score (unbounded), or None if insufficient data.
        """
        if TRIGGER_ASSET not in prices.columns:
            return None
        col = prices[TRIGGER_ASSET].dropna()
        d0  = resolve_day0(evt_date_str, col.index)
        if d0 is None or d0 not in col.index:
            return None
        idx0  = col.index.get_loc(d0)
        start = max(0, idx0 - window_td)
        hist  = col.iloc[start:idx0]        # exclude Day 0 itself from reference window
        if len(hist) < 20:
            return None
        mu  = float(hist.mean())
        sig = float(hist.std())
        if sig < 1e-9:
            return 0.0
        p0  = float(col.loc[d0])
        return round((p0 - mu) / sig, 3)

    # Pass 1: compute z-scores from prices where available
    for _en, _ed in EVENTS:
        z = _compute_trigger_zscore(_ed)
        TRIGGER_PERCENTILE_MAP[_en] = z     # stores z-score; None for pre-1985 events

    # Pass 2: for events still None (pre-1985 / MANUAL_DATA), derive z-score
    # from the MACRO_CONTEXT trigger value cross-ranked against modern history.
    # We standardise using the full available price history (mu, sigma) so the
    # z-score is on the same scale as the modern events above.
    if TRIGGER_ASSET in prices.columns:
        _full = prices[TRIGGER_ASSET].dropna()
        _full_mu  = float(_full.mean())
        _full_sig = float(_full.std()) or 1.0
        for _en in list(TRIGGER_PERCENTILE_MAP.keys()):
            if TRIGGER_PERCENTILE_MAP[_en] is None:
                _mc_val = MACRO_CONTEXT.get(_en, {}).get("trigger",
                          MACRO_CONTEXT.get(_en, {}).get("oil", None))
                if _mc_val and _mc_val > 0:
                    TRIGGER_PERCENTILE_MAP[_en] = round((_mc_val - _full_mu) / _full_sig, 3)
                else:
                    TRIGGER_PERCENTILE_MAP[_en] = 0.0   # neutral (at-mean) fallback

    # Handle MANUAL_DATA events not yet mapped
    for _en, _cfg in MANUAL_DATA.items():
        if _en not in TRIGGER_PERCENTILE_MAP:
            _mc_val = MACRO_CONTEXT.get(_en, {}).get("trigger",
                      MACRO_CONTEXT.get(_en, {}).get("oil", None))
            if _mc_val and _mc_val > 0 and TRIGGER_ASSET in prices.columns:
                _full = prices[TRIGGER_ASSET].dropna()
                _z = (_mc_val - float(_full.mean())) / (float(_full.std()) or 1.0)
                TRIGGER_PERCENTILE_MAP[_en] = round(_z, 3)
            else:
                TRIGGER_PERCENTILE_MAP[_en] = 0.0

    n_mapped = sum(1 for v in TRIGGER_PERCENTILE_MAP.values() if v is not None)
    print(f"  ✅  TRIGGER_ZSCORE_MAP: {n_mapped}/{len(EVENTS)} events mapped  (window=5yr, σ-kernel)")
    _sorted = sorted(TRIGGER_PERCENTILE_MAP.items(), key=lambda x: x[1] if x[1] is not None else -1e9)
    _preview = _sorted[:3] + _sorted[-2:] if len(_sorted) > 5 else _sorted
    for _en, _z in _preview:
        _z_txt = f"{_z:+.2f}" if _z is not None else "n/a"
        print(f"      {_en:<35s} → z = {_z_txt}")



# Ensure helper accessors exist even on full-cache short-circuit paths.
if "poi_ret" not in globals():
    def poi_ret(label, evt_name, offset, tol=2):
        s = event_returns.get(label,{}).get(evt_name)
        if s is None:
            return np.nan
        if offset in s.index:
            return float(s.loc[offset])
        nb = s.index[np.abs(np.array(s.index)-offset)<=tol]
        return float(s.loc[nb[0]]) if len(nb) else np.nan

if "get_subset" not in globals():
    def get_subset(mode="all", cls=None, group=None, manual=None):
        if mode=="class"  and cls:
            return [l for l in ALL_LABELS if ASSET_META.get(l,{}).get("class")==cls]
        if mode=="group"  and group:
            return [l for l in CUSTOM_GROUPS.get(group,[]) if l in prices.columns]
        if mode=="manual" and manual:
            return [l for l in manual if l in prices.columns]
        return [l for l in ALL_LABELS if l in ASSET_META]

# ── FRED warning banner ──────────────────────────────────────
if fred_tickers:
    from IPython.display import display, HTML
    _fred_req = _FRED_STATUS.get("requested", [])
    _fred_loaded = _FRED_STATUS.get("loaded", [])
    _fred_skipped = _FRED_STATUS.get("skipped_no_key", [])
    _fred_failed = _FRED_STATUS.get("failed_provider", [])
    if _fred_skipped:
        display(HTML(
            "<div style='background:#1f2937;border:1px solid #58a6ff;border-radius:6px;"
            "padding:10px 14px;margin:8px 0;font-family:monospace;font-size:13px'>"
            "<b style='color:#79c0ff'>ℹ  FRED disabled — series skipped</b><br>"
            "<span style='color:#c9d1d9'>Requested: "
            + str(len(_fred_req)) + " | Loaded: " + str(len(_fred_loaded)) +
            " | Skipped (no key): " + str(len(_fred_skipped)) + "</span><br>"
            "<span style='color:#8b949e'>Set FRED_API_KEY as a local environment variable to enable these series.</span></div>"
        ))
    elif _fred_failed:
        display(HTML(
            "<div style='background:#3d1f00;border:1px solid #e3b341;border-radius:6px;"
            "padding:10px 14px;margin:8px 0;font-family:monospace;font-size:13px'>"
            "<b style='color:#e3b341'>⚠  FRED data partially unavailable</b><br>"
            "<span style='color:#ffa657'>Failed provider series: "
            + ", ".join(_fred_failed) +
            "</span><br>"
            "<span style='color:#8b949e'>Requested: "
            + str(len(_fred_req)) + " | Loaded: " + str(len(_fred_loaded)) +
            " | Failed: " + str(len(_fred_failed)) +
            ". All yfinance data is unaffected.</span></div>"
        ))
    else:
        display(HTML(
            "<div style='background:#0d1f0d;border:1px solid #3fb950;border-radius:6px;"
            "padding:8px 14px;margin:8px 0;font-family:monospace;font-size:13px'>"
            "<b style='color:#3fb950'>✅  All FRED series pulled successfully</b></div>"
        ))

print("\n✅ Data ready.")

try:
    _stage_check("cache-save")
    if CACHE_ENABLED:
        _snapshot = {
            "prices": prices,
            "ASSET_META": ASSET_META,
            "ASSET_ORDER": ASSET_ORDER,
        }
        if CACHE_FULL_SNAPSHOT:
            _snapshot.update({
                "event_returns": event_returns,
                "ALL_LABELS": ALL_LABELS,
                "ALL_CLASSES": ALL_CLASSES,
                "CLASS_TO_ASSETS": CLASS_TO_ASSETS,
                "TRIGGER_PERCENTILE_MAP": TRIGGER_PERCENTILE_MAP,
                "FRED_STATUS": _FRED_STATUS,
            })
        _atomic_pickle_save(_CACHE_PATH, _snapshot)
        print(f"  [cache] SAVED → {_CACHE_PATH}")
except Exception as _cache_save_err:
    print(f"  [cache] SAVE FAILED: {_cache_save_err}")

In [ ]:
# §3.2 — DATA AUDIT
# ─────────────────────────────────────────────────────────────
# Run after §3.1. Surfaces data quality issues before charting:
#   • Assets with <50% coverage across the event universe
#   • Events with fewer than 3 assets with data at t0
#   • POI offsets that fall outside the pulled data window for any event
#   • TRIGGER_ASSET data availability check
# Does not modify any data — read-only diagnostic.

_warn = []

# Asset coverage: % of events with at least one non-NaN return at t0
_all_events = [n for n, _ in EVENTS]
print("Asset coverage at t0 (across all events):")
def _low_cov_cause(lbl, pct):
    meta = ASSET_META.get(lbl, {})
    if meta.get("source") == "fred":
        if lbl in (_FRED_STATUS.get("skipped_no_key", []) or []):
            return "FRED disabled"
        if lbl in (_FRED_STATUS.get("failed_provider", []) or []):
            return "provider unavailable"
        return "structurally limited historical coverage"
    if lbl not in prices.columns:
        return "provider unavailable"
    if pct <= 0:
        return "provider unavailable"
    return "structurally limited historical coverage"

_low_cov = []
for lbl in ASSET_ORDER:
    if lbl not in event_returns: continue
    has_data = sum(1 for en in _all_events if en in event_returns[lbl]
                   and not pd.isna(event_returns[lbl][en].get(0, float('nan'))))
    pct = has_data / len(_all_events) * 100
    if pct < 50:
        _low_cov.append((lbl, pct, _low_cov_cause(lbl, pct)))

if _low_cov:
    print("  ⚠️  Low coverage (<50% of events have t0 data):")
    for lbl, pct, cause in sorted(_low_cov, key=lambda x: x[1]):
        print(f"    {display_label(lbl):<30s} {pct:5.0f}%  |  {cause}")
else:
    print("  ✅ All assets ≥50% coverage")

# Event coverage: how many assets have data at t0 per event
print("\nEvent coverage at t0 (how many assets have data):")
_sparse_events = []
for en, _ in EVENTS:
    n = sum(1 for lbl in ASSET_ORDER
            if lbl in event_returns and en in event_returns[lbl]
            and not pd.isna(event_returns[lbl][en].get(0, float('nan'))))
    if n < 5:
        _sparse_events.append((en, n))
        print(f"  ⚠️  {en:<35s} only {n} assets at t0")

if not _sparse_events:
    print("  ✅ All events have ≥5 assets at t0")

# TRIGGER_ASSET availability
ta_missing = [en for en, _ in EVENTS
              if TRIGGER_ASSET not in event_returns
              or en not in event_returns.get(TRIGGER_ASSET, {})]
if ta_missing:
    print(f"\n  ⚠️  TRIGGER_ASSET '{TRIGGER_ASSET}' missing for: {ta_missing}")
else:
    print(f"\n  ✅ TRIGGER_ASSET '{TRIGGER_ASSET}' available for all events")

# SIMILARITY_ASSET_POOL availability
print("\nSimilarity pool availability:")
for a in SIMILARITY_ASSET_POOL:
    n = sum(1 for en, _ in EVENTS if a in event_returns and en in event_returns.get(a, {}))
    flag = "✅" if n >= len(EVENTS) * 0.7 else "⚠️ "
    print(f"  {flag} {display_label(a):<30s} {n}/{len(EVENTS)} events")

print("\n✅ Data audit complete.")


In [ ]:
# §3.3 — CACHE STATUS
# ─────────────────────────────────────────────────────────────
# Cache now runs directly inside §3.1 (cache-first).
# This cell is only a quick status panel.

from pathlib import Path
import os

_cp = globals().get("_CACHE_PATH", None)
_ch = globals().get("_CACHE_HIT", None)
_cf = globals().get("_CACHE_FULL_HIT", None)

print("Cache status")
print(f"  path: {_cp if _cp else 'n/a (run §3.1 first)'}")
print(f"  hit:  {_ch}")
print(f"  full: {_cf}")

if _cp and os.path.exists(_cp):
    p = Path(_cp)
    print(f"  size: {p.stat().st_size:,} bytes")
    print(f"  mtime: {p.stat().st_mtime}")
else:
    print("  file: missing")

---
## § 4 · CHARTS
> Pure chart-builder functions. Each takes data arguments and returns a `go.Figure`.  
> No widget state, no side effects. Safe to re-run independently.  
> All charts read from `event_returns`, `ASSET_META`, `EVENTS`, and `POIS`.


In [ ]:
def build_overlay_fig(asset_label, anchor_mode="day0", step_day=0):
    """
    anchor_mode: "day0"   → zero at Day 0 (step_day ignored)
                 "stepin" → re-zero at step_day, show full path
    † events: plotted as lines+markers at POI offsets only.
              Y-axis range is set from non-† events; † traces clip at axis
              boundary (cliponaxis=True) so outliers don't compress the scale.
              Full values still visible on hover.
    """
    unit       = unit_label(asset_label)
    cls_       = ASSET_META.get(asset_label, {}).get("class", "")
    offsets    = list(range(-PRE_WINDOW_TD, POST_WINDOW_TD + 1))
    poi_offsets = [o for _, o in POIS]
    poi_labels  = [l for l, _ in POIS]
    fig        = go.Figure()
    has_recon  = False  # track if any † event exists for this asset

    # ── Collect non-† values to set axis range ────────────────
    non_recon_vals = []
    for en, _ in EVENTS:
        if en.endswith("†"): continue
        s = event_returns.get(asset_label, {}).get(en)
        if s is None: continue
        s_plot = s.reindex(offsets)
        if anchor_mode == "stepin" and step_day != 0:
            aidx = s.index[np.abs(np.array(s.index) - step_day) <= 2]
            av   = float(s.loc[aidx[0]]) if len(aidx) else 0.0
            s_plot = s_plot - av
        non_recon_vals.extend(s_plot.dropna().tolist())

    # Axis range: 2nd–98th pctile of non-† data + 15% padding
    # † traces will clip at this boundary (values still in hover)
    if non_recon_vals:
        y_lo  = np.percentile(non_recon_vals, 2)
        y_hi  = np.percentile(non_recon_vals, 98)
        pad   = max((y_hi - y_lo) * 0.15, 1.0)
        y_range = [y_lo - pad, y_hi + pad]
    else:
        y_range = None

    # ── Draw all events ───────────────────────────────────────
    for j, (en, _) in enumerate(EVENTS):
        _recon = en.endswith("†")
        s = event_returns.get(asset_label, {}).get(en)
        if s is None: continue

        color = EVENT_COLORS[j % len(EVENT_COLORS)]
        dash  = EVENT_DASHES[j % len(EVENT_DASHES)]

        if _recon:
            has_recon = True
            poi_xs = sorted([o for o in poi_offsets if o in s.index])
            poi_ys = []
            for o in poi_xs:
                val = float(s.loc[o])
                if anchor_mode == "stepin" and step_day != 0:
                    aidx = s.index[np.abs(np.array(s.index) - step_day) <= 2]
                    av   = float(s.loc[aidx[0]]) if len(aidx) else 0.0
                    val  = val - av
                poi_ys.append(val)
            fig.add_trace(go.Scatter(
                x=poi_xs, y=poi_ys,
                name=en, mode="lines+markers",
                line=dict(color=color, dash=dash, width=1.4),
                marker=dict(size=7, color=color, symbol="circle",
                            line=dict(color="#ffffff", width=0.8)),
                cliponaxis=True,   # clip to axis range, don't expand scale
                hovertemplate=f"<b>{en}</b> †<br>Day %{{x}}: %{{y:.2f}} {unit}<extra></extra>",
            ))
        else:
            s_plot = s.reindex(offsets)
            if anchor_mode == "stepin" and step_day != 0:
                aidx = s.index[np.abs(np.array(s.index) - step_day) <= 2]
                av   = float(s.loc[aidx[0]]) if len(aidx) else 0.0
                s_plot = s_plot - av
            fig.add_trace(go.Scatter(
                x=offsets, y=s_plot.values,
                name=en, mode="lines",
                line=dict(color=color, dash=dash, width=1.8),
                hovertemplate=f"<b>{en}</b><br>Day %{{x}}: %{{y:.2f}} {unit}<extra></extra>",
            ))

    # ── Reference lines ───────────────────────────────────────
    fig.add_vline(x=0, line=dict(color=COL_ZERO, width=1.5, dash="dash"),
                  annotation_text="Day 0", annotation_font=dict(color=COL_ZERO))
    if anchor_mode == "stepin" and step_day != 0:
        fig.add_vline(x=step_day,
                      line=dict(color="#ffa657", width=1.5, dash="dash"),
                      annotation_text=f"Step-in D+{step_day}",
                      annotation_font=dict(color="#ffa657"))
    fig.add_hline(y=0, line=dict(color=COL_MUTED, width=0.6))
    for poi_l, poi_o in POIS:
        if poi_o != 0:
            fig.add_vline(x=poi_o,
                          line=dict(color=COL_GRID, width=0.8, dash="dot"),
                          annotation_text=poi_l,
                          annotation_font=dict(color=COL_MUTED, size=9))

    anchor_note  = f" | anchored at Step-in D+{step_day}" if anchor_mode == "stepin" else ""
    recon_note   = "  <i style=\'font-size:10px;color:#8b949e\'>(† clipped — hover for full value)</i>" if has_recon else ""

    yaxis_cfg = dict(
        title=f"Return ({unit})",
        gridcolor=COL_GRID, zerolinecolor=COL_GRID,
        linecolor=COL_GRID, tickcolor=COL_MUTED,
        tickfont=dict(color=COL_MUTED),
    )
    # No range restriction — show the full graph including † outliers

    layout = dark_layout(
        xaxis=dict(title="Trading days from Day 0",
                   tickvals=poi_offsets, ticktext=poi_labels,
                   gridcolor=COL_GRID, zerolinecolor=COL_GRID,
                   linecolor=COL_GRID, tickcolor=COL_MUTED,
                   tickfont=dict(color=COL_MUTED)),
        yaxis=yaxis_cfg,
        title=f"<b>{display_label(asset_label)}</b> [{cls_}]{anchor_note}{recon_note}",
        legend=dict(orientation="v", x=1.01, y=1,
                    bgcolor=BG_PANEL, bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT, size=10)),
        hovermode="x unified",
        width=CHART_W, height=CHART_H,
        margin=dict(r=240),
    )
    fig.update_layout(**layout)
    return fig

# ── Widgets ───────────────────────────────────────────────────
w1_cls   = widgets.Dropdown(options=ALL_CLASSES, description="Class:",
                             layout=widgets.Layout(width="200px"),
                             style={"description_width": "50px"})
w1_asset = widgets.Dropdown(description="Asset:",
                             layout=widgets.Layout(width="260px"),
                             style={"description_width": "50px"})
w1_prev  = widgets.Button(description="◀", layout=widgets.Layout(width="40px"))
w1_next  = widgets.Button(description="▶", layout=widgets.Layout(width="40px"))
w1_anchor = widgets.ToggleButtons(
    options=[("Day-0 anchor", "day0"), ("Step-in anchor", "stepin")],
    value="day0", description="",
    layout=widgets.Layout(width="300px"),
)
w1_slider = widgets.IntSlider(
    value=0, min=-PRE_WINDOW_TD, max=POST_WINDOW_TD,
    step=1, description="Step-in:",
    continuous_update=False,
    layout=widgets.Layout(width="480px"),
    style={"description_width": "70px"},
)
w1_slider_row = widgets.HBox([w1_slider])
w1_slider_row.layout.visibility = "hidden"
w1_out = widgets.Output()
w1_state = make_refresh_state()

def w1_update_assets(cls_val):
    opts = [l for l in CLASS_TO_ASSETS.get(cls_val, []) if l in event_returns]
    cur = w1_asset.value if w1_asset.value in opts else None
    w1_asset.options = opts
    if cur is not None:
        w1_asset.value = cur
    elif opts:
        w1_asset.value = opts[0]


def w1_refresh(*_):
    asset = w1_asset.value
    if not asset:
        return
    key = (w1_cls.value, asset, w1_anchor.value, int(w1_slider.value))
    run_guarded_refresh(
        w1_state,
        key,
        lambda: out_fig(build_overlay_fig(asset, w1_anchor.value, w1_slider.value), w1_out),
        label="Overlay chart",
    )


def w1_on_cls(change):
    if change.get("name") != "value":
        return
    w1_update_assets(change["new"])
    w1_refresh()


def w1_on_asset(change):
    if change.get("name") == "value":
        w1_refresh()


def w1_on_anchor(change):
    if change.get("name") != "value":
        return
    w1_slider_row.layout.visibility = "visible" if change["new"] == "stepin" else "hidden"
    w1_refresh()


def w1_on_slider(change):
    if change.get("name") == "value":
        w1_refresh()


def w1_prev_click(_):
    opts = list(w1_asset.options)
    if not opts:
        return
    i = opts.index(w1_asset.value) if w1_asset.value in opts else 0
    w1_asset.value = opts[(i - 1) % len(opts)]


def w1_next_click(_):
    opts = list(w1_asset.options)
    if not opts:
        return
    i = opts.index(w1_asset.value) if w1_asset.value in opts else 0
    w1_asset.value = opts[(i + 1) % len(opts)]


w1_cls.observe(w1_on_cls, names="value")
w1_asset.observe(w1_on_asset, names="value")
w1_anchor.observe(w1_on_anchor, names="value")
w1_slider.observe(w1_on_slider, names="value")
w1_prev.on_click(w1_prev_click)
w1_next.on_click(w1_next_click)

w1_update_assets(w1_cls.value)
w1_state["ready"] = True
w1_refresh()


In [ ]:
# §4.2 — CHART: HEATMAP
# Returns a go.Figure heatmap of return at each POI for each event.
# Colour scale: RdYlGn (green=positive) reversed for rates (green=falling yield).
# Controls: class→asset selector
#           Day-0 / Step-in anchor toggle + slider
# ============================================================

def build_heatmap_fig(asset_label, anchor_mode="day0", step_day=0):
    poi_lbls  = [l for l, _ in POIS]
    evt_lbls  = [e[0] for e in EVENTS]
    unit      = unit_label(asset_label)
    is_rates  = ASSET_META.get(asset_label, {}).get("is_rates_bp", False)
    cls_      = ASSET_META.get(asset_label, {}).get("class", "")

    mat = np.full((len(evt_lbls), len(poi_lbls)), np.nan)
    for i, ev in enumerate(evt_lbls):
        for j, (pl, po) in enumerate(POIS):
            raw = poi_ret(asset_label, ev, po)
            if anchor_mode == "stepin" and step_day != 0 and not np.isnan(raw):
                anchor_val = poi_ret(asset_label, ev, step_day)
                raw = raw - anchor_val if not np.isnan(anchor_val) else np.nan
            mat[i, j] = raw

    show_text = len(evt_lbls) <= 18
    text_mat = None
    hovertemplate = "<b>%{y}</b><br>%{x}: %{z:+.1f} " + unit + "<extra></extra>"
    if show_text:
        text_mat = [[f"{mat[i,j]:+.1f}" if not np.isnan(mat[i,j]) else "N/A"
                     for j in range(len(poi_lbls))] for i in range(len(evt_lbls))]
        hovertemplate = "<b>%{y}</b><br>%{x}: %{text} " + unit + "<extra></extra>"

    vmax   = np.nanmax(np.abs(mat)) if not np.all(np.isnan(mat)) else 5
    cscale = "RdYlGn_r" if is_rates else "RdYlGn"

    _hm_kwargs = dict(
        z=mat, x=poi_lbls, y=evt_lbls,
        colorscale=cscale, zmid=0, zmin=-vmax, zmax=vmax,
        colorbar=dict(title=dict(text=unit, font=dict(color=COL_TEXT)), thickness=14,
                      tickfont=dict(color=COL_TEXT)),
        hovertemplate=hovertemplate,
    )
    if show_text:
        _hm_kwargs["text"] = text_mat
        _hm_kwargs["texttemplate"] = "%{text}"

    fig = go.Figure(go.Heatmap(**_hm_kwargs))

    anchor_note = f" | re-zeroed at D+{step_day}" if anchor_mode=="stepin" else ""
    fig.update_layout(**dark_layout(
        title=f"<b>{display_label(asset_label)}</b> [{cls_}] — {unit}{anchor_note}",
        xaxis=dict(title="Horizon (POI)", gridcolor=COL_GRID,
                   linecolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title="Event", autorange="reversed",
                   gridcolor=COL_GRID, linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        width=CHART_W, height=max(420, len(evt_lbls) * 42 + 100),
        margin=dict(t=70, b=40),
    ))
    return fig


In [ ]:
# ── Widgets ───────────────────────────────────────────────────
w2_cls    = widgets.Dropdown(options=ALL_CLASSES, description="Class:",
                              layout=widgets.Layout(width="200px"),
                              style={"description_width": "50px"})
w2_asset  = widgets.Dropdown(description="Asset:",
                              layout=widgets.Layout(width="260px"),
                              style={"description_width": "50px"})
w2_anchor = widgets.ToggleButtons(
    options=[("Day-0 anchor", "day0"), ("Step-in anchor", "stepin")],
    value="day0", description="",
    layout=widgets.Layout(width="300px"),
)
w2_slider = widgets.IntSlider(
    value=0, min=-PRE_WINDOW_TD, max=POST_WINDOW_TD,
    step=1, description="Step-in:",
    continuous_update=False,
    layout=widgets.Layout(width="480px"),
    style={"description_width": "70px"},
)
w2_slider_row = widgets.HBox([w2_slider])
w2_slider_row.layout.visibility = "hidden"
w2_out = widgets.Output()
w2_state = make_refresh_state()

def w2_update_assets(cls_val):
    opts = [l for l in CLASS_TO_ASSETS.get(cls_val, []) if l in event_returns]
    cur = w2_asset.value if w2_asset.value in opts else None
    w2_asset.options = opts
    if cur is not None:
        w2_asset.value = cur
    elif opts:
        w2_asset.value = opts[0]


def w2_refresh(*_):
    if not w2_asset.value:
        return
    key = (w2_cls.value, w2_asset.value, w2_anchor.value, int(w2_slider.value))
    run_guarded_refresh(
        w2_state,
        key,
        lambda: out_fig(build_heatmap_fig(w2_asset.value, w2_anchor.value, w2_slider.value), w2_out),
        label="Heatmap chart",
    )


def w2_on_class(change):
    if change.get("name") != "value":
        return
    w2_update_assets(change["new"])
    w2_refresh()


def w2_on_asset(change):
    if change.get("name") == "value":
        w2_refresh()


def w2_on_anchor(change):
    if change.get("name") != "value":
        return
    w2_slider_row.layout.visibility = "visible" if change["new"] == "stepin" else "hidden"
    w2_refresh()


def w2_on_slider(change):
    if change.get("name") == "value":
        w2_refresh()


w2_cls.observe(w2_on_class, names="value")
w2_asset.observe(w2_on_asset, names="value")
w2_anchor.observe(w2_on_anchor, names="value")
w2_slider.observe(w2_on_slider, names="value")
w2_update_assets(w2_cls.value)
w2_state["ready"] = True
if not globals().get("AUTOMATION_MODE", False):
    w2_refresh()


In [ ]:
# §4.3 — CHART: REGIME SCATTER
# Returns a go.Figure scatter of asset-X vs asset-Y return at a chosen POI.
# Step-forward mode re-bases both axes from a chosen step-in day.
# Controls: class→asset for X and Y
#           all POIs incl t+3M
#           step-forward slider
# ============================================================

_SCATTER_FIG_CACHE = {"key": None, "fig": None}

def build_scatter_fig(asset_x, asset_y, poi_offset, step_day=None):
    cache_key = (
        asset_x,
        asset_y,
        poi_offset,
        step_day,
        len(EVENTS),
        len(event_returns.get(asset_x, {})),
        len(event_returns.get(asset_y, {})),
    )
    if _SCATTER_FIG_CACHE["key"] == cache_key and _SCATTER_FIG_CACHE["fig"] is not None:
        return _SCATTER_FIG_CACHE["fig"]

    poi_lbl = next((l for l, o in POIS if o == poi_offset), str(poi_offset))
    ux, uy  = unit_label(asset_x), unit_label(asset_y)
    fig     = go.Figure()

    xs, ys, evs, cols = [], [], [], []
    for j, (en, _) in enumerate(EVENTS):
        if step_day is None:
            x = poi_ret(asset_x, en, poi_offset)
            y = poi_ret(asset_y, en, poi_offset)
        else:
            xs0 = poi_ret(asset_x, en, step_day)
            ys0 = poi_ret(asset_y, en, step_day)
            xp  = poi_ret(asset_x, en, poi_offset)
            yp  = poi_ret(asset_y, en, poi_offset)
            x   = xp - xs0 if not any(np.isnan([xp, xs0])) else np.nan
            y   = yp - ys0 if not any(np.isnan([yp, ys0])) else np.nan
        if np.isnan(x) or np.isnan(y):
            continue
        xs.append(float(x))
        ys.append(float(y))
        evs.append(en)
        cols.append(EVENT_COLORS[j % len(EVENT_COLORS)])

    if xs:
        fig.add_trace(go.Scatter(
            x=xs,
            y=ys,
            mode="markers",
            text=evs,
            marker=dict(
                color=cols,
                size=10,
                line=dict(color=BG_PANEL, width=1),
            ),
            hovertemplate=(f"<b>%{{text}}</b><br>{asset_x}: %{{x:.2f}} {ux}"
                           f"<br>{asset_y}: %{{y:.2f}} {uy}<extra></extra>"),
            showlegend=False,
        ))

    fig.add_hline(y=0, line=dict(color=COL_MUTED, width=0.8, dash="dash"))
    fig.add_vline(x=0, line=dict(color=COL_MUTED, width=0.8, dash="dash"))

    step_note = f" | Fwd from D+{step_day}" if step_day is not None else ""
    fig.update_layout(**dark_layout(
        title=f"<b>{display_label(asset_x)}</b> vs <b>{display_label(asset_y)}</b> @ {poi_lbl}{step_note}",
        xaxis=dict(title=f"{display_label(asset_x)} ({ux})", gridcolor=COL_GRID,
                   zerolinecolor=COL_ZERO, linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title=f"{display_label(asset_y)} ({uy})", gridcolor=COL_GRID,
                   zerolinecolor=COL_ZERO, linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        width=CHART_W, height=CHART_H,
        margin=dict(r=60),
    ))
    _SCATTER_FIG_CACHE["key"] = cache_key
    _SCATTER_FIG_CACHE["fig"] = fig
    return fig

# ── Widgets ───────────────────────────────────────────────────
w3_xcls   = widgets.Dropdown(options=ALL_CLASSES, description="X Class:",
                               layout=widgets.Layout(width="190px"),
                               style={"description_width": "60px"})
w3_xasset = widgets.Dropdown(description="X:",
                               layout=widgets.Layout(width="220px"),
                               style={"description_width": "25px"})
w3_ycls   = widgets.Dropdown(options=ALL_CLASSES, description="Y Class:",
                               layout=widgets.Layout(width="190px"),
                               style={"description_width": "60px"})
w3_yasset = widgets.Dropdown(description="Y:",
                               layout=widgets.Layout(width="220px"),
                               style={"description_width": "25px"})
w3_poi    = widgets.Dropdown(
    options=[(f"{l} (D{o:+d})", o) for l, o in POIS],
    value=0, description="Horizon:",
    layout=widgets.Layout(width="200px"),
    style={"description_width": "65px"},
)
w3_step_toggle = widgets.ToggleButton(
    value=False, description="Step-forward", button_style="",
    layout=widgets.Layout(width="130px"),
)
w3_slider = widgets.IntSlider(
    value=0, min=-PRE_WINDOW_TD, max=POST_WINDOW_TD - 5,
    step=1, description="Step-in:",
    continuous_update=False,
    layout=widgets.Layout(width="480px"),
    style={"description_width": "70px"},
)
w3_slider_row = widgets.HBox([w3_slider])
w3_slider_row.layout.visibility = "hidden"
w3_out = widgets.Output()
w3_state = make_refresh_state()

def w3_update_assets(cls_val, dd):
    opts = [l for l in CLASS_TO_ASSETS.get(cls_val, []) if l in event_returns]
    cur = dd.value if dd.value in opts else None
    dd.options = opts
    if cur is not None:
        dd.value = cur
    elif opts:
        dd.value = opts[0]


def w3_refresh(*_):
    ax_, ay_ = w3_xasset.value, w3_yasset.value
    poi_off  = w3_poi.value
    step_day = w3_slider.value if w3_step_toggle.value else None
    if not (ax_ and ay_ and poi_off is not None):
        return
    key = (w3_xcls.value, ax_, w3_ycls.value, ay_, poi_off, bool(w3_step_toggle.value), int(w3_slider.value))
    run_guarded_refresh(
        w3_state,
        key,
        lambda: out_fig(build_scatter_fig(ax_, ay_, poi_off, step_day), w3_out),
        label="Scatter chart",
    )


def w3_on_xcls(change):
    if change.get("name") != "value":
        return
    w3_update_assets(change["new"], w3_xasset)
    w3_refresh()


def w3_on_ycls(change):
    if change.get("name") != "value":
        return
    w3_update_assets(change["new"], w3_yasset)
    w3_refresh()


def w3_on_step(change):
    if change.get("name") != "value":
        return
    w3_slider_row.layout.visibility = "visible" if change["new"] else "hidden"
    w3_refresh()


w3_xcls.observe(w3_on_xcls, names="value")
w3_ycls.observe(w3_on_ycls, names="value")
w3_xasset.observe(lambda change: w3_refresh() if change.get("name") == "value" else None, names="value")
w3_yasset.observe(lambda change: w3_refresh() if change.get("name") == "value" else None, names="value")
w3_poi.observe(lambda change: w3_refresh() if change.get("name") == "value" else None, names="value")
w3_slider.observe(lambda change: w3_refresh() if change.get("name") == "value" else None, names="value")
w3_step_toggle.observe(w3_on_step, names="value")

# Defaults: WTI vs S&P 500
w3_xcls.value = "Oil & Energy"
w3_update_assets("Oil & Energy", w3_xasset)
if "WTI Crude (spot)" in w3_xasset.options:
    w3_xasset.value = "WTI Crude (spot)"
w3_ycls.value = "Equities"
w3_update_assets("Equities", w3_yasset)
if "S&P 500" in w3_yasset.options:
    w3_yasset.value = "S&P 500"
w3_state["ready"] = True
if not globals().get("AUTOMATION_MODE", False):
    w3_refresh()


In [ ]:
# §4.4 — CHART: RETURN PATH
# Full continuous return path for each event across the analysis window.
# Excludes manually-reconstructed events (†) since they only have POI snapshots.
# Default anchor: Day 0. Step-in slider re-zeros at chosen day.
# Full path always shown (nothing hidden).
# ============================================================

def build_path_fig(asset_label, anchor_mode="day0", step_day=0):
    unit     = unit_label(asset_label)
    cls_     = ASSET_META.get(asset_label, {}).get("class", "")
    post_max = max(o for _, o in POIS if o >= 0)
    pre_min  = min(o for _, o in POIS if o <= 0)
    fig      = go.Figure()

    for j, (en, _) in enumerate(EVENTS):
        if en.endswith("†"): continue
        s = event_returns[asset_label].get(en)
        if s is None: continue
        # Full window — never hidden
        s_win = s[(s.index >= pre_min) & (s.index <= post_max)].sort_index()
        if len(s_win) < 2: continue

        if anchor_mode == "stepin" and step_day != 0:
            ai = s.index[np.abs(np.array(s.index) - step_day) <= 2]
            anchor_val = float(s.loc[ai[0]]) if len(ai) else 0.0
        else:
            anchor_val = 0.0   # Day 0 is already zero by construction

        s_plot = s_win - anchor_val
        c = EVENT_COLORS[j % len(EVENT_COLORS)]
        d = EVENT_DASHES[j % len(EVENT_DASHES)]

        fig.add_trace(go.Scattergl(
            x=list(s_plot.index), y=list(s_plot.values),
            name=en, mode="lines",
            line=dict(color=c, dash=d, width=1.8),
            hovertemplate=f"<b>{en}</b><br>Day %{{x}}: %{{y:.2f}} {unit}<extra></extra>",
        ))

    # Reference lines
    fig.add_hline(y=0, line=dict(color=COL_MUTED, width=0.7, dash="dash"))
    fig.add_vline(x=0, line=dict(color=COL_ZERO, width=1.2, dash="dash"),
                  annotation_text="Day 0",
                  annotation_font=dict(color=COL_ZERO))
    if anchor_mode == "stepin" and step_day != 0:
        fig.add_vline(x=step_day,
                      line=dict(color="#ffa657", width=1.2, dash="dash"),
                      annotation_text=f"Step-in D+{step_day}",
                      annotation_font=dict(color="#ffa657"))

    poi_offsets = [o for _, o in POIS]
    poi_lbl_map = {o: l for l, o in POIS}
    for po in poi_offsets:
        fig.add_vline(x=po, line=dict(color=COL_GRID, width=0.8, dash="dot"))

    anchor_note = f" | anchored at D+{step_day}" if anchor_mode=="stepin" else ""
    fig.update_layout(**dark_layout(
        title=f"<b>{display_label(asset_label)}</b> [{cls_}] — Return Path{anchor_note}",
        xaxis=dict(title="Trading days from Day 0",
                   tickvals=poi_offsets,
                   ticktext=[poi_lbl_map.get(o, str(o)) for o in poi_offsets],
                   gridcolor=COL_GRID, zerolinecolor=COL_GRID,
                   linecolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title=f"{unit} from anchor",
                   gridcolor=COL_GRID, zerolinecolor=COL_ZERO,
                   linecolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        legend=dict(orientation="v", x=1.01, y=1,
                    bgcolor=BG_PANEL, bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT, size=10)),
        hovermode="x unified",
        width=CHART_W, height=CHART_H,
        margin=dict(r=240),
    ))
    return fig

# ── Widgets ───────────────────────────────────────────────────
w4_cls    = widgets.Dropdown(options=ALL_CLASSES, description="Class:",
                              layout=widgets.Layout(width="200px"),
                              style={"description_width": "50px"})
w4_asset  = widgets.Dropdown(description="Asset:",
                              layout=widgets.Layout(width="260px"),
                              style={"description_width": "50px"})
w4_anchor = widgets.ToggleButtons(
    options=[("Day-0 anchor", "day0"), ("Step-in anchor", "stepin")],
    value="day0", description="",
    layout=widgets.Layout(width="300px"),
)
w4_slider = widgets.IntSlider(
    value=0, min=-PRE_WINDOW_TD, max=POST_WINDOW_TD - 5,
    step=1, description="Step-in:",
    continuous_update=False,
    layout=widgets.Layout(width="480px"),
    style={"description_width": "70px"},
)
w4_slider_row = widgets.HBox([w4_slider])
w4_slider_row.layout.visibility = "hidden"
w4_out = widgets.Output()
w4_state = {"busy": False, "last": None, "ready": False}

def w4_update_assets(cls_val):
    opts = [l for l in CLASS_TO_ASSETS.get(cls_val, []) if l in event_returns]
    cur = w4_asset.value if w4_asset.value in opts else None
    w4_asset.options = opts
    if cur is not None:
        w4_asset.value = cur
    elif opts:
        w4_asset.value = opts[0]

def w4_refresh(*_):
    if not w4_state["ready"] or w4_state["busy"] or not w4_asset.value:
        return
    key = (w4_cls.value, w4_asset.value, w4_anchor.value, int(w4_slider.value))
    if key == w4_state["last"]:
        return
    w4_state["busy"] = True
    try:
        with w4_out:
            clear_output(wait=True)
            build_path_fig(w4_asset.value, w4_anchor.value, w4_slider.value).show()
        w4_state["last"] = key
    finally:
        w4_state["busy"] = False

def w4_on_class(change):
    if change.get("name") != "value":
        return
    w4_update_assets(change["new"])
    w4_refresh()

def w4_on_asset(change):
    if change.get("name") == "value":
        w4_refresh()

def w4_on_anchor(change):
    if change.get("name") != "value":
        return
    w4_slider_row.layout.visibility = "visible" if change["new"] == "stepin" else "hidden"
    w4_refresh()

def w4_on_slider(change):
    if change.get("name") == "value":
        w4_refresh()

w4_cls.observe(w4_on_class, names="value")
w4_asset.observe(w4_on_asset, names="value")
w4_anchor.observe(w4_on_anchor, names="value")
w4_slider.observe(w4_on_slider, names="value")
w4_update_assets(w4_cls.value)
w4_state["ready"] = True
if not globals().get("AUTOMATION_MODE", False):
    w4_refresh()

In [ ]:
# §4.5 — CHART: VIX PATH
# Absolute VIX level across the full window per event.
# Overlays median ± IQR band. Today vline added if LIVE data is loaded.
# Single panel: absolute VIX level across full window per event.
# POI markers as vertical lines. Median + IQR band overlaid.
# ============================================================

_VIX_FIG_CACHE = {"key": None, "fig": None}

def build_vix_fig():
    if VIX_LABEL not in prices.columns:
        print("VIX not available."); return None

    _live = globals().get("LIVE", {})
    _is_ov = bool(globals().get("wl_day_override_chk") and globals()["wl_day_override_chk"].value)
    cache_key = (
        tuple((en, str(ed)) for en, ed in EVENTS),
        len(prices.index),
        _live.get("name"),
        _live.get("day_n"),
        _is_ov,
    )
    if _VIX_FIG_CACHE.get("key") == cache_key and _VIX_FIG_CACHE.get("fig") is not None:
        return go.Figure(_VIX_FIG_CACHE["fig"])

    poi_offsets = [o for _, o in POIS]
    poi_lbl_map = {o: l for l, o in POIS}
    pre_min  = min(poi_offsets) - 2
    post_max = max(poi_offsets) + 2

    fig        = go.Figure()
    vix_at_poi = []
    poi_x_all, poi_y_all, poi_text_all, poi_color_all = [], [], [], []

    for j, (en, ed) in enumerate(EVENTS):
        if en.endswith("†"):
            continue
        d0 = resolve_day0(ed, prices.index)
        if d0 is None or d0 not in prices.index:
            continue
        idx0 = prices.index.get_loc(d0)

        xs, ys = [], []
        for off in range(pre_min, post_max + 1):
            ti = idx0 + off
            if 0 <= ti < len(prices.index):
                xs.append(off)
                ys.append(float(prices[VIX_LABEL].iloc[ti]))

        lvls = []
        for off in poi_offsets:
            ti = idx0 + off
            val = float(prices[VIX_LABEL].iloc[ti]) if 0 <= ti < len(prices.index) else np.nan
            lvls.append(val)
            if not np.isnan(val):
                poi_x_all.append(off)
                poi_y_all.append(val)
                poi_text_all.append(en)
                poi_color_all.append(EVENT_COLORS[j % len(EVENT_COLORS)])
        vix_at_poi.append(lvls)

        fig.add_trace(go.Scattergl(
            x=xs, y=ys, name=en, mode="lines",
            line=dict(color=EVENT_COLORS[j % len(EVENT_COLORS)], dash=EVENT_DASHES[j % len(EVENT_DASHES)], width=1.5),
            hovertemplate=f"<b>{en}</b><br>Day %{{x}}: VIX %{{y:.1f}}<extra></extra>",
        ))

    if poi_x_all:
        fig.add_trace(go.Scattergl(
            x=poi_x_all, y=poi_y_all, mode="markers", name="POI markers",
            marker=dict(color=poi_color_all, size=7, symbol="circle", line=dict(color=BG_PANEL, width=1)),
            text=poi_text_all,
            hovertemplate="<b>%{text}</b> POI<br>Day %{x}: VIX %{y:.1f}<extra></extra>",
            showlegend=False,
        ))

    if vix_at_poi:
        arr  = np.array(vix_at_poi, dtype=float)
        med  = np.nanmedian(arr, axis=0)
        q25  = np.nanpercentile(arr, 25, axis=0)
        q75  = np.nanpercentile(arr, 75, axis=0)
        xs_p = poi_offsets
        fig.add_trace(go.Scatter(
            x=xs_p + xs_p[::-1],
            y=list(q75) + list(q25[::-1]),
            fill="toself", fillcolor="rgba(255,255,255,0.18)",
            line=dict(color="rgba(255,255,255,0.35)", width=1),
            name="IQR band", showlegend=True,
        ))
        fig.add_trace(go.Scatter(
            x=xs_p, y=med, name="Median",
            mode="lines+markers",
            line=dict(color="#ffffff", width=2.5, dash="dash"),
            marker=dict(symbol="diamond", size=8, color="#ffffff"),
        ))

    fig.add_vline(x=0, line=dict(color=COL_ZERO, width=1.5, dash="dash"),
                  annotation_text="Day 0",
                  annotation_font=dict(color=COL_ZERO))
    if _live.get("day_n") is not None:
        _dn   = _live["day_n"]
        _ln   = _live.get("name", "Live")
        _vcol = "#d29922" if _is_ov else "#ffa657"
        _vlbl = f"Day+{_dn} override ({_ln})" if _is_ov else f"Today D+{_dn} ({_ln})"
        fig.add_vline(x=_dn,
                      line=dict(color=_vcol, width=2, dash="dot"),
                      annotation_text=_vlbl,
                      annotation_font=dict(color=_vcol, size=10))
    for po, pl in poi_lbl_map.items():
        if po != 0:
            fig.add_vline(x=po, line=dict(color=COL_GRID, width=0.8, dash="dot"),
                          annotation_text=pl,
                          annotation_font=dict(color=COL_MUTED, size=9))

    fig.update_layout(**dark_layout(
        title="VIX — Absolute Level | All Events | POI markers + Median ± IQR",
        xaxis=dict(title="Trading days from Day 0",
                   tickvals=poi_offsets,
                   ticktext=[poi_lbl_map[o] for o in poi_offsets],
                   gridcolor=COL_GRID, zerolinecolor=COL_GRID,
                   linecolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title="VIX Level",
                   gridcolor=COL_GRID, zerolinecolor=COL_GRID,
                   linecolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        legend=dict(orientation="v", x=1.01, y=1,
                    bgcolor=BG_PANEL, bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT, size=9)),
        hovermode="x unified",
        width=CHART_W, height=580,
        margin=dict(r=220),
    ))
    _VIX_FIG_CACHE["key"] = cache_key
    _VIX_FIG_CACHE["fig"] = go.Figure(fig)
    return fig

if not globals().get("AUTOMATION_MODE", False):
    fig5 = build_vix_fig()
    if fig5:
        fig5.show()

In [ ]:
# §4.6 — CHART: BOX & WHISKER
# Distribution of returns at selected POIs across all events.
# One subplot per POI. Individual event dots overlaid with jitter.
# Thicker box borders, wider boxes, event dots overlaid.
# Controls: group/class selector, POI multi-select, band toggle.
# ============================================================

def build_box_fig(labels, poi_lbls_sel, show_band=False):
    sel_pois = [(l, o) for l, o in POIS if l in poi_lbls_sel]
    if not sel_pois or not labels: return go.Figure()

    n_cols = len(sel_pois)
    fig    = make_subplots(
        rows=1, cols=n_cols,
        subplot_titles=[f"Horizon: {l}" for l, _ in sel_pois],
        horizontal_spacing=0.06,
    )

    for k, (poi_lbl, poi_off) in enumerate(sel_pois):
        for lbl in labels:
            vals = [poi_ret(lbl, e[0], poi_off) for e in EVENTS]
            vals = [v for v in vals if not np.isnan(v)]
            if not vals: continue
            cls_  = ASSET_META.get(lbl, {}).get("class", "")
            color = CLASS_COLORS.get(cls_, "#58a6ff")

            # Box trace — thicker borders, wider
            fig.add_trace(go.Box(
                y=vals, name=lbl,
                marker=dict(color=color, size=6,
                            line=dict(color=BG_PANEL, width=1)),
                line=dict(color=color, width=2.5),   # thick border
                fillcolor=color.replace("#", "rgba(") if False else color,
                opacity=0.35,
                boxmean="sd",
                width=0.65,                           # wide box
                hovertemplate=f"<b>{lbl}</b><br>%{{y:.2f}}<extra></extra>",
                legendgroup=lbl, showlegend=(k == 0),
            ), row=1, col=k + 1)

            # Individual event dots (jitter)
            fig.add_trace(go.Box(
                y=vals, name=lbl,
                boxpoints="all", jitter=0.4, pointpos=0,
                marker=dict(color=color, size=5, opacity=0.7,
                            line=dict(color=BG_PANEL, width=0.5)),
                line=dict(color="rgba(0,0,0,0)"),
                fillcolor="rgba(0,0,0,0)",
                whiskerwidth=0,
                showlegend=False,
                legendgroup=lbl,
                width=0.65,
                hovertemplate=f"<b>{lbl}</b><br>%{{y:.2f}}<extra></extra>",
            ), row=1, col=k + 1)

        if show_band:
            all_v = [poi_ret(lbl, e[0], poi_off)
                     for lbl in labels for e in EVENTS]
            all_v = [v for v in all_v if not np.isnan(v)]
            if all_v:
                mu, sig = np.mean(all_v), np.std(all_v)
                fig.add_hline(y=mu,
                              line=dict(color="#ffa657", width=1.5, dash="dash"),
                              annotation_text=f"μ={mu:.1f}",
                              annotation_font=dict(color="#ffa657"),
                              row=1, col=k + 1)
                fig.add_hrect(y0=mu - sig, y1=mu + sig,
                              fillcolor="#ffa657", opacity=0.05,
                              line_width=0, row=1, col=k + 1)

        fig.add_hline(y=0,
                      line=dict(color=COL_MUTED, width=0.7, dash="dash"),
                      row=1, col=k + 1)

    # Apply dark styling to all subplots
    for i in range(1, n_cols + 1):
        xkey = "xaxis" if i == 1 else f"xaxis{i}"
        ykey = "yaxis" if i == 1 else f"yaxis{i}"
        fig.update_layout(**{
            xkey: dict(gridcolor=COL_GRID, linecolor=COL_GRID,
                       tickfont=dict(color=COL_MUTED), tickangle=45),
            ykey: dict(gridcolor=COL_GRID, zerolinecolor=COL_ZERO,
                       linecolor=COL_GRID, tickfont=dict(color=COL_MUTED),
                       title="Return"),
        })

    fig.update_layout(
        paper_bgcolor=BG,
        plot_bgcolor=BG_PANEL,
        font=dict(color=COL_TEXT, family="monospace"),
        boxmode="overlay",
        legend=dict(x=1.01, y=1, bgcolor=BG_PANEL, bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT, size=9)),
        width=min(1600, max(CHART_W, len(labels) * 65 * n_cols // n_cols + 200)),
        height=CHART_H,
        margin=dict(r=220, t=60),
    )
    return fig

# ── Widgets ───────────────────────────────────────────────────
group_opts   = ["— All Assets —"] + sorted(CUSTOM_GROUPS.keys()) + ALL_CLASSES
poi_lbl_list = [l for l, _ in POIS]

w6_group = widgets.Dropdown(
    options=group_opts, value="Risk Barometer",
    description="Group:", layout=widgets.Layout(width="280px"),
    style={"description_width": "55px"},
)
w6_pois  = widgets.SelectMultiple(
    options=poi_lbl_list,
    value=[l for l in poi_lbl_list if l.startswith("t+")],
    description="POIs:",
    layout=widgets.Layout(width="180px", height="160px"),
    style={"description_width": "40px"},
)
w6_band  = widgets.Checkbox(value=False, description="Group mean±1σ band",
                             layout=widgets.Layout(width="200px"))
w6_out   = widgets.Output()
w6_state = make_refresh_state()

def w6_get_labels(grp):
    if grp == "— All Assets —":  return get_subset("all")
    if grp in CUSTOM_GROUPS:     return get_subset("group", group=grp)
    if grp in ALL_CLASSES:       return get_subset("class", cls=grp)
    return []


def w6_refresh(*_):
    lbls = w6_get_labels(w6_group.value)
    pois = list(w6_pois.value)
    key = (w6_group.value, tuple(pois), bool(w6_band.value))
    run_guarded_refresh(
        w6_state,
        key,
        lambda: out_fig(build_box_fig(lbls, pois, w6_band.value) if lbls and pois else None, w6_out),
        label="Box chart",
    )


w6_group.observe(lambda change: w6_refresh() if change.get("name") == "value" else None, names="value")
w6_pois.observe(lambda change: w6_refresh() if change.get("name") == "value" else None, names="value")
w6_band.observe(lambda change: w6_refresh() if change.get("name") == "value" else None, names="value")
w6_state["ready"] = True
if not globals().get("AUTOMATION_MODE", False):
    w6_refresh()


In [ ]:
# §4.7 — CHART: SUMMARY TABLE
# Mean ± std return at each POI across selected events.
# Cells colour-graded by magnitude and direction (green=positive for non-rates).
# Fixed: cell fill colours now use rgba() — no hex+alpha.
# ============================================================

def build_summary_df(labels):
    poi_lbls = [l for l, _ in POIS]
    rows = []
    for lbl in labels:
        cls_ = ASSET_META.get(lbl, {}).get("class", "")
        row = {"Asset": lbl, "Class": cls_}
        for poi_lbl, po in POIS:
            vals = [poi_ret(lbl, e[0], po) for e in EVENTS]
            vals = [v for v in vals if not np.isnan(v)]
            mu = np.mean(vals) if vals else np.nan
            sig = np.std(vals) if vals else np.nan
            row[poi_lbl] = f"{mu:+.1f} (±{sig:.1f})" if not np.isnan(mu) else "N/A"
        rows.append(row)
    return pd.DataFrame(rows, columns=["Asset", "Class"] + poi_lbls)

def build_summary_fig(labels):
    poi_lbls = [l for l, _ in POIS]
    header_vals = ["Asset", "Class"] + poi_lbls

    col_asset, col_class = [], []
    col_data = [[] for _ in poi_lbls]
    clr_asset, clr_class = [], []
    clr_data = [[] for _ in poi_lbls]

    all_mu = []
    for lbl in labels:
        for _, po in POIS:
            vals = [poi_ret(lbl, e[0], po) for e in EVENTS]
            vals = [v for v in vals if not np.isnan(v)]
            mu = np.mean(vals) if vals else np.nan
            if not np.isnan(mu):
                all_mu.append(abs(mu))

    vmax = max(all_mu) if all_mu else 5.0

    for lbl in labels:
        cls_ = ASSET_META.get(lbl, {}).get("class", "")
        col_asset.append(lbl)
        col_class.append(cls_)
        clr_asset.append("rgba(33,38,45,1)")
        hex_c = CLASS_COLORS.get(cls_, "#8b949e").lstrip("#")
        r0, g0, b0 = int(hex_c[0:2], 16), int(hex_c[2:4], 16), int(hex_c[4:6], 16)
        clr_class.append(f"rgba({r0},{g0},{b0},0.18)")

        for j, (_, po) in enumerate(POIS):
            vals = [poi_ret(lbl, e[0], po) for e in EVENTS]
            vals = [v for v in vals if not np.isnan(v)]
            mu = np.mean(vals) if vals else np.nan
            sig = np.std(vals) if vals else np.nan
            txt = f"{mu:+.1f} (±{sig:.1f})" if not np.isnan(mu) else "N/A"
            col_data[j].append(txt)

            if np.isnan(mu):
                clr_data[j].append("rgba(33,38,45,1)")
            else:
                intens = min(abs(mu) / (vmax + 1e-9), 1.0)
                is_rates = ASSET_META.get(lbl, {}).get("is_rates_bp", False)
                sign_good = (mu > 0) if not is_rates else (mu < 0)
                alpha = 0.15 + 0.50 * intens
                if sign_good:
                    clr_data[j].append(f"rgba(63,185,80,{alpha:.2f})")
                else:
                    clr_data[j].append(f"rgba(247,129,102,{alpha:.2f})")

    all_values = [col_asset, col_class] + col_data
    all_colours = [clr_asset, clr_class] + clr_data

    fig = go.Figure(go.Table(
        header=dict(
            values=[f"<b>{h}</b>" for h in header_vals],
            fill_color="#1c2128",
            font=dict(color=COL_HEAD, size=11, family="monospace"),
            align="center", height=34,
            line=dict(color=COL_GRID, width=1),
        ),
        cells=dict(
            values=all_values,
            fill_color=all_colours,
            align="center",
            font=dict(color=COL_HEAD, size=10, family="monospace"),
            height=28,
            line=dict(color=COL_GRID, width=0.5),
        )
    ))
    fig.update_layout(
        title=dict(text="Summary Table — μ (±σ) at each POI",
                   font=dict(color=COL_HEAD, size=13)),
        paper_bgcolor=BG,
        font=dict(color=COL_TEXT, family="monospace"),
        width=CHART_W,
        height=max(400, len(labels) * 30 + 140),
        margin=dict(t=60, b=20, l=10, r=10),
    )
    return fig

def render_summary_output(labels, out):
    if not labels:
        with out:
            clear_output(wait=True)
        return
    # Plotly tables get very slow in Jupyter when we serialize the full asset universe.
    # Keep the same figure for smaller groups, and switch to a plain dataframe display for large sets.
    if len(labels) <= 24:
        out_fig(build_summary_fig(labels), out)
        return
    df = build_summary_df(labels)
    with out:
        clear_output(wait=True)
        display(df)

# ── Widgets ───────────────────────────────────────────────────
w7_group = widgets.Dropdown(
    options=group_opts, value="— All Assets —",
    description="Group:", layout=widgets.Layout(width="280px"),
    style={"description_width": "55px"},
)
w7_out = widgets.Output()
w7_state = make_refresh_state()

def w7_refresh(*_):
    lbls = w6_get_labels(w7_group.value)
    key = (w7_group.value, len(lbls))
    run_guarded_refresh(
        w7_state,
        key,
        lambda: render_summary_output(lbls, w7_out),
        label="Summary table",
    )


w7_group.observe(lambda change: w7_refresh() if change.get("name") == "value" else None, names="value")
w7_state["ready"] = True
if not globals().get("AUTOMATION_MODE", False):
    w7_refresh()


In [ ]:
# §4.8 — CHART: STEP-IN ANALYSIS
# Forward return distributions from a chosen step-in day to each POI.
# Includes ranked table and correlation heatmap for the selected asset group.
# Controls: group/class selector, step-in slider, fwd POI select
# ============================================================

def build_stepin_returns(step_day, fwd_offsets, labels):
    results = {l: {o: [] for o in fwd_offsets} for l in labels}
    for en, _ in EVENTS:
        for l in labels:
            s = event_returns[l].get(en)
            if s is None: continue
            si = s.index[np.abs(np.array(s.index) - step_day) <= 2]
            if not len(si): continue
            step_val = float(s.loc[si[0]])
            for fo in fwd_offsets:
                if fo <= step_day: continue
                fi = s.index[np.abs(np.array(s.index) - fo) <= 2]
                if not len(fi): continue
                results[l][fo].append(float(s.loc[fi[0]]) - step_val)
    return results

def build_stepin_figs(step_day, labels, fwd_poi_lbls):
    sel_fwd = [(l, o) for l, o in POIS if l in fwd_poi_lbls and o > step_day]
    if not sel_fwd or not labels: return None, None, None
    fwd_offs = [o for _, o in sel_fwd]
    si_rets  = build_stepin_returns(step_day, fwd_offs, labels)

    # ── Box & whisker ─────────────────────────────────────────
    n_fwd   = len(sel_fwd)
    fig_box = make_subplots(rows=1, cols=n_fwd,
                             subplot_titles=[f"To {l}" for l, _ in sel_fwd],
                             horizontal_spacing=0.06)
    for k, (pl, po) in enumerate(sel_fwd):
        for lbl in labels:
            vals = si_rets[lbl].get(po, [])
            if not vals: continue
            cls_  = ASSET_META.get(lbl, {}).get("class", "")
            color = CLASS_COLORS.get(cls_, "#58a6ff")
            fig_box.add_trace(go.Box(
                y=vals, name=lbl,
                line=dict(color=color, width=2.5),
                fillcolor=color, opacity=0.35,
                width=0.65, boxmean="sd",
                marker=dict(color=color, size=5,
                            line=dict(color=BG_PANEL, width=0.5)),
                legendgroup=lbl, showlegend=(k == 0),
                hovertemplate=f"<b>{lbl}</b><br>%{{y:.2f}}<extra></extra>",
            ), row=1, col=k + 1)
            fig_box.add_trace(go.Box(
                y=vals, name=lbl,
                boxpoints="all", jitter=0.4, pointpos=0,
                marker=dict(color=color, size=5, opacity=0.65),
                line=dict(color="rgba(0,0,0,0)"),
                fillcolor="rgba(0,0,0,0)",
                whiskerwidth=0, showlegend=False, legendgroup=lbl, width=0.65,
            ), row=1, col=k + 1)
        fig_box.add_hline(y=0, line=dict(color=COL_MUTED, width=0.7, dash="dash"),
                          row=1, col=k + 1)

    for i in range(1, n_fwd + 1):
        xk = "xaxis" if i == 1 else f"xaxis{i}"
        yk = "yaxis" if i == 1 else f"yaxis{i}"
        fig_box.update_layout(**{
            xk: dict(gridcolor=COL_GRID, linecolor=COL_GRID,
                     tickfont=dict(color=COL_MUTED), tickangle=45),
            yk: dict(gridcolor=COL_GRID, zerolinecolor=COL_ZERO,
                     linecolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        })
    fig_box.update_layout(
        paper_bgcolor=BG, plot_bgcolor=BG_PANEL,
        font=dict(color=COL_TEXT, family="monospace"),
        title=dict(text=f"Step-In D+{step_day} — Forward Return Distributions",
                   font=dict(color=COL_HEAD, size=13)),
        boxmode="overlay",
        legend=dict(x=1.01, y=1, bgcolor=BG_PANEL, bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT, size=9)),
        width=min(1600, max(CHART_W, len(labels) * 65 * n_fwd // n_fwd + 200)),
        height=CHART_H, margin=dict(r=220, t=60),
    )

    # ── Ranked table ─────────────────────────────────────────
    first_off = sel_fwd[0][1]
    first_lbl = sel_fwd[0][0]
    rows = []
    for lbl in labels:
        vals = si_rets[lbl].get(first_off, [])
        if not vals: continue
        rows.append({
            "Asset": lbl, "Class": ASSET_META.get(lbl,{}).get("class",""),
            "Median": np.median(vals), "Mean": np.mean(vals),
            "Std": np.std(vals), "Min": np.min(vals), "Max": np.max(vals),
            "N": len(vals),
        })
    rows.sort(key=lambda r: r["Median"], reverse=True)

    rank_colors = []
    for r in rows:
        a = min(0.15 + abs(r["Median"]) / (max(abs(x["Median"]) for x in rows) + 1e-9) * 0.50, 0.65)
        rank_colors.append(
            f"rgba(63,185,80,{a:.2f})"   if r["Median"] > 0 else
            f"rgba(247,129,102,{a:.2f})"
        )

    fig_rank = go.Figure(go.Table(
        header=dict(
            values=[f"<b>{c}</b>" for c in
                    ["#","Asset","Class","Median","Mean","Std","Min","Max","N"]],
            fill_color="#1c2128",
            font=dict(color=COL_HEAD, size=11, family="monospace"),
            align="center", height=30,
            line=dict(color=COL_GRID, width=1),
        ),
        cells=dict(
            values=[
                list(range(1, len(rows)+1)),
                [r["Asset"]              for r in rows],
                [r["Class"]              for r in rows],
                [f"{r['Median']:+.2f}"   for r in rows],
                [f"{r['Mean']:+.2f}"     for r in rows],
                [f"{r['Std']:.2f}"       for r in rows],
                [f"{r['Min']:+.2f}"      for r in rows],
                [f"{r['Max']:+.2f}"      for r in rows],
                [r["N"]                  for r in rows],
            ],
            fill_color=[rank_colors] * 9,
            align="center",
            font=dict(color=COL_HEAD, size=10, family="monospace"),
            height=26,
            line=dict(color=COL_GRID, width=0.5),
        )
    ))
    fig_rank.update_layout(
        title=dict(text=f"Step-In D+{step_day} — Ranked → {first_lbl}",
                   font=dict(color=COL_HEAD, size=13)),
        paper_bgcolor=BG,
        font=dict(color=COL_TEXT, family="monospace"),
        width=CHART_W,
        height=max(350, len(rows) * 28 + 120),
        margin=dict(t=60, b=20, l=10, r=10),
    )

    # ── Heatmap ───────────────────────────────────────────────
    fwd_lbl_list = [l for l, _ in sel_fwd]
    mat_h = np.full((len(labels), len(sel_fwd)), np.nan)
    for i, lbl in enumerate(labels):
        for j2, (_, fo) in enumerate(sel_fwd):
            vals = si_rets[lbl].get(fo, [])
            mat_h[i, j2] = np.median(vals) if vals else np.nan

    text_h = [[f"{mat_h[i,j2]:+.1f}" if not np.isnan(mat_h[i,j2]) else "N/A"
               for j2 in range(len(sel_fwd))] for i in range(len(labels))]
    vmax_h = np.nanmax(np.abs(mat_h)) if not np.all(np.isnan(mat_h)) else 5

    fig_hm = go.Figure(go.Heatmap(
        z=mat_h, x=fwd_lbl_list, y=labels,
        text=text_h, texttemplate="%{text}",
        colorscale="RdYlGn", zmid=0, zmin=-vmax_h, zmax=vmax_h,
        colorbar=dict(title=dict(text="Median Ret", font=dict(color=COL_TEXT)), thickness=14,
                      tickfont=dict(color=COL_TEXT)),
        hovertemplate="<b>%{y}</b><br>%{x}: %{text}<extra></extra>",
    ))
    fig_hm.update_layout(**dark_layout(
        title=f"Step-In D+{step_day} — Median Forward Return Heatmap",
        xaxis=dict(title="Forward Horizon", gridcolor=COL_GRID,
                   linecolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title="Asset", autorange="reversed",
                   gridcolor=COL_GRID, linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        width=CHART_W,
        height=max(420, len(labels) * 38 + 120),
    ))
    return fig_box, fig_rank, fig_hm

# ── Widgets ───────────────────────────────────────────────────
fwd_poi_opts = [l for l, o in POIS if o > 0]

w8_group  = widgets.Dropdown(
    options=group_opts, value="Risk Barometer",
    description="Group:", layout=widgets.Layout(width="280px"),
    style={"description_width": "55px"},
)
w8_slider = widgets.IntSlider(
    value=STEP_IN_PRIMARY,
    min=-PRE_WINDOW_TD, max=POST_WINDOW_TD - 5, step=1,
    description="Step-In Day:",
    continuous_update=False,
    layout=widgets.Layout(width="550px"),
    style={"description_width": "100px"},
)
w8_pois   = widgets.SelectMultiple(
    options=fwd_poi_opts, value=fwd_poi_opts,
    description="Fwd POIs:",
    layout=widgets.Layout(width="180px", height="140px"),
    style={"description_width": "65px"},
)
w8_out = widgets.Output()
w8_state = make_refresh_state()

def w8_refresh(*_):
    lbls     = w6_get_labels(w8_group.value)
    step_day = w8_slider.value
    fwd_pois = list(w8_pois.value)
    key = (w8_group.value, int(step_day), tuple(fwd_pois))

    def _render():
        with w8_out:
            clear_output(wait=True)
            if not lbls or not fwd_pois:
                return
            fb, fr, fh = build_stepin_figs(step_day, lbls, fwd_pois)
            if fb: fb.show()
            if fr: fr.show()
            if fh: fh.show()

    run_guarded_refresh(w8_state, key, _render, label="Step-in analysis")


w8_group.observe(lambda change: w8_refresh() if change.get("name") == "value" else None, names="value")
w8_slider.observe(lambda change: w8_refresh() if change.get("name") == "value" else None, names="value")
w8_pois.observe(lambda change: w8_refresh() if change.get("name") == "value" else None, names="value")
w8_state["ready"] = True
if not globals().get("AUTOMATION_MODE", False):
    w8_refresh()


---
## § 5 · LIVE EVENT ENGINE
> Powers the real-time analogue matching workflow.  
> Run in order: **§5.1** (pull data) → **§5.2** (score analogues) → **§5.3–5.5** (charts).  
> These cells define functions only; the dashboard in §6 wires them to widgets.


In [ ]:
# §5.1 — LIVE DATA PULL
# ─────────────────────────────────────────────────────────────
# Pulls yfinance daily closes + FRED rates/credit for a live event window.
# Business-day offsets computed from actual trading calendar (not calendar days)
# to match the historical event_returns series exactly.
#
# Reads:  wl_name, wl_date, wl_tags, wl_oil, wl_cpi, wl_fed (widgets from §6)
#         ASSET_META, ASSET_ORDER, ALL_LABELS, LIVE_FRED_MAP, FRED_API_KEY
# Writes: LIVE dict — shared state consumed by §5.2–5.5 and all Live tabs
#
# LIVE dict keys:
#   name, day0, tags, trigger, cpi, fed  — event metadata
#   trigger = TRIGGER_ASSET level at Day 0 (unit depends on asset class)
#   sim_assets                        — assets used for cosine similarity
#   prices, returns                   — raw and return series for live window
#   day_n                             — current trading-day offset from Day 0
#   bday_index                        — business day calendar for offset mapping
import datetime as dt

# Legacy inline config removed in v3.
# Live pull now uses the canonical config from §1.x / §3.1 helpers.

def pull_live_data(_=None):
    try:
        day0 = pd.Timestamp(wl_date.value.strip())
    except Exception:
        wl_status.value = "❌ Invalid date — use YYYY-MM-DD"; return
    wl_status.value = "⟳  Pulling live data…"
    today      = pd.Timestamp.today().normalize()
    fetch_from = day0 - pd.tseries.offsets.BDay(5)
    fetch_to   = today + pd.Timedelta(days=1)

    # ── yfinance pull ─────────────────────────────────────
    yf_labels  = [l for l in ALL_LABELS
                    if ASSET_META.get(l,{}).get("source")=="yf"
                    and l in prices.columns]
    yf_tickers = list(dict.fromkeys([ASSET_META[l]["ticker"] for l in yf_labels]))
    try:
        raw = yf.download(yf_tickers, start=fetch_from, end=fetch_to,
                            auto_adjust=True, progress=False)["Close"]
        if isinstance(raw, pd.Series): raw = raw.to_frame(yf_tickers[0])
        raw.index = pd.to_datetime(raw.index)
    except Exception as e:
        wl_status.value = f"❌ yfinance error: {e}"; return

    bdays = raw.index[raw.index >= day0]
    if len(bdays) == 0:
        wl_status.value = "❌ No trading data from Day 0"; return
    actual_day0 = bdays[0]
    prior_dates = raw.index[raw.index < actual_day0]
    if len(prior_dates) == 0:
        wl_status.value = "❌ No prior close found"; return
    prior_date  = prior_dates[-1]
    bday_index  = raw.index  # full business day calendar for offset mapping

    live_rets = {}
    def _make_offset_series(s_fwd, denom, is_bp, invert):
        offsets = []
        for d in s_fwd.index:
            n = len(bday_index[(bday_index >= actual_day0) & (bday_index <= d)]) - 1
            offsets.append(n)
        if is_bp:
            rets = (s_fwd.values - denom) * 100
        else:
            rets = (s_fwd.values / denom - 1) * 100
            if not invert: rets = -rets
        return pd.Series(rets, index=offsets)

    for l in yf_labels:
        tkr = ASSET_META[l]["ticker"]
        if tkr not in raw.columns: continue
        s = raw[tkr].dropna()
        if prior_date not in s.index: continue
        denom = s.loc[prior_date]
        if denom == 0 or np.isnan(denom): continue
        s_fwd = s[s.index >= actual_day0]
        meta  = ASSET_META[l]
        live_rets[l] = _make_offset_series(
            s_fwd, denom, meta.get("is_rates_bp",False), meta.get("invert",True))

    n_yf = len(live_rets)

    # ── FRED pull ──────────────────────────────────────────
    n_fred = 0
    fred_status = {
        "requested": list(LIVE_FRED_MAP.keys()),
        "loaded": [],
        "skipped_no_key": [],
        "failed_provider": [],
    }
    _fred_fetch_fn = globals().get("_fred_fetch")
    _fred_key = globals().get("FRED_API_KEY", None)
    if not callable(_fred_fetch_fn):
        fred_status["failed_provider"] = list(fred_status["requested"])
    elif not globals().get("_fred_key_configured", lambda *_: False)(_fred_key):
        fred_status["skipped_no_key"] = list(fred_status["requested"])
    else:
        for lbl, fred_id in LIVE_FRED_MAP.items():
            try:
                fs = _fred_fetch_fn(fred_id, fetch_from, fetch_to)
                fs = fs.dropna()
                fs.index = pd.to_datetime(fs.index)
                fs_from  = fs[fs.index >= actual_day0]
                if len(fs_from) == 0:
                    continue
                fp = fs[fs.index < actual_day0]
                if len(fp) == 0:
                    continue
                fdenom = float(fp.iloc[-1])
                offsets = []
                for d in fs_from.index:
                    n = len(bday_index[(bday_index >= actual_day0)
                                        & (bday_index <= d)]) - 1
                    offsets.append(n)
                rets = (fs_from.values - fdenom) * 100
                live_rets[lbl] = pd.Series(rets, index=offsets)
                n_fred += 1
                fred_status["loaded"].append(lbl)
            except Exception as _fe:
                fred_status["failed_provider"].append(lbl)

    print(
        f"  Live FRED summary: requested={len(fred_status['requested'])}"
        f" | loaded={len(fred_status['loaded'])}"
        f" | skipped_no_key={len(fred_status['skipped_no_key'])}"
        f" | failed_provider={len(fred_status['failed_provider'])}"
    )
    if fred_status["skipped_no_key"]:
        print(f"    skipped_no_key: {fred_status['skipped_no_key']}")
    if fred_status["failed_provider"]:
        print(f"    failed_provider: {fred_status['failed_provider']}")

    day_n = len(bday_index[bday_index >= actual_day0]) - 1
    last_date = bday_index[bday_index >= actual_day0][-1].date() if day_n >= 0 else actual_day0.date()

    # ── Compute live trigger percentile (scale-invariant macro scoring) ──
    # Percentile rank of the current TRIGGER_ASSET price within its own
    # rolling 5-year price history ending at the live Day 0.
    # This is what _macro_sim() uses for the trigger component.
    # Compute live trigger z-score using same 5-year rolling window as §3.1.
    # Stored in LIVE["trigger_pctile"] (legacy key name — now holds z-score).
    # _macro_sim() reads this via LIVE.get("trigger_pctile") → live_z.
    _trigger_pctile = None
    if TRIGGER_ASSET in prices.columns:
        try:
            _col  = prices[TRIGGER_ASSET].dropna()
            _past = _col[_col.index <= actual_day0]
            if len(_past) >= 20:
                _start = max(0, len(_past) - 1260)   # ~5 years of trading days
                _hist  = _past.iloc[_start:]
                _mu    = float(_hist.mean())
                _sig   = float(_hist.std()) or 1.0
                _p0    = wl_oil.value
                if _p0 > 0:
                    _trigger_pctile = round((_p0 - _mu) / _sig, 3)
                    print(f"  ✅  {TRIGGER_ASSET} at Day 0: {_p0:.2f} → z = {_trigger_pctile:+.2f}  "
                          f"(5yr μ={_mu:.2f}, σ={_sig:.2f})")
        except Exception as _e:
            print(f"  ⚠️  trigger z-score computation failed: {_e}")

    LIVE.update({"name":wl_name.value.strip(),"day0":actual_day0,
                    "tags":set(wl_tags.value),"trigger":wl_oil.value,
                    "trigger_pctile": _trigger_pctile,
                    "cpi":wl_cpi.value,"fed":wl_fed.value,
                    "sim_assets":list(wl_sim_assets.value),
                    "fred_status":fred_status,
                    "prices":raw,"returns":live_rets,
                    "day_n":day_n,"bday_index":bday_index})
    # ── Day-N override: substitute computed day_n if widget is active ──
    _eff_day_n = day_n  # effective day_n for status display
    try:
        if "wl_day_override_chk" in dir() and wl_day_override_chk.value:
            _ov = int(wl_day_override_n.value)
            # Clamp to available data range
            _max_off = max((max(s.index) for s in live_rets.values() if len(s)), default=day_n)
            _ov = max(0, min(_ov, _max_off))
            LIVE["day_n"] = _ov
            _eff_day_n = _ov
    except Exception as _ov_e:
        print(f"  ⚠️  Day override ignored: {_ov_e}")
    fred_note = (
        f"  |  FRED {len(fred_status['loaded'])}/{len(fred_status['requested'])} loaded"
        + (f" | skipped(no key): {len(fred_status['skipped_no_key'])}" if fred_status['skipped_no_key'] else "")
        + (f" | failed: {len(fred_status['failed_provider'])}" if fred_status['failed_provider'] else "")
    ) if fred_status['requested'] else ""
    _ov_note  = f"  [OVERRIDE → Day+{_eff_day_n}]" if ("wl_day_override_chk" in dir() and wl_day_override_chk.value) else ""
    wl_status.value = (f"✅  {LIVE['name']}  |  Day 0: {actual_day0.date()}"
                        f"  |  Day+{_eff_day_n} (last close: {last_date})"
                        f"{_ov_note}  |  {n_yf} yf + {n_fred} FRED assets{fred_note}")
    print(wl_status.value)

# wl_refresh.on_click handled by dashboard cell

In [ ]:
# §5.2 — ANALOGUE MATCHING ENGINE
# ─────────────────────────────────────────────────────────────
# Scores every historical event against the current live event using a
# weighted composite of three similarity measures (weights from §1.6):
#
#   quant  — cosine similarity of live vs historical asset returns at Day+N
#   tag    — Jaccard similarity of event tags (EVENT_TAGS in §1.2)
#   macro  — macro context match: Brent level, CPI regime, Fed stance
#
# _sel_ev(cutoff) — canonical selector used across §5.2–5.5 and §6.
#   Returns event names with composite score >= cutoff, or all events
#   if scores have not been computed yet (safe fallback).
#
# Reads: LIVE, EVENTS, EVENT_TAGS, MACRO_CONTEXT, ANALOGUE_WEIGHTS
# Writes: _scores_cache (populated by run_analogue_match, read by _sel_ev)
from numpy.linalg import norm

def _check_live():
    if LIVE["returns"] is None:
        print("❌ Run Cell L1 and hit ⟳ Refresh Live Data first."); return False
    return True

def _cosine(a, b):
    mask = ~(np.isnan(a)|np.isnan(b))
    if mask.sum() < 2: return 0.0
    a_, b_ = a[mask], b[mask]
    d = norm(a_)*norm(b_)
    return float(np.dot(a_,b_)/d) if d > 0 else 0.0

def _tag_sim(ta, tb):
    if not ta and not tb: return 1.0
    return len(ta&tb)/len(ta|tb) if ta|tb else 0.0

def _macro_sim(trigger_val, cpi, fed, hm, evt_name=""):
    """
    Macro similarity score (0–1) between live event and a historical event.
    trigger_val : live TRIGGER_ASSET nominal level at Day 0 (for fallback only)
    cpi, fed    : live CPI regime and Fed stance strings
    hm          : MACRO_CONTEXT[event_name] dict
    evt_name    : historical event name → looks up TRIGGER_PERCENTILE_MAP

    TRIGGER COMPARISON — ROLLING Z-SCORE + GAUSSIAN KERNEL:
      Both live and historical trigger levels are expressed as z-scores
      within TRIGGER_ASSET's own rolling 5-year price history ending at
      Day 0 (stored in TRIGGER_PERCENTILE_MAP — legacy name kept).
      Similarity uses a Gaussian kernel:
          trigger_score = exp(-0.5 * (z_live - z_hist)^2 / sigma^2)
      where sigma = TRIGGER_ZSCORE_SIGMA (§1.6, default 1.5).

      Why z-score over percentile rank: percentile compresses the tails
      (95th vs 99th pctile look nearly identical). Z-score preserves tail
      sensitivity — a 3-sigma event is correctly identified as more extreme
      than a 2-sigma event. The Gaussian kernel gives smooth, continuous
      similarity decay rather than a hard linear cutoff.

      Scale-invariant: Brent $11 (1998, z≈-1.8) correctly matches other
      low-oil regimes. $91 (2022, z≈+1.2) matches elevated-oil precedents.
      Works identically for rates, FX, equities, and commodities.

    FALLBACK (if TRIGGER_PERCENTILE_MAP missing):
      Rates → absolute difference capped at 200bps.
      Other → raw price ratio min/max.

    Sub-weights: trigger 40% | cpi 35% | fed 25%
    """
    s = 0.0

    # ── Trigger: percentile-rank similarity ───────────────────
    # TRIGGER_PERCENTILE_MAP stores z-scores (legacy name kept for compatibility).
    # Both values are z-scores within a rolling 5-year window ending at Day 0.
    live_z = LIVE.get("trigger_pctile")   # set by pull_live_data()
    hist_z = (globals().get("TRIGGER_PERCENTILE_MAP", {}) or {}).get(evt_name)

    if live_z is not None and hist_z is not None:
        # Gaussian kernel: peaks at 1.0 when z-scores match, decays smoothly
        # TRIGGER_ZSCORE_SIGMA controls kernel width (see §1.6)
        sigma = TRIGGER_ZSCORE_SIGMA
        z_sim = float(np.exp(-0.5 * ((live_z - hist_z) / sigma) ** 2))
        s += z_sim * 0.40
    else:
        # Fallback: nominal comparison (missing map — shouldn't occur post §3.1)
        hv = hm.get("trigger", hm.get("oil", 0))
        if hv > 0 and trigger_val > 0:
            is_rates = ASSET_META.get(TRIGGER_ASSET, {}).get("is_rates_bp", False)
            if is_rates:
                s += max(0.0, 1.0 - abs(trigger_val - hv) / 200.0) * 0.40
            else:
                s += min(trigger_val, hv) / max(trigger_val, hv) * 0.40

    # ── CPI regime: binary match ──────────────────────────────
    s += 0.35 if cpi == hm.get("cpi", "") else 0.0

    # ── Fed stance: binary match ──────────────────────────────
    s += 0.25 if fed == hm.get("fed", "") else 0.0

    return s

def run_analogue_match():
    global _TRADE_ROWS_CACHE
    _TRADE_ROWS_CACHE = {}
    if not _check_live(): return None
    dn = LIVE["day_n"]
    sa = list(LIVE.get("sim_assets") or [])
    # Cap dn to last offset with actual data (guards against data lag / same-day close)
    _avail = set()
    for _a in sa:
        _lr = LIVE["returns"].get(_a)
        if _lr is not None:
            _avail.update(_lr.dropna().index.tolist())
    if _avail:
        dn = min(dn, max(_avail))
    lt = LIVE["tags"];  lo = LIVE["trigger"]
    lc = LIVE["cpi"];   lf = LIVE["fed"]

    def _live_val(a, dn):
        """Return live return for asset a at the closest offset <= dn (tolerance 2).
        Falls back to np.nan if no data within range.
        """
        lr = LIVE["returns"].get(a)
        if lr is None or len(lr) == 0:
            return np.nan
        idx_arr = lr.index[np.abs(np.array(lr.index, dtype=float) - dn) <= 2]
        if len(idx_arr) == 0:
            return np.nan
        below = idx_arr[idx_arr <= dn]
        best  = below[-1] if len(below) else idx_arr[0]
        return float(lr.loc[best])

    def _usable_assets(cands):
        vals = np.array([_live_val(a, dn) if a in LIVE["returns"] else np.nan for a in cands])
        mask = ~np.isnan(vals)
        return vals, mask, int(mask.sum())

    live_vec, valid_mask, n_valid = _usable_assets(sa)
    if n_valid < 3:
        print(f"⚠️  Only {n_valid}/{len(sa)} sim assets have live data at Day+{dn}.")
        print("   WITH data:    " + str([a for a, v in zip(sa, valid_mask) if v]))
        print("   WITHOUT data: " + str([a for a, v in zip(sa, valid_mask) if not v]))
        fallback = [
            a for a in [TRIGGER_ASSET, "VIX", "Gold", "DXY", "S&P 500"]
            if a in LIVE["returns"] and not np.isnan(_live_val(a, dn))
        ]
        if len(fallback) >= 3 and tuple(fallback) != tuple(sa):
            print("   ↻ Falling back to yfinance-first sim assets: " + str(fallback))
            sa = fallback
            LIVE["sim_assets"] = list(sa)
            live_vec, valid_mask, n_valid = _usable_assets(sa)
        else:
            print("   Tip: use yfinance-only sim assets — Brent Futures, VIX, Gold, DXY, S&P 500")
        if n_valid == 0:
            print("   ❌ No sim assets have data at this day offset."
                  " Check override day vs available data range.")
            return []
        if n_valid < 3:
            print(f"   Proceeding with {n_valid} asset(s) — scores approximate.")
    # ── B1: Build live path vector (Day0 → Day+dn) ──────────
    # For each asset, collect return at each integer offset 0..dn
    # using nearest-match lookup. Concatenate across all sim assets.
    def _path_vec(returns_dict, assets, dn):
        """Flat vector of returns at each offset 0..dn for all assets.
        Missing values filled with 0.0 so cosine is always defined.
        Returns (vec, offsets_used) — same shape for live and historical.
        """
        offsets = list(range(0, dn + 1))
        vecs = []
        for a in assets:
            if a not in returns_dict:
                vecs.append(np.zeros(len(offsets)))
                continue
            s = returns_dict[a]
            row = []
            for o in offsets:
                idx_arr = s.index[np.abs(np.array(s.index, dtype=float) - o) <= 1]
                if len(idx_arr):
                    below = idx_arr[idx_arr <= o]
                    best  = below[-1] if len(below) else idx_arr[0]
                    row.append(float(s.loc[best]))
                else:
                    row.append(0.0)
            vecs.append(row)
        return np.array(vecs).flatten()

    live_path_vec = _path_vec(LIVE["returns"], sa, dn)

    scores = []
    for en,_ in EVENTS:
        hv   = np.array([poi_ret(a,en,dn) for a in sa])
        q    = (_cosine(live_vec,hv)+1)/2   # point-in-time (original)

        # B1 — path cosine: compare full Day0→Day+dn trajectory
        hist_path_dict = {a: event_returns.get(a,{}).get(en, pd.Series(dtype=float)) for a in sa}
        hist_path_vec  = _path_vec(hist_path_dict, sa, dn)
        pq = (_cosine(live_path_vec, hist_path_vec)+1)/2  # path-based quant

        t    = _tag_sim(lt, EVENT_TAGS.get(en,set()))
        m    = _macro_sim(lo, lc, lf,
                          MACRO_CONTEXT.get(en, {"oil": 0, "cpi": "low", "fed": "hold"}),
                          evt_name=en)
        # Use live weight sliders if available, else fall back to config
        wq = _wl2_wq.value if "_wl2_wq" in dir() else ANALOGUE_WEIGHTS["quant"]
        wt = _wl2_wt.value if "_wl2_wt" in dir() else ANALOGUE_WEIGHTS["tag"]
        wm = _wl2_wm.value if "_wl2_wm" in dir() else ANALOGUE_WEIGHTS["macro"]
        _ws = wq + wt + wm
        if _ws > 0: wq, wt, wm = wq/_ws, wt/_ws, wm/_ws
        # Composite uses path quant (B1) as the quant component
        comp = wq*pq + wt*t + wm*m
        scores.append({"event":en,"composite":comp,
                        "quant":pq,"quant_pt":q,   # pq=path, q=point-in-time
                        "tag":t,"macro":m,
                        "wq":wq,"wt":wt,"wm":wm})
    scores.sort(key=lambda x: x["composite"], reverse=True)
    return scores

def _rank_fig(scores):
    evts = [s["event"] for s in scores]
    fig  = go.Figure()
    # Use weights stored in first score entry (all same run)
    _wq = scores[0].get("wq", ANALOGUE_WEIGHTS["quant"]) if scores else ANALOGUE_WEIGHTS["quant"]
    _wt = scores[0].get("wt", ANALOGUE_WEIGHTS["tag"])   if scores else ANALOGUE_WEIGHTS["tag"]
    _wm = scores[0].get("wm", ANALOGUE_WEIGHTS["macro"]) if scores else ANALOGUE_WEIGHTS["macro"]
    fig.add_trace(go.Bar(name=f"Path Quant ({_wq:.0%})", x=evts,
        y=[s["quant"]*s.get("wq",_wq) for s in scores], marker_color="#58a6ff"))
    fig.add_trace(go.Bar(name=f"Tag ({_wt:.0%})", x=evts,
        y=[s["tag"]*s.get("wt",_wt) for s in scores], marker_color="#3fb950"))
    fig.add_trace(go.Bar(name=f"Macro ({_wm:.0%})", x=evts,
        y=[s["macro"]*s.get("wm",_wm) for s in scores], marker_color="#e3b341"))
    # Point-in-time quant shown as line for comparison (B1)
    fig.add_trace(go.Scatter(name="Point-in-Time Q", x=evts,
        y=[s.get("quant_pt", s["quant"]) for s in scores],
        mode="markers", marker=dict(color="#a371f7", size=8, symbol="diamond"),
        yaxis="y2"))
    fig.add_hline(y=wl2_cut.value,
        line=dict(color="#ff7b72",width=1.5,dash="dash"),
        annotation_text=f"Cutoff {wl2_cut.value:.2f}  "
                        f"| Q:{_wl2_wq.value:.0%} T:{_wl2_wt.value:.0%} "
                        f"M:{_wl2_wm.value:.0%}",
        annotation_font=dict(color="#ff7b72"))
    fig.update_layout(**dark_layout(
        title=f"Analogue Ranking — {LIVE['name']} at Day+{LIVE['day_n']}",
        xaxis=dict(tickangle=35,gridcolor=COL_GRID,linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED,size=9)),
        yaxis=dict(title="Score contribution",gridcolor=COL_GRID,
                   linecolor=COL_GRID,tickfont=dict(color=COL_MUTED)),
        barmode="stack",
        yaxis2=dict(title="Point-in-Time Q", overlaying="y", side="right",
                    range=[0,1], showgrid=False,
                    tickfont=dict(color="#a371f7",size=9),
                    titlefont=dict(color="#a371f7",size=10)),
        legend=dict(x=1.01,y=1,bgcolor=BG_PANEL,bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT,size=10)),
        width=CHART_W,height=460,margin=dict(b=140,r=160)))
    return fig

def _score_tbl(scores, cutoff):
    sel = sum(1 for s in scores if s["composite"] >= cutoff)
    rc  = []
    for s in scores:
        a = min(0.15+s["composite"]*0.45,0.65)
        rc.append(f"rgba(63,185,80,{a:.2f})" if s["composite"]>=cutoff
                  else "rgba(33,38,45,0.8)")
    fig = go.Figure(go.Table(
        header=dict(values=["<b>Event</b>","<b>Composite</b>","<b>Path Q</b>","<b>Pt Q</b>",
                            "<b>Tag</b>","<b>Macro</b>","<b>Use?</b>"],
            fill_color="#1c2128",font=dict(color=COL_HEAD,size=11,family="monospace"),
            align="center",height=30,line=dict(color=COL_GRID,width=1)),
        cells=dict(values=[
            [s["event"] for s in scores],
            [f"{s['composite']:.3f}" for s in scores],
            [f"{s['quant']:.3f}" for s in scores],
            [f"{s.get('quant_pt',s['quant']):.3f}" for s in scores],
            [f"{s['tag']:.3f}" for s in scores],
            [f"{s['macro']:.3f}" for s in scores],
            ["✅" if s["composite"]>=cutoff else "—" for s in scores],
        ], fill_color=[rc]*6, align="center",
           font=dict(color=COL_HEAD,size=10,family="monospace"),
           height=26,line=dict(color=COL_GRID,width=0.5))))
    fig.update_layout(
        title=dict(text=f"Scores — cutoff {cutoff:.2f} → {sel} analogues selected",
                   font=dict(color=COL_HEAD,size=13)),
        paper_bgcolor=BG,font=dict(color=COL_TEXT,family="monospace"),
        width=CHART_W,height=max(380,len(scores)*28+120),
        margin=dict(t=60,b=20,l=10,r=10))
    return fig

# ── Analogue weight sliders ──────────────────────────────────
# These feed directly into run_analogue_match() at call time.
# Renormalised automatically so they always sum to 1.
_sl_w = {"description_width": "80px"}
_wl2_wq = widgets.FloatSlider(value=ANALOGUE_WEIGHTS["quant"], min=0.0, max=1.0, step=0.05,
    description="Quant:", continuous_update=False,
    layout=widgets.Layout(width="340px"), style=_sl_w)
_wl2_wt = widgets.FloatSlider(value=ANALOGUE_WEIGHTS["tag"], min=0.0, max=1.0, step=0.05,
    description="Tag:", continuous_update=False,
    layout=widgets.Layout(width="340px"), style=_sl_w)
_wl2_wm = widgets.FloatSlider(value=ANALOGUE_WEIGHTS["macro"], min=0.0, max=1.0, step=0.05,
    description="Macro:", continuous_update=False,
    layout=widgets.Layout(width="340px"), style=_sl_w)
_wl2_wtot_lbl = widgets.Label(value="Weights sum: 1.00  (auto-renormalised)")

wl2_state = make_refresh_state()

def _update_weight_label(*_):
    s = _wl2_wq.value + _wl2_wt.value + _wl2_wm.value
    _wl2_wtot_lbl.value = f"Weights sum: {s:.2f}  (auto-renormalised to 1.0)"

_wl2_wq.observe(_update_weight_label, "value")
_wl2_wt.observe(_update_weight_label, "value")
_wl2_wm.observe(_update_weight_label, "value")

# ── Cutoff slider (L2 canonical — linked to L3 via observe) ──
wl2_cut = widgets.FloatSlider(value=0.50,min=0.0,max=1.0,step=0.01,
    description="Score cutoff:",continuous_update=False,
    layout=widgets.Layout(width="440px"),style={"description_width":"100px"})
wl2_run = widgets.Button(description="▶  Run Analogue Match",button_style="primary",
    layout=widgets.Layout(width="210px"))
wl2_out = widgets.Output()
_scores_cache = []

def _wl2_run(*_):
    global _scores_cache
    key = (LIVE.get("name"), LIVE.get("day_n"), float(wl2_cut.value), float(_wl2_wq.value), float(_wl2_wt.value), float(_wl2_wm.value))

    def _render():
        with wl2_out:
            clear_output(wait=True)
            sc = run_analogue_match()
            if sc is None:
                return
            _scores_cache = sc
            _rank_fig(sc).show()
            _score_tbl(sc,wl2_cut.value).show()

    wl2_state["last"] = None
    run_guarded_refresh(wl2_state, key, _render, label="Standalone analogue match")


def _wl2_cut(*_):
    if not _scores_cache:
        return
    key = ("cut", float(wl2_cut.value), len(_scores_cache))
    run_guarded_refresh(
        wl2_state,
        key,
        lambda: (clear_output(wait=True), _rank_fig(_scores_cache).show(), _score_tbl(_scores_cache,wl2_cut.value).show()) if False else None,
        label="Standalone analogue cutoff",
    )
    with wl2_out:
        clear_output(wait=True)
        _rank_fig(_scores_cache).show()
        _score_tbl(_scores_cache,wl2_cut.value).show()
    wl2_state["last"] = key


wl2_run.on_click(_wl2_run)
wl2_cut.observe(_wl2_cut,names="value")
# Link wl2_cut → wl3_cut (L3 has its own slider that stays in sync)
def _wl2_cut_sync(change):
    if "wl3_cut" in dir() and abs(wl3_cut.value - change["new"]) > 1e-9:
        wl3_cut.value = change["new"]
wl2_cut.observe(_wl2_cut_sync, "value")

# ── Shared state ──────────────────────────────────────────────
# _scores_cache : populated by run_analogue_match(), read by all Live tabs
# _shared_rows  : populated by §6 L4 run, read by §6 L5 detail view
_scores_cache = []
_shared_rows  = []

def _sel_ev(cutoff=None):
    """Return event names with composite score >= cutoff.
    Falls back to all EVENTS if analogue scores not yet computed.
    Ensures at least ANALOGUE_MIN_COUNT scored events when available."""
    c = cutoff if cutoff is not None else (wl2_cut.value if "wl2_cut" in dir() else 0.0)
    if not _scores_cache:
        return [e[0] for e in EVENTS]
    ranked = [s["event"] for s in _scores_cache]
    sel = [s["event"] for s in _scores_cache if s["composite"] >= c]
    min_n = int(globals().get("ANALOGUE_MIN_COUNT", 1) or 1)
    if len(sel) < min_n:
        sel = ranked[: min(min_n, len(ranked))]
    return sel or ranked


In [ ]:
# §5.3 — LIVE PATH vs ANALOGUES + FORWARD RETURN HEATMAP
# Overlay: live return path vs selected analogue events + weighted composite.
# Heatmap: median forward return from Day+N to each remaining POI.

def _check_scores():
    if not _scores_cache:
        print("⚠️  Analogue scores not computed — run Cell L2 first for filtered results.\n"
              "   Falling back to all events equally weighted."); return False
    return True

def _composite(label, sel_ev):
    if not _scores_cache: return None
    sm = {s["event"]:s["composite"] for s in _scores_cache}
    wv, ws = {}, {}
    for en in sel_ev:
        s = event_returns.get(label,{}).get(en)
        if s is None: continue
        w = sm.get(en,0.0)
        for off,val in s.items():
            wv[off] = wv.get(off,0.0)+val*w
            ws[off] = ws.get(off,0.0)+w
    if not wv: return None
    return pd.Series({o:wv[o]/ws[o] for o in wv if ws.get(o,0)>0}).sort_index()

def _overlay_fig(asset, cutoff):
    unit  = unit_label(asset)
    dlbl  = display_label(asset)
    cls_  = ASSET_META.get(asset,{}).get("class","")
    offs  = list(range(-PRE_WINDOW_TD,POST_WINDOW_TD+1))
    sel   = _sel_ev(cutoff)
    fig   = go.Figure()
    for j,(en,_) in enumerate(EVENTS):
        if en not in sel: continue
        s = event_returns[asset].get(en)
        if s is None: continue
        fig.add_trace(go.Scatter(x=offs,y=s.reindex(offs).values,name=en,mode="lines",
            line=dict(color=EVENT_COLORS[j%len(EVENT_COLORS)],
                      dash=EVENT_DASHES[j%len(EVENT_DASHES)],width=1.2),
            opacity=0.45,
            hovertemplate=f"<b>{en}</b><br>Day %{{x}}: %{{y:.2f}} {unit}<extra></extra>"))
    comp = _composite(asset, sel)
    if comp is not None:
        fig.add_trace(go.Scatter(x=list(comp.index),y=list(comp.values),
            name="⬜ Composite",mode="lines",
            line=dict(color="#ffffff",width=3,dash="dash"),
            hovertemplate=f"<b>Composite</b><br>Day %{{x}}: %{{y:.2f}} {unit}<extra></extra>"))
    lr = LIVE["returns"].get(asset)
    _dn = LIVE.get("day_n")
    if lr is not None and _dn is not None:
        lr_clip = lr[lr.index <= _dn]  # clip to current day_n (respects override)
        fig.add_trace(go.Scatter(x=list(lr_clip.index),y=list(lr_clip.values),
            name=f"▶ {LIVE['name']}",mode="lines+markers",
            line=dict(color="#ffa657",width=3),marker=dict(size=5,color="#ffa657"),
            hovertemplate=f"<b>{LIVE['name']}</b><br>Day %{{x}}: %{{y:.2f}} {unit}<extra></extra>"))
    poi_map = {o:l for l,o in POIS}
    fig.add_vline(x=0,line=dict(color=COL_ZERO,width=1.5,dash="dash"),
                  annotation_text="Day 0",annotation_font=dict(color=COL_ZERO))
    if _dn is not None:
        _is_ov = globals().get("wl_day_override_chk") and globals()["wl_day_override_chk"].value
        _ov_col = "#d29922" if _is_ov else "#ffa657"
        _ov_lbl = f"Day+{_dn} [OV]" if _is_ov else f"Today D+{_dn}"
        fig.add_vline(x=_dn,
                      line=dict(color=_ov_col,width=1.5,dash="dot"),
                      annotation_text=_ov_lbl,
                      annotation_font=dict(color=_ov_col))
    for po,pl in poi_map.items():
        if po != 0:
            fig.add_vline(x=po,line=dict(color=COL_GRID,width=0.8,dash="dot"),
                          annotation_text=pl,annotation_font=dict(color=COL_MUTED,size=9))
    fig.update_layout(**dark_layout(
        title=f"<b>{dlbl}</b> [{cls_}] — Live vs Analogues (cutoff {cutoff:.2f})",
        xaxis=dict(title="Trading days from Day 0",
                   tickvals=[o for _,o in POIS],ticktext=[l for l,_ in POIS],
                   gridcolor=COL_GRID,zerolinecolor=COL_GRID,
                   linecolor=COL_GRID,tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title=f"Return ({unit})",gridcolor=COL_GRID,zerolinecolor=COL_ZERO,
                   linecolor=COL_GRID,tickfont=dict(color=COL_MUTED)),
        legend=dict(orientation="v",x=1.01,y=1,bgcolor=BG_PANEL,bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT,size=9)),
        hovermode="x unified",width=CHART_W,height=CHART_H,margin=dict(r=260)))
    return fig

def _fwd_heatmap(labels, cutoff):
    sel  = _sel_ev(cutoff)
    dn   = LIVE["day_n"] or 0
    fwd  = [(l,o) for l,o in POIS if o > dn]
    if not fwd or not labels: return None
    mat  = np.full((len(labels),len(fwd)),np.nan)
    for i,lbl in enumerate(labels):
        for j,(_,po) in enumerate(fwd):
            vals = []
            for en in sel:
                sv=poi_ret(lbl,en,dn); fv=poi_ret(lbl,en,po)
                if not any(np.isnan([sv,fv])): vals.append(fv-sv)
            mat[i,j] = np.median(vals) if vals else np.nan
    txt  = [[f"{mat[i,j]:+.1f}" if not np.isnan(mat[i,j]) else "N/A"
             for j in range(len(fwd))] for i in range(len(labels))]
    vmax = np.nanmax(np.abs(mat)) if not np.all(np.isnan(mat)) else 5
    fig  = go.Figure(go.Heatmap(z=mat,x=[l for l,_ in fwd],
        y=[display_label(l) for l in labels],
        text=txt,texttemplate="%{text}",
        colorscale="RdYlGn",zmid=0,zmin=-vmax,zmax=vmax,
        colorbar=dict(title=dict(text="Median Fwd Ret",font=dict(color=COL_TEXT)),
                      thickness=14,tickfont=dict(color=COL_TEXT)),
        hovertemplate="<b>%{y}</b><br>%{x}: %{text}<extra></extra>"))
    fig.update_layout(**dark_layout(
        title=f"Forward Returns from Day+{dn} — {len(sel)} analogues",
        xaxis=dict(title="Horizon",gridcolor=COL_GRID,linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title="Asset",autorange="reversed",gridcolor=COL_GRID,
                   linecolor=COL_GRID,tickfont=dict(color=COL_MUTED)),
        width=CHART_W,height=max(420,len(labels)*36+120)))
    return fig

wl3_cls  = widgets.Dropdown(options=ALL_CLASSES,description="Class:",
    layout=widgets.Layout(width="200px"),style={"description_width":"50px"})
wl3_ast  = widgets.Dropdown(description="Asset:",
    layout=widgets.Layout(width="260px"),style={"description_width":"50px"})
wl3_prev = widgets.Button(description="◀",layout=widgets.Layout(width="40px"))
wl3_next = widgets.Button(description="▶",layout=widgets.Layout(width="40px"))
wl3_cut  = widgets.FloatSlider(value=0.50,min=0.0,max=1.0,step=0.01,
    description="Cutoff:",continuous_update=False,
    layout=widgets.Layout(width="360px"),style={"description_width":"60px"})
wl3_grp  = widgets.Dropdown(
    options=["— All Assets —"]+sorted(CUSTOM_GROUPS.keys())+ALL_CLASSES,
    value="Risk Barometer",description="Fwd Group:",
    layout=widgets.Layout(width="280px"),style={"description_width":"80px"})
wl3_out  = widgets.Output()
wl3_state = make_refresh_state()

def _wl3_upd(cls_val):
    opts = [l for l in CLASS_TO_ASSETS.get(cls_val,[]) if l in event_returns]
    cur = wl3_ast.value if wl3_ast.value in opts else None
    wl3_ast.options = opts
    if cur is not None:
        wl3_ast.value = cur
    elif opts:
        wl3_ast.value = opts[0]


def _wl3_ref(*_):
    key = (wl3_cls.value, wl3_ast.value, wl3_grp.value, float(wl3_cut.value), LIVE.get("name"), LIVE.get("day_n"))
    def _render():
        with wl3_out:
            clear_output(wait=True)
            if LIVE["returns"] is None: print("❌ Run Cell L1 first."); return
            _check_scores()
            c = wl3_cut.value
            if wl3_ast.value: _overlay_fig(wl3_ast.value,c).show()
            g = wl3_grp.value
            if g=="— All Assets —":    lb=get_subset("all")
            elif g in CUSTOM_GROUPS:   lb=get_subset("group",group=g)
            else:                      lb=get_subset("class",cls=g)
            fh = _fwd_heatmap(lb,c)
            if fh: fh.show()
    run_guarded_refresh(wl3_state, key, _render, label="Standalone live path")


def _wl3_p(_):
    opts=list(wl3_ast.options)
    if opts: wl3_ast.value=opts[(opts.index(wl3_ast.value)-1)%len(opts)]
def _wl3_n(_):
    opts=list(wl3_ast.options)
    if opts: wl3_ast.value=opts[(opts.index(wl3_ast.value)+1)%len(opts)]

wl3_cls.observe(lambda c:(_wl3_upd(c["new"]),_wl3_ref()) if c.get("name")=="value" else None,names="value")
wl3_ast.observe(lambda c:_wl3_ref() if c.get("name")=="value" else None,names="value")
wl3_cut.observe(lambda c:_wl3_ref() if c.get("name")=="value" else None,names="value")
wl3_grp.observe(lambda c:_wl3_ref() if c.get("name")=="value" else None,names="value")
wl3_prev.on_click(_wl3_p); wl3_next.on_click(_wl3_n)
_wl3_upd(wl3_cls.value)
wl3_state["ready"] = True
_wl3_ref()


In [ ]:
# §5.4 — TRADE IDEAS TABLE
# Ranked by |median / IQR| score. Includes hit rate, vol-adj return,
# live deviation vs analogue distribution, and convergence status flag.
# Row dict keys: lbl, cls, ticker, dir, med, mean, std, iqr, stars, n,
#                n_total, unit, score, hit_rate, vol_adj, live_gap,
#                live_pctile, status, fwd_vals, corr
# Coverage filter: rows where n/n_total < min_cov go to weak-signal section.

def _stars(iqr, med):
    if abs(med) < 0.01: return "☆☆☆☆☆"
    r = iqr/(abs(med)+1e-9)
    if r < 0.40: return "★★★★★"
    if r < 0.70: return "★★★★☆"
    if r < 1.00: return "★★★☆☆"
    if r < 1.50: return "★★☆☆☆"
    return "★☆☆☆☆"

_TRADE_ROWS_CACHE = {}
_TRADE_ROWS_CACHE_LAST = {"hit": False, "rows": 0, "key": None}

def _trade_cache_key(labels, cutoff, fwd_days, dn, sel):
    return (
        tuple(labels),
        round(float(cutoff), 6),
        int(fwd_days),
        int(dn),
        tuple(sel),
        LIVE.get("name"),
        LIVE.get("date"),
    )

def _compute_rows(labels, cutoff, fwd_days):
    """Core computation shared by L4 and L5.
    fwd_days: trading days forward from current live day (dn).
    fo = dn + fwd_days  (absolute offset from Day 0 into event_returns).
    Cached per live state so repeated L4/L5 renders feel instant.
    """
    if not _check_live(): return []
    _check_scores()
    sel    = _sel_ev(cutoff)
    n_sel  = len(sel)
    dn     = LIVE["day_n"] or 0
    fo     = dn + int(fwd_days)
    if fo <= dn: return []
    _ckey = _trade_cache_key(labels, cutoff, fwd_days, dn, sel)
    if _ckey in _TRADE_ROWS_CACHE:
        _TRADE_ROWS_CACHE_LAST.update({"hit": True, "rows": len(_TRADE_ROWS_CACHE[_ckey]), "key": _ckey})
        return [dict(r) for r in _TRADE_ROWS_CACHE[_ckey]]

    rows = []
    for lbl in labels:
        fwd_vals = []
        for en in sel:
            sv=poi_ret(lbl,en,dn); fv=poi_ret(lbl,en,fo)
            if not any(np.isnan([sv,fv])): fwd_vals.append(fv-sv)
        if len(fwd_vals) < 1: continue

        med  = np.median(fwd_vals)
        iqr  = np.percentile(fwd_vals,75)-np.percentile(fwd_vals,25)
        mn   = np.mean(fwd_vals)
        sd   = np.std(fwd_vals)
        unit = unit_label(lbl)

        _hits = np.sum([1 if (med>0 and v>0) or (med<0 and v<0) else 0
                        for v in fwd_vals])
        # Laplace-smoothed hit-rate avoids misleading 100% for n=1.
        hit_rate = (_hits + 1.0) / (len(fwd_vals) + 2.0)

        all_daily = []
        for en in sel:
            s = event_returns.get(lbl,{}).get(en)
            if s is None: continue
            s_ = s[s.index.isin(range(dn+1,fo+1))]
            if len(s_) > 1:
                all_daily.extend(list(np.diff(s_.values)))
        hist_vol = np.std(all_daily) if len(all_daily) > 1 else 1.0
        vol_adj  = med / (hist_vol + 1e-9)

        # ── Risk-adjusted metrics ─────────────────────────────
        sharpe   = mn / (sd + 1e-9)          # mean / std of fwd returns
        down     = [v for v in fwd_vals if v < 0]
        dsd      = np.std(down) if len(down) > 1 else (sd + 1e-9)
        sortino  = mn / (dsd + 1e-9)          # mean / downside std
        worst    = float(np.min(fwd_vals))     # worst analogue outcome
        skewness = float(pd.Series(fwd_vals).skew()) if len(fwd_vals) >= 3 else 0.0
        mdd_v, _, _ = _max_drawdown_from_d0(lbl, sel)
        calmar   = mn / (abs(mdd_v) + 1e-9) if mdd_v is not None else np.nan

        live_ret_dn = None
        lr = LIVE["returns"].get(lbl)
        if lr is not None and dn in lr.index:
            live_ret_dn = float(lr.loc[dn])

        hist_at_dn = []
        for en in sel:
            v = poi_ret(lbl,en,dn)
            if not np.isnan(v): hist_at_dn.append(v)

        live_gap    = np.nan
        live_pctile = np.nan
        status      = "—"
        if live_ret_dn is not None and len(hist_at_dn) >= 2:
            comp_med    = np.median(hist_at_dn)
            live_gap    = live_ret_dn - comp_med
            pctile      = float(np.mean([1 if live_ret_dn > v else 0
                                         for v in hist_at_dn])) * 100
            live_pctile = pctile
            if pctile < 25:   status = "🟢 Still open"
            elif pctile < 50: status = "🟡 On track"
            elif pctile < 75: status = "🟠 Chasing"
            else:             status = "🔴 Extended"

        # Robust score normalization:
        # - floor IQR to avoid division blow-ups when distribution is too tight
        # - damp score for low sample size
        # - penalize weak coverage / weak directional consistency so low-quality
        #   tails remain visible but rank below better-supported ideas
        iqr_floor = max(iqr, abs(med) * 0.25, 0.25)
        sample_adj = min(1.0, np.sqrt(len(fwd_vals) / 5.0))
        coverage = (len(fwd_vals) / n_sel) if n_sel > 0 else 1.0
        coverage_adj = min(1.0, max(0.35, coverage))
        hit_adj = min(1.0, max(0.35, abs(hit_rate - 0.50) / 0.50))
        quality_adj = (sample_adj * 0.45) + (coverage_adj * 0.35) + (hit_adj * 0.20)
        score = min((abs(med) / (iqr_floor + 1e-9)) * quality_adj, 10.0)

        if coverage >= 0.60 and len(fwd_vals) >= 5:
            data_conf = "high"
        elif coverage >= 0.30 and len(fwd_vals) >= 2:
            data_conf = "medium"
        else:
            data_conf = "low"

        meta = ASSET_META.get(lbl, {})
        rows.append({
            "lbl":     lbl,
            "cls":     meta.get("class", ""),
            "ticker":  meta.get("ticker", ""),
            "dir":     "LONG" if med > 0 else "SHORT",
            "med":     med, "mean": mn, "std": sd, "iqr": iqr,
            "stars":   _stars(iqr, med),
            "n":       len(fwd_vals),
            "n_total": n_sel,           # total selected analogues
            "unit":    unit,
            "score":   score,
            "data_conf": data_conf,
            "hit_rate": hit_rate, "vol_adj": vol_adj,
            "sharpe":  sharpe, "sortino": sortino,
            "calmar":  calmar, "skew": skewness, "worst": worst,
            "live_gap": live_gap, "live_pctile": live_pctile,
            "status":  status, "fwd_vals": fwd_vals,
        })
    rows.sort(key=lambda r: r["score"], reverse=True)

    # Intra-correlation flag
    top     = rows[:15]
    fwd_mat = np.array([r["fwd_vals"][:len(sel)] for r in top
                        if len(r["fwd_vals"]) == len(sel)])
    corr_flags = [""] * len(rows)
    if fwd_mat.shape[0] > 1 and fwd_mat.shape[1] > 1:
        try:
            C = np.corrcoef(fwd_mat)
            for i in range(len(top)):
                for j in range(i+1, len(top)):
                    if i < C.shape[0] and j < C.shape[1]:
                        if abs(C[i,j]) > 0.75:
                            corr_flags[i] = "⚠️"
                            corr_flags[j] = "⚠️"
        except Exception:
            pass
    for i, r in enumerate(rows):
        r["corr"] = corr_flags[i]
    _TRADE_ROWS_CACHE[_ckey] = [dict(r) for r in rows]
    _TRADE_ROWS_CACHE_LAST.update({"hit": False, "rows": len(rows), "key": _ckey})
    if len(_TRADE_ROWS_CACHE) > 24:
        for _k in list(_TRADE_ROWS_CACHE)[:-24]:
            _TRADE_ROWS_CACHE.pop(_k, None)
    return [dict(r) for r in rows]


In [ ]:

def _summary_table_fig(rows, fwd_days, cutoff, min_cov=0.0):
    """Return (main_fig, weak_fig) — two separate Plotly figures.
    main_fig : assets where n/n_total >= min_cov
    weak_fig : assets where n/n_total < min_cov  (None if none)
    Columns: # | Asset | Ticker | Class | Dir | Horizon | Median | Hit% |
               VolAdj | Gap | Pctile | Status | Corr | Conv | N/Total
    """
    if not rows:
        return None, None

    dn    = LIVE["day_n"] or 0
    n_sel = len(_sel_ev(cutoff))
    cov_pct = int(min_cov * 100)

    main_rows = [r for r in rows
                 if r["n_total"] == 0 or (r["n"] / r["n_total"]) >= min_cov]
    weak_rows = [r for r in rows
                 if r["n_total"] > 0 and (r["n"] / r["n_total"]) < min_cov]

    def _make_fig(rws, start_rank=1, section_label="Trade Ideas"):
        if not rws: return None
        cc = []
        for r in rws:
            conv  = min(0.12 + (5 - r["stars"].count("☆")) * 0.10, 0.62)
            cc.append(f"rgba(63,185,80,{conv:.2f})" if r["dir"] == "LONG"
                      else f"rgba(247,129,102,{conv:.2f})")
        hdrs = ["<b>#</b>","<b>Asset</b>","<b>Ticker</b>","<b>Class</b>",
                "<b>Dir</b>","<b>Horizon</b>","<b>Median</b>","<b>Mean</b>",
                "<b>Hit%</b>","<b>Sharpe</b>","<b>Sortino</b>","<b>Calmar</b>",
                "<b>Skew</b>","<b>Worst</b>",
                "<b>VolAdj</b>","<b>Gap</b>","<b>Pctile</b>",
                "<b>Status</b>","<b>Corr</b>","<b>Data Conf</b>","<b>Conv</b>","<b>N/Tot</b>"]
        col_vals = [
            list(range(start_rank, start_rank + len(rws))),
            [display_label(r["lbl"])          for r in rws],
            [r["ticker"]                      for r in rws],
            [r["cls"]                         for r in rws],
            [r["dir"]                         for r in rws],
            [f"+{fwd_days}d"                    for r in rws],
            [f"{r['med']:+.1f}{r['unit']}"    for r in rws],
            [f"{r['mean']:+.1f}{r['unit']}"   for r in rws],
            [f"{r['hit_rate']*100:.0f}%"      for r in rws],
            [f"{r['sharpe']:+.2f}"            for r in rws],
            [f"{r['sortino']:+.2f}"           for r in rws],
            [f"{r['calmar']:+.2f}" if not np.isnan(r.get("calmar", np.nan)) else "—"
             for r in rws],
            [f"{r['skew']:+.2f}"             for r in rws],
            [f"{r['worst']:+.1f}{r['unit']}" for r in rws],
            [f"{r['vol_adj']:+.2f}"           for r in rws],
            [f"{r['live_gap']:+.1f}" if not np.isnan(r["live_gap"]) else "—"
             for r in rws],
            [f"{r['live_pctile']:.0f}th" if not np.isnan(r["live_pctile"]) else "—"
             for r in rws],
            [r["status"]                      for r in rws],
            [r["corr"]                        for r in rws],
            [r.get("data_conf", "low")        for r in rws],
            [r["stars"]                       for r in rws],
            [f"{r['n']}/{r['n_total']}"          for r in rws],
        ]
        n_cols = len(hdrs)
        fig = go.Figure(go.Table(
            header=dict(
                values=hdrs,
                fill_color="#1c2128",
                font=dict(color=COL_HEAD, size=10, family="monospace"),
                align="center", height=30, line=dict(color=COL_GRID, width=1)),
            cells=dict(
                values=col_vals,
                fill_color=[cc] * n_cols,
                align="center",
                font=dict(color=COL_HEAD, size=9, family="monospace"),
                height=26, line=dict(color=COL_GRID, width=0.5))))
        title_str = (
            f"{section_label}  |  {LIVE['name']}  |  "
            f"Day+{dn} → Day+{dn+fwd_days} (+{fwd_days}d)  |  {n_sel} analogues  "
            f"(score cutoff {cutoff:.2f}  |  coverage ≥{cov_pct}%)\n"
            f"🟢 Still open  🟡 On track  🟠 Chasing  🔴 Extended  ⚠️ Correlated"
        )
        fig.update_layout(
            title=dict(text=title_str, font=dict(color=COL_HEAD, size=11)),
            paper_bgcolor=BG,
            font=dict(color=COL_TEXT, family="monospace"),
            width=CHART_W,
            height=max(400, len(rws) * 26 + 160),
            margin=dict(t=80, b=20, l=10, r=10),
        )
        return fig

    main_fig = _make_fig(main_rows, start_rank=1,
                         section_label="Trade Ideas")
    weak_fig = _make_fig(weak_rows, start_rank=len(main_rows) + 1,
                         section_label=f"⚠ Weak Signal  (coverage < {cov_pct}%)")
    return main_fig, weak_fig


In [ ]:

# ── Shared state for downstream detail functions ──────────────
_rows_cache  = []
_horiz_cache = "t+1M"
_cut_cache   = 0.50

# Legacy standalone L4 widgets were removed from auto-bootstrap.
# The real interactive trade-ideas workflow now lives in §6.9 TAB L4.


In [ ]:
# §5.5 — DETAILED TRADE ANALYSIS
# Per-asset drill-down: stats write-up (NEW, above charts) then
# signal decay chart, per-analogue dot plot, live deviation fan,
# and intra-group correlation heatmap.

# ── Stats write-up (new) ──────────────────────────────────────

def _per_poi_stats(lbl, sel_ev, dn):
    """Per-POI stats table: Q1/median/Q3/hit-rate across all forward horizons."""
    unit     = unit_label(lbl)
    horizons = [(l, o) for l, o in POIS if o > dn]
    rows_out = []
    for hl, ho in horizons:
        vals = []
        for en in sel_ev:
            sv = poi_ret(lbl, en, dn)
            fv = poi_ret(lbl, en, ho)
            if not any(np.isnan([sv, fv])):
                vals.append(fv - sv)
        if not vals:
            continue
        med = np.median(vals)
        mn_v  = np.mean(vals); sd_v = np.std(vals)
        down_v = [v for v in vals if v < 0]
        dsd_v  = np.std(down_v) if len(down_v) > 1 else (sd_v + 1e-9)
        rows_out.append({
            "horizon": hl,
            "n":       len(vals),
            "q1":      np.percentile(vals, 25),
            "med":     med,
            "q3":      np.percentile(vals, 75),
            "hit":     np.mean([1 if (med > 0 and v > 0) or (med < 0 and v < 0) else 0
                                 for v in vals]),
            "sharpe":  mn_v / (sd_v + 1e-9),
            "sortino": mn_v / (dsd_v + 1e-9),
            "skew":    float(pd.Series(vals).skew()) if len(vals) >= 3 else 0.0,
            "worst":   float(np.min(vals)),
            "unit":    unit,
        })
    return rows_out


def _max_drawdown_from_d0(lbl, sel_ev):
    """Max drawdown from Day 0 across all analogue paths.
    For † events (POI-only), computed from POI snapshots only — labelled approx."""
    unit    = unit_label(lbl)
    ddwns   = []
    approx  = False
    for en in sel_ev:
        s = event_returns.get(lbl, {}).get(en)
        if s is None:
            continue
        pos_idx = sorted([o for o in s.index if o >= 0])
        if not pos_idx:
            continue
        if en.endswith("†"):
            approx = True
        vals = [float(s.loc[o]) for o in pos_idx]
        # Drawdown = max peak-to-trough from Day 0 entry
        peak = 0.0
        dd   = 0.0
        for v in vals:
            if v > peak:
                peak = v
            dd = min(dd, v - peak)
        ddwns.append(dd)
    if not ddwns:
        return None, unit, approx
    return float(np.median(ddwns)), unit, approx


def _writeup_fig(lbl, row, sel_ev, cutoff):
    """Per-POI stats table figure + prose rationale below as annotation."""
    dn       = LIVE["day_n"] or 0
    dlbl     = display_label(lbl)
    poi_rows = _per_poi_stats(lbl, sel_ev, dn)
    mdd, unit, mdd_approx = _max_drawdown_from_d0(lbl, sel_ev)

    if not poi_rows:
        return None

    # ── Build stats table ─────────────────────────────────────
    hdrs = ["<b>Horizon</b>", "<b>N</b>",
            "<b>Q1</b>", "<b>Median</b>", "<b>Q3</b>", "<b>Hit%</b>",
            "<b>Sharpe</b>", "<b>Sortino</b>", "<b>Skew</b>", "<b>Worst</b>"]
    col_vals = [
        [r["horizon"]                         for r in poi_rows],
        [r["n"]                               for r in poi_rows],
        [f"{r['q1']:+.1f}{r['unit']}"         for r in poi_rows],
        [f"{r['med']:+.1f}{r['unit']}"        for r in poi_rows],
        [f"{r['q3']:+.1f}{r['unit']}"         for r in poi_rows],
        [f"{r['hit']*100:.0f}%"               for r in poi_rows],
        [f"{r['sharpe']:+.2f}"                for r in poi_rows],
        [f"{r['sortino']:+.2f}"               for r in poi_rows],
        [f"{r['skew']:+.2f}"                  for r in poi_rows],
        [f"{r['worst']:+.1f}{r['unit']}"      for r in poi_rows],
    ]
    # Max drawdown row appended as a separator row
    if mdd is not None:
        approx_tag = " (approx†)" if mdd_approx else ""
        for col_i, col_data in enumerate(col_vals):
            if col_i == 0:
                col_data.append(f"Max DD from D0{approx_tag}")
            elif col_i == 3:   # Median column
                col_data.append(f"{mdd:+.1f}{unit}")
            else:
                col_data.append("—")

    n_data_rows = len(poi_rows) + (1 if mdd is not None else 0)

    med_colors = []
    for r in poi_rows:
        med_colors.append("rgba(63,185,80,0.25)" if r["med"] > 0
                          else "rgba(247,129,102,0.25)")
    if mdd is not None:
        med_colors.append("rgba(88,166,255,0.15)")  # drawdown row — blue tint

    fill_grid = [["#1c2128"] * n_data_rows] * len(hdrs)
    fill_grid[3] = med_colors   # colour the median column by direction

    fig = go.Figure(go.Table(
        header=dict(
            values=hdrs,
            fill_color="#1c2128",
            font=dict(color=COL_HEAD, size=10, family="monospace"),
            align="center", height=28, line=dict(color=COL_GRID, width=1)),
        cells=dict(
            values=col_vals,
            fill_color=fill_grid,
            align="center",
            font=dict(color=COL_HEAD, size=9, family="monospace"),
            height=24, line=dict(color=COL_GRID, width=0.5))))

    # ── Prose rationale ───────────────────────────────────────
    dir_str  = row["dir"]
    med_val  = row["med"]
    hit_pct  = row["hit_rate"] * 100
    best_hor = max(poi_rows, key=lambda r: abs(r["med"]))["horizon"] if poi_rows else "—"

    dir_word = "long" if dir_str == "LONG" else "short"
    signal_strength = (
        "strong" if row["stars"].count("★") >= 4
        else "moderate" if row["stars"].count("★") >= 2
        else "weak"
    )

    # Top analogues if scores cached and above cutoff
    analogue_names = []
    if _scores_cache:
        thresh   = cutoff
        top_anl  = sorted(
            [(sc["event"], sc["composite"]) for sc in _scores_cache
             if sc["composite"] >= thresh],
            key=lambda x: x[1], reverse=True
        )[:3]
        analogue_names = [a[0] for a in top_anl]

    analogue_txt = ""
    if analogue_names:
        analogue_txt = (
            f" The closest historical parallels are "
            + ", ".join(f"<b>{a}</b>" for a in analogue_names[:2])
            + (f" and <b>{analogue_names[2]}</b>" if len(analogue_names) > 2 else "")
            + "."
        )

    prose = (
        f"<b style='color:#ffa657'>{dlbl}</b> — "
        f"signal favours <b>{dir_word}</b> with a {signal_strength} conviction "
        f"(conv: {row['stars']}).<br>"
        f"Median forward return at best horizon ({best_hor}): "
        f"<b>{med_val:+.1f}{unit}</b>, "
        f"hit rate <b>{hit_pct:.0f}%</b> across {row['n']} analogues."
        f"{analogue_txt}"
    )
    if mdd is not None:
        approx_note = " (approx — POI snapshots only†)" if mdd_approx else ""
        prose += (
            f"<br>Median path drawdown from Day 0: "
            f"<b>{mdd:+.1f}{unit}</b>{approx_note}."
        )

    fig.update_layout(
        title=dict(
            text=f"<b>{dlbl}</b>  Stats Summary  |  {dir_str}  |  {row['status']}",
            font=dict(color=COL_HEAD, size=12)),
        paper_bgcolor=BG,
        font=dict(color=COL_TEXT, family="monospace"),
        width=CHART_W,
        height=max(280, n_data_rows * 24 + 140),
        margin=dict(t=50, b=60, l=10, r=10),
        annotations=[dict(
            x=0, y=-0.08, xref="paper", yref="paper",
            text=prose, showarrow=False, align="left",
            font=dict(color=COL_TEXT, size=10, family="monospace"),
            xanchor="left",
        )],
    )
    return fig

def _decay_fig(lbl, sel_ev, cutoff):
    """Hit rate + median return across all positive POI horizons."""
    dn    = LIVE["day_n"] or 0
    unit  = unit_label(lbl)
    dlbl  = display_label(lbl)
    horizons = [(l,o) for l,o in POIS if o > dn]
    if not horizons: return None

    meds,hits,hnames = [],[],[]
    for hl,ho in horizons:
        vals = []
        for en in sel_ev:
            sv=poi_ret(lbl,en,dn); fv=poi_ret(lbl,en,ho)
            if not any(np.isnan([sv,fv])): vals.append(fv-sv)
        if not vals: continue
        med = np.median(vals)
        hit = np.mean([1 if (med>0 and v>0) or (med<0 and v<0) else 0 for v in vals])
        meds.append(med); hits.append(hit*100); hnames.append(hl)

    if not hnames: return None
    best_idx = int(np.argmax([abs(m) for m in meds]))

    fig = go.Figure()
    colors = ["#3fb950" if m>0 else "#ff7b72" for m in meds]
    fig.add_trace(go.Bar(name="Median Fwd Ret",x=hnames,y=meds,
        marker_color=colors,opacity=0.8,
        hovertemplate="<b>%{x}</b><br>Median: %{y:+.2f} "+unit+"<extra></extra>"))
    fig.add_trace(go.Scatter(name="Hit Rate %",x=hnames,y=hits,
        mode="lines+markers",marker=dict(size=8,color="#e3b341"),
        line=dict(color="#e3b341",width=2),yaxis="y2",
        hovertemplate="<b>%{x}</b><br>Hit rate: %{y:.0f}%<extra></extra>"))
    fig.add_shape(type="rect",
        x0=best_idx-0.4,x1=best_idx+0.4,y0=0,y1=1,
        xref="x",yref="paper",
        fillcolor="rgba(88,166,255,0.08)",line=dict(color="#58a6ff",width=1))
    fig.update_layout(**dark_layout(
        title=f"<b>{dlbl}</b> — Signal Decay Across Horizons",
        xaxis=dict(title="Horizon",gridcolor=COL_GRID,linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title=f"Median Fwd Ret ({unit})",gridcolor=COL_GRID,
                   linecolor=COL_GRID,tickfont=dict(color=COL_MUTED),
                   zeroline=True,zerolinecolor=COL_ZERO),
        yaxis2=dict(overlaying="y",side="right",
                    range=[0,105],gridcolor=COL_GRID,
                    tickfont=dict(color="#e3b341"),
                    title=dict(text="Hit Rate %",font=dict(color="#e3b341"))),
        legend=dict(x=0.01,y=0.99,bgcolor=BG_PANEL,bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT,size=10)),
        width=720,height=340,margin=dict(r=80,t=60,b=50)))
    return fig

def _dot_plot_fig(lbl, sel_ev, horiz_lbl, cutoff):
    """Per-analogue dot plot at chosen horizon with live return marked."""
    dn   = LIVE["day_n"] or 0
    fo   = next((o for l,o in POIS if l==horiz_lbl),None)
    unit = unit_label(lbl)
    dlbl = display_label(lbl)
    if fo is None or fo <= dn: return None

    vals, evts = [], []
    for en in sel_ev:
        sv=poi_ret(lbl,en,dn); fv=poi_ret(lbl,en,fo)
        if not any(np.isnan([sv,fv])):
            vals.append(fv-sv); evts.append(en)
    if not vals: return None

    colors = ["#3fb950" if v>0 else "#ff7b72" for v in vals]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=vals, y=evts, mode="markers",
        marker=dict(size=12,color=colors,line=dict(color="#ffffff",width=0.5)),
        hovertemplate="<b>%{y}</b><br>%{x:+.2f} "+unit+"<extra></extra>"))
    med = np.median(vals)
    fig.add_vline(x=med,line=dict(color="#ffffff",width=2,dash="dash"),
                  annotation_text=f"Median {med:+.1f}",
                  annotation_font=dict(color="#ffffff",size=10))
    fig.add_vline(x=0,line=dict(color=COL_ZERO,width=1,dash="dot"))
    lr = LIVE["returns"].get(lbl)
    if lr is not None and dn in lr.index:
        live_now = float(lr.loc[dn])
        fig.add_vline(x=live_now,line=dict(color="#ffa657",width=2,dash="dot"),
                      annotation_text=f"Live now {live_now:+.1f}",
                      annotation_font=dict(color="#ffa657",size=10),
                      annotation_position="bottom right")
    fig.update_layout(**dark_layout(
        title=f"<b>{dlbl}</b> — Per-Analogue Returns: Day+{dn} → {horiz_lbl}",
        xaxis=dict(title=f"Forward Return ({unit})",gridcolor=COL_GRID,
                   zerolinecolor=COL_ZERO,linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title="",gridcolor=COL_GRID,linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED,size=9)),
        showlegend=False,width=720,height=max(280,len(vals)*36+100),
        margin=dict(l=160,r=40,t=60,b=50)))
    return fig

def _live_deviation_fig(lbl, sel_ev, cutoff):
    """Live path vs analogue fan with absolute gap and percentile band."""
    dn   = LIVE["day_n"] or 0
    unit = unit_label(lbl)
    dlbl = display_label(lbl)
    offs = list(range(0,dn+1))
    if not offs: return None

    paths = {}
    for en in sel_ev:
        s = event_returns.get(lbl,{}).get(en)
        if s is None: continue
        paths[en] = [float(s.get(o,np.nan)) for o in offs]

    if not paths: return None
    mat  = np.array(list(paths.values()))
    p25  = np.nanpercentile(mat,25,axis=0)
    p75  = np.nanpercentile(mat,75,axis=0)
    comp = np.nanmedian(mat,axis=0)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=offs+offs[::-1],
        y=list(p75)+list(p25[::-1]),
        fill="toself",
        fillcolor="rgba(88,166,255,0.12)",
        line=dict(color="rgba(0,0,0,0)"),
        name="IQR band (25–75th)",showlegend=True))
    fig.add_trace(go.Scatter(x=offs,y=comp,name="Composite median",
        mode="lines",line=dict(color="#ffffff",width=2,dash="dash"),
        hovertemplate="Day %{x}: %{y:+.2f} "+unit+"<extra></extra>"))
    lr = LIVE["returns"].get(lbl)
    if lr is not None:
        live_y = [lr.get(o,np.nan) for o in offs]
        fig.add_trace(go.Scatter(x=offs,y=live_y,
            name=f"▶ {LIVE['name']}",mode="lines+markers",
            line=dict(color="#ffa657",width=3),
            marker=dict(size=6,color="#ffa657"),
            hovertemplate="Day %{x}: %{y:+.2f} "+unit+"<extra></extra>"))
        if dn in lr.index:
            live_v  = float(lr.loc[dn])
            comp_v  = float(comp[-1]) if not np.isnan(comp[-1]) else np.nan
            gap     = live_v - comp_v if not np.isnan(comp_v) else np.nan
            pctile  = float(np.mean([1 if live_v>v else 0
                                      for v in mat[:,dn] if not np.isnan(v)]))*100
            ann_txt = (f"Gap vs composite: {gap:+.1f}{unit}<br>"
                       f"Percentile rank: {pctile:.0f}th")
            fig.add_annotation(x=dn,y=live_v,text=ann_txt,
                showarrow=True,arrowhead=2,arrowcolor="#ffa657",
                bgcolor=BG_CELL,bordercolor="#ffa657",
                font=dict(color="#ffa657",size=10))

    fig.update_layout(**dark_layout(
        title=f"<b>{dlbl}</b> — Live vs Analogue Distribution",
        xaxis=dict(title="Trading days from Day 0",gridcolor=COL_GRID,
                   zerolinecolor=COL_GRID,linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title=f"Return ({unit})",gridcolor=COL_GRID,
                   zerolinecolor=COL_ZERO,linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        legend=dict(x=0.01,y=0.99,bgcolor=BG_PANEL,bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT,size=10)),
        hovermode="x unified",width=720,height=360,
        margin=dict(r=40,t=60,b=50)))
    return fig

def _corr_heatmap_fig(rows, sel_ev, cutoff):
    """Correlation of forward returns across top ideas."""
    dn  = LIVE["day_n"] or 0
    fo  = next((o for l,o in POIS if l==_horiz_cache),None)
    if fo is None or fo <= dn or len(rows) < 2: return None
    top  = rows[:min(12,len(rows))]
    lbls = [display_label(r["lbl"]) for r in top]
    mat  = []
    for r in top:
        row = []
        for en in sel_ev:
            sv=poi_ret(r["lbl"],en,dn); fv=poi_ret(r["lbl"],en,fo)
            row.append(fv-sv if not any(np.isnan([sv,fv])) else np.nan)
        mat.append(row)
    mat = np.array(mat)
    valid_cols = ~np.any(np.isnan(mat),axis=0)
    mat = mat[:,valid_cols]
    if mat.shape[1] < 2: return None
    try:
        C = np.corrcoef(mat)
    except Exception:
        return None
    text = [[f"{C[i,j]:.2f}" for j in range(len(top))] for i in range(len(top))]
    fig = go.Figure(go.Heatmap(
        z=C,x=lbls,y=lbls,text=text,texttemplate="%{text}",
        colorscale="RdBu",zmid=0,zmin=-1,zmax=1,
        colorbar=dict(title=dict(text="Corr",font=dict(color=COL_TEXT)),
                      thickness=12,tickfont=dict(color=COL_TEXT)),
        hovertemplate="<b>%{y}</b> vs <b>%{x}</b>: %{text}<extra></extra>"))
    fig.update_layout(**dark_layout(
        title=f"Top-{len(top)} Ideas — Return Correlation Across {mat.shape[1]} Analogues",
        xaxis=dict(tickangle=35,gridcolor=COL_GRID,linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED,size=9)),
        yaxis=dict(autorange="reversed",gridcolor=COL_GRID,linecolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED,size=9)),
        width=720,height=max(380,len(top)*50+120),
        margin=dict(t=60,b=120,l=120,r=40)))
    return fig

# ── Widgets ───────────────────────────────────────────────────
wl5_hor = widgets.Dropdown(
    options=[l for l,o in POIS if o>0],value="t+1M",
    description="Horizon:",layout=widgets.Layout(width="160px"),
    style={"description_width":"65px"})
wl5_cut = widgets.FloatSlider(value=0.50,min=0.0,max=1.0,step=0.01,
    description="Cutoff:",continuous_update=False,
    layout=widgets.Layout(width="320px"),style={"description_width":"60px"})
wl5_grp = widgets.Dropdown(
    options=["— All Assets —"]+sorted(CUSTOM_GROUPS.keys())+ALL_CLASSES,
    value="— All Assets —",description="Group:",
    layout=widgets.Layout(width="260px"),style={"description_width":"60px"})
wl5_ast = widgets.Dropdown(description="Detail Asset:",
    layout=widgets.Layout(width="260px"),style={"description_width":"90px"})
wl5_out_tbl  = widgets.Output()
wl5_out_det  = widgets.Output()
_l5_rows = []
wl5_tbl_state = make_refresh_state()
wl5_det_state = make_refresh_state()

def _wl5_update_table(*_):
    global _l5_rows
    key = (wl5_grp.value, wl5_hor.value, float(wl5_cut.value), LIVE.get("name"), LIVE.get("day_n"))
    def _render():
        global _l5_rows
        with wl5_out_tbl:
            clear_output(wait=True)
            if LIVE["returns"] is None: print("❌ Run Cell L1 first."); return
            g = wl5_grp.value
            if g=="— All Assets —":    lb=get_subset("all")
            elif g in CUSTOM_GROUPS:   lb=get_subset("group",group=g)
            else:                      lb=get_subset("class",cls=g)
            rows = _compute_rows(lb,wl5_cut.value,wl5_hor.value)
            _l5_rows = rows
            opts = [display_label(r["lbl"])+" — "+r["dir"] for r in rows]
            wl5_ast.options = opts if opts else ["(no data)"]
            if opts: wl5_ast.value = opts[0]
            fig = _summary_table_fig(rows,wl5_hor.value,wl5_cut.value)
            if fig: fig.show()
    run_guarded_refresh(wl5_tbl_state, key, _render, label="Standalone detail table")


def _wl5_update_detail(*_):
    if not _l5_rows or not wl5_ast.value or wl5_ast.value=="(no data)": return
    key = (wl5_ast.value, wl5_hor.value, float(wl5_cut.value), len(_l5_rows))
    def _render():
        with wl5_out_det:
            clear_output(wait=True)
            idx = wl5_ast.options.index(wl5_ast.value) if wl5_ast.value in wl5_ast.options else 0
            if idx >= len(_l5_rows): return
            row = _l5_rows[idx]
            lbl = row["lbl"]
            sel = _sel_ev(wl5_cut.value)
            cut = wl5_cut.value
            hor = wl5_hor.value
            display(widgets.HTML(
                f"<h4 style='color:#ffa657;margin:4px 0'>"
                f"▶ {display_label(lbl)}  |  {row['dir']}  |  "
                f"{row['status']}  |  Conv: {row['stars']}</h4>"))
            f1 = _decay_fig(lbl,sel,cut)
            f2 = _dot_plot_fig(lbl,sel,hor,cut)
            f3 = _live_deviation_fig(lbl,sel,cut)
            f4 = _corr_heatmap_fig(_l5_rows,sel,cut)
            if f1: f1.show()
            if f2: f2.show()
            if f3: f3.show()
            if f4:
                display(widgets.HTML("<h4 style='color:#c9d1d9;margin:8px 0'>Intra-Idea Correlation</h4>"))
                f4.show()
    run_guarded_refresh(wl5_det_state, key, _render, label="Standalone detail charts")

wl5_hor.observe(lambda c:_wl5_update_table() if c.get("name")=="value" else None,names="value")
wl5_cut.observe(lambda c:_wl5_update_table() if c.get("name")=="value" else None,names="value")
wl5_grp.observe(lambda c:_wl5_update_table() if c.get("name")=="value" else None,names="value")
wl5_ast.observe(lambda c:_wl5_update_detail() if c.get("name")=="value" else None,names="value")
wl5_tbl_state["ready"] = True
wl5_det_state["ready"] = True


---

In [ ]:
# §5.6 — C4: MAX ADVERSE EXCURSION (MAE) & DRAWDOWN PATH
# ─────────────────────────────────────────────────────────────
# For each asset × analogue × forward window: worst daily close
# between entry (Day+dn) and exit (Day+fo), and the full path.
# Used by A1 screener and E2 portfolio stress.

def _mae_for_asset(lbl, sel_ev, dn, fo):
    """Return dict with MAE stats across all selected analogues.
    mae_vals : list of worst intra-period drawdown (always <=0)
    path_mat : list of return paths from dn to fo (each is a list)
    """
    mae_vals = []
    path_mat = []
    for en in sel_ev:
        s = event_returns.get(lbl, {}).get(en)
        if s is None: continue
        window = s[(s.index >= dn) & (s.index <= fo)]
        if len(window) < 2: continue
        # Normalise path to start at 0 at dn
        v0 = poi_ret(lbl, en, dn)
        if np.isnan(v0): continue
        path = (window - v0).values.tolist()
        path_mat.append(path)
        # MAE = worst point in path relative to entry
        if len(path) > 0:
            mae_vals.append(min(path))  # most negative = worst drawdown
    if not mae_vals:
        return {"mae_med": np.nan, "mae_worst": np.nan,
                "mae_vals": [], "path_mat": []}
    return {
        "mae_med":   float(np.median(mae_vals)),
        "mae_worst": float(np.min(mae_vals)),
        "mae_vals":  mae_vals,
        "path_mat":  path_mat,
    }

def _mae_ratio(med_fwd, mae_med):
    """Reward/risk ratio: median forward return / |median MAE|.
    Higher = better risk-adjusted entry. NaN if no MAE data."""
    if np.isnan(mae_med) or abs(mae_med) < 1e-9: return np.nan
    return med_fwd / abs(mae_med)


In [ ]:
# §5.7 — C3: CROSS-ASSET LEAD-LAG MATRIX
# ─────────────────────────────────────────────────────────────
# For each pair of assets across selected analogues, compute
# the average offset at which one asset's move precedes another.
# Resolution: POIS offsets. Toggle to daily in the tab widget.
# Output: NxN matrix — entry (i,j) = avg days asset_i leads asset_j.
# Positive = asset_i moves first; negative = asset_j moves first.

def _leadlag_matrix(labels, sel_ev, offsets=None):
    """Compute lead-lag matrix at given offsets (POIS by default).
    Returns (matrix np.ndarray, labels list).
    """
    if offsets is None:
        offsets = sorted(set(o for _,o in POIS if o >= 0))
    n = len(labels)
    # For each analogue, get return series at each offset
    # lead_lag[i][j] = mean signed offset difference when i moved before j
    cumulative = np.zeros((n, n))
    counts     = np.zeros((n, n))
    for en in sel_ev:
        # Get return at each offset for each asset (cumulative from Day0)
        ret_mat = []  # shape: (n_assets, n_offsets)
        for lbl in labels:
            row = [poi_ret(lbl, en, o) for o in offsets]
            ret_mat.append(row)
        ret_mat = np.array(ret_mat)  # (n, n_offsets)

        # For each asset pair, find which POI showed the larger move first
        for i in range(n):
            for j in range(n):
                if i == j: continue
                ri = ret_mat[i]
                rj = ret_mat[j]
                if np.all(np.isnan(ri)) or np.all(np.isnan(rj)): continue
                # Find first POI where |move| exceeds 1 unit for each asset
                def _first_move(r):
                    for k, v in enumerate(r):
                        if not np.isnan(v) and abs(v) > 1.0:
                            return offsets[k]
                    return None
                fi = _first_move(ri); fj = _first_move(rj)
                if fi is not None and fj is not None:
                    cumulative[i,j] += fi - fj   # negative = i moved before j
                    counts[i,j] += 1
    with np.errstate(invalid="ignore"):
        mat = np.where(counts > 0, cumulative / counts, np.nan)
    return mat, labels

def _leadlag_fig(labels, sel_ev, offsets=None, title_suffix=""):
    mat, lbls = _leadlag_matrix(labels, sel_ev, offsets)
    n = len(lbls)
    short = [display_label(l)[:14] for l in lbls]
    # Annotation text
    txt = [[f"{mat[i,j]:+.1f}d" if not np.isnan(mat[i,j]) else ""
            for j in range(n)] for i in range(n)]
    fig = go.Figure(go.Heatmap(
        z=mat, x=short, y=short, text=txt,
        texttemplate="%{text}",
        colorscale=[[0,"#1f6feb"],[0.5,"#21262d"],[1,"#da3633"]],
        zmid=0, zmin=-20, zmax=20,
        colorbar=dict(title="Days<br>lead(+)/lag(-)",
                      tickfont=dict(color=COL_MUTED, size=9),
                      titlefont=dict(color=COL_MUTED, size=10)),
        hovertemplate="<b>%{y}</b> leads <b>%{x}</b> by %{z:.1f}d<extra></extra>",
    ))
    fig.update_layout(**dark_layout(
        title=f"<b>Lead-Lag Matrix</b> — {len(sel_ev)} analogues{title_suffix}",
        xaxis=dict(tickangle=45, tickfont=dict(color=COL_MUTED, size=9)),
        yaxis=dict(tickfont=dict(color=COL_MUTED, size=9)),
        width=CHART_W, height=max(420, n*32+120),
        margin=dict(b=120, l=140, r=80)))
    return fig


In [ ]:
# §5.8 — B3: REVERSE ANALOGUE LOOKUP
# ─────────────────────────────────────────────────────────────
# Pattern-first matching: given the current cross-asset return
# pattern in LIVE["returns"], find the best historical analogues
# purely on quant cosine similarity — no event label required.
# Reads from LIVE automatically. The event label from L1 config
# is ignored in scoring (tag and macro weights set to 0).
# Useful for unlabelled events or as a pure market-pattern check.

def run_reverse_lookup(assets=None):
    """Run analogue match using ONLY path cosine similarity.
    assets: list of asset labels to use (default: SIMILARITY_ASSET_POOL)
    Returns list of score dicts sorted by path_quant descending.
    """
    if not _check_live(): return None
    dn = LIVE["day_n"] or 0
    sa = assets if assets else LIVE["sim_assets"]

    # Build live path vector (same as run_analogue_match B1)
    def _live_val(a, dn):
        lr = LIVE["returns"].get(a)
        if lr is None or len(lr) == 0: return np.nan
        idx_arr = lr.index[np.abs(np.array(lr.index, dtype=float) - dn) <= 2]
        if len(idx_arr) == 0: return np.nan
        below = idx_arr[idx_arr <= dn]
        best  = below[-1] if len(below) else idx_arr[0]
        return float(lr.loc[best])

    def _path_vec_rev(returns_dict, assets, dn):
        offsets = list(range(0, dn + 1))
        vecs = []
        for a in assets:
            if a not in returns_dict:
                vecs.append(np.zeros(len(offsets)))
                continue
            s = returns_dict[a]
            row = []
            for o in offsets:
                idx_arr = s.index[np.abs(np.array(s.index, dtype=float) - o) <= 1]
                if len(idx_arr):
                    below = idx_arr[idx_arr <= o]
                    best  = below[-1] if len(below) else idx_arr[0]
                    row.append(float(s.loc[best]))
                else:
                    row.append(0.0)
            vecs.append(row)
        return np.array(vecs).flatten()

    live_path = _path_vec_rev(LIVE["returns"], sa, dn)
    scores = []
    for en, _ in EVENTS:
        hist_dict = {a: event_returns.get(a,{}).get(en, pd.Series(dtype=float)) for a in sa}
        hist_path = _path_vec_rev(hist_dict, sa, dn)
        pq = (_cosine(live_path, hist_path) + 1) / 2
        scores.append({"event": en, "path_quant": pq,
                        "tag": _tag_sim(LIVE["tags"], EVENT_TAGS.get(en, set())),
                        "macro_cpi": MACRO_CONTEXT.get(en,{}).get("cpi","?"),
                        "macro_fed": MACRO_CONTEXT.get(en,{}).get("fed","?")})
    scores.sort(key=lambda x: x["path_quant"], reverse=True)
    return scores

def _reverse_fig(scores, top_n=None):
    if not scores: return None
    if top_n: scores = scores[:top_n]
    evts  = [s["event"] for s in scores]
    pqs   = [s["path_quant"] for s in scores]
    tags  = [f"CPI:{s['macro_cpi']} Fed:{s['macro_fed']}" for s in scores]
    colors = [f"rgba(88,166,255,{0.3+pq*0.7:.2f})" for pq in pqs]
    fig = go.Figure(go.Bar(
        x=evts, y=pqs, text=[f"{v:.3f}" for v in pqs],
        textposition="outside",
        marker_color=colors,
        hovertemplate="<b>%{x}</b><br>Path Similarity: %{y:.3f}<br>%{customdata}<extra></extra>",
        customdata=tags,
    ))
    fig.add_hline(y=0.5, line=dict(color="#ff7b72", width=1.5, dash="dash"),
                  annotation_text="0.50", annotation_font=dict(color="#ff7b72"))
    fig.update_layout(**dark_layout(
        title=f"<b>Reverse Pattern Lookup</b> — Day+{LIVE.get('day_n',0)} "
              f"path similarity (no event label)",
        xaxis=dict(tickangle=35, tickfont=dict(color=COL_MUTED, size=9)),
        yaxis=dict(title="Path Cosine Similarity", range=[0,1.05],
                   gridcolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        width=CHART_W, height=420, margin=dict(b=140)))
    return fig


In [ ]:
# §5.9 — A1+A3+A4: SIGNAL SCREENER
# ─────────────────────────────────────────────────────────────
# A1: conviction filter — hit rate, coverage, MAE, vol-adj
# A3: disagreement/bimodal flag — analogue direction split
# A4: correlation cluster — flag redundant exposure

def _screener_rows(labels, cutoff, fwd_days,
                   min_hit=0.60, min_cov=0.50, min_rr=0.80):
    """Compute screener rows with A1/A3/A4 signals.
    min_hit : minimum hit rate to pass
    min_cov : minimum analogue coverage fraction
    min_rr  : minimum reward/risk ratio (|median fwd| / |MAE median|)
    Returns list of dicts with all signal fields.
    """
    if not _check_live(): return []
    _check_scores()
    sel = _sel_ev(cutoff)
    n_sel = len(sel)
    if n_sel == 0: return []
    dn  = LIVE["day_n"] or 0
    fo  = dn + int(fwd_days)
    rows = []
    for lbl in labels:
        fwd_vals = []
        for en in sel:
            sv = poi_ret(lbl, en, dn)
            fv = poi_ret(lbl, en, fo)
            if not any(np.isnan([sv, fv])): fwd_vals.append(fv - sv)
        n_cov = len(fwd_vals)
        cov   = n_cov / max(n_sel, 1)
        if n_cov < 2: continue

        med      = float(np.median(fwd_vals))
        mn       = float(np.mean(fwd_vals))
        sd       = float(np.std(fwd_vals))
        hit_rate = float(np.mean([1 if (med>0 and v>0) or (med<0 and v<0)
                                  else 0 for v in fwd_vals]))

        # A3 — disagreement: fraction of analogues against the median direction
        n_against = sum(1 for v in fwd_vals if (med>0 and v<0) or (med<0 and v>0))
        disagree_frac = n_against / max(n_cov, 1)
        # Bimodal flag: disagreement > 35% AND std/|med| > 1.5
        bimodal = disagree_frac > 0.35 and (sd / (abs(med)+1e-9)) > 1.5

        # C4 MAE
        mae_d = _mae_for_asset(lbl, sel, dn, fo)
        mae_med   = mae_d["mae_med"]
        mae_worst = mae_d["mae_worst"]
        rr_ratio  = _mae_ratio(med, mae_med)

        # A1 conviction gate
        conviction = "🔴 Skip"
        if (hit_rate >= min_hit and cov >= min_cov
                and not bimodal
                and (np.isnan(rr_ratio) or rr_ratio >= min_rr)):
            if hit_rate >= 0.75 and cov >= 0.70:
                conviction = "🟢 Act"
            else:
                conviction = "🟡 Monitor"
        elif bimodal:
            conviction = "⚠️ Bimodal"

        rows.append({
            "lbl":          lbl,
            "direction":    "▲ Long" if med > 0 else "▼ Short",
            "med":          med,
            "hit_rate":     hit_rate,
            "cov":          cov,
            "n_cov":        n_cov,
            "mae_med":      mae_med,
            "mae_worst":    mae_worst,
            "rr_ratio":     rr_ratio,
            "disagree_frac":disagree_frac,
            "bimodal":      bimodal,
            "conviction":   conviction,
            "fwd_vals":     fwd_vals,
            "unit":         unit_label(lbl),
        })
    # A4 — correlation clustering: flag assets with |corr| > 0.85 to a higher-ranked asset
    # Build correlation matrix from fwd_vals across assets that passed
    act_rows = [r for r in rows if r["conviction"] in ("🟢 Act","🟡 Monitor")]
    if len(act_rows) >= 2:
        lbls_act = [r["lbl"] for r in act_rows]
        # Align fwd_vals to common analogues
        fv_map = {r["lbl"]: r["fwd_vals"] for r in act_rows}
        min_len = min(len(v) for v in fv_map.values())
        if min_len >= 3:
            mat = np.array([fv_map[l][:min_len] for l in lbls_act])
            with np.errstate(invalid="ignore"):
                corr = np.corrcoef(mat)
            for i, r in enumerate(act_rows):
                redundant_with = []
                for j in range(i):
                    if abs(corr[i,j]) > 0.85:
                        redundant_with.append(f"{display_label(lbls_act[j])} ({corr[i,j]:+.2f})")
                r["redundant"] = ", ".join(redundant_with) if redundant_with else ""
        else:
            for r in act_rows: r["redundant"] = ""
    for r in rows:
        if "redundant" not in r: r["redundant"] = ""

    rows.sort(key=lambda x: (
        {"🟢 Act":0,"🟡 Monitor":1,"⚠️ Bimodal":2,"🔴 Skip":3}
        .get(x["conviction"],4), -x["hit_rate"]))
    return rows

def _screener_fig(rows, fwd_days, cutoff):
    if not rows: return None
    act   = [r for r in rows if r["conviction"] != "🔴 Skip"]
    skip  = [r for r in rows if r["conviction"] == "🔴 Skip"]
    show  = act + skip[:10]  # cap skipped to keep table manageable
    cols  = ["<b>Asset</b>","<b>Signal</b>","<b>Median</b>",
             "<b>Hit%</b>","<b>Cov</b>","<b>MAE med</b>",
             "<b>R/R</b>","<b>Bimodal?</b>","<b>Redundant with</b>"]
    color_map = {"🟢 Act":"rgba(63,185,80,0.5)",
                 "🟡 Monitor":"rgba(210,153,34,0.4)",
                 "⚠️ Bimodal":"rgba(248,81,73,0.3)",
                 "🔴 Skip":"rgba(33,38,45,0.7)"}
    rc = [color_map.get(r["conviction"],"rgba(33,38,45,0.7)") for r in show]
    def _fmt(v, fmt): return fmt%v if not np.isnan(v) else "—"
    fig = go.Figure(go.Table(
        header=dict(values=cols, fill_color="#1c2128",
                    font=dict(color=COL_HEAD,size=11,family="monospace"),
                    align="center", height=30, line=dict(color=COL_GRID,width=1)),
        cells=dict(
            values=[
                [display_label(r["lbl"]) for r in show],
                [r["conviction"] for r in show],
                [f"{r['med']:+.2f}{r['unit']}" for r in show],
                [f"{r['hit_rate']:.0%}" for r in show],
                [f"{r['cov']:.0%} ({r['n_cov']})" for r in show],
                [_fmt(r["mae_med"],"%+.2f") for r in show],
                [_fmt(r["rr_ratio"],"%.2f") for r in show],
                ["⚠️" if r["bimodal"] else "" for r in show],
                [r["redundant"] or "—" for r in show],
            ],
            fill_color=[rc]*9,
            font=dict(color=COL_HEAD,size=10,family="monospace"),
            align="center", height=26,
            line=dict(color=COL_GRID,width=0.5))))
    n_act = sum(1 for r in rows if r["conviction"]=="🟢 Act")
    n_mon = sum(1 for r in rows if r["conviction"]=="🟡 Monitor")
    fig.update_layout(
        title=dict(text=(f"Signal Screener — cutoff {cutoff:.2f} | "
                         f"+{fwd_days}d horizon | "
                         f"🟢 {n_act} Act  🟡 {n_mon} Monitor"),
                   font=dict(color=COL_HEAD,size=13)),
        paper_bgcolor=BG, font=dict(color=COL_TEXT,family="monospace"),
        width=CHART_W, height=max(400, len(show)*26+120),
        margin=dict(t=60,b=20,l=10,r=10))
    return fig

_screener_cache = []


In [ ]:
# §5.10 — D1: PRE-POSITIONING PLAYBOOK
# ─────────────────────────────────────────────────────────────
# What did assets do in the N trading days BEFORE Day 0 across analogues?
# Tells you how markets positioned (or failed to position) ahead of the shock.
# Uses PRE_WINDOW data already in event_returns (negative offsets).
# NO forward-looking signal — purely historical pre-event behaviour.

def _prepos_rows(labels, sel_ev, pre_days=10):
    """Compute pre-event return stats for each asset.
    pre_days: trading days before Day 0 to look back (uses -pre_days offset).
    Returns list of dicts with pre-event stats.
    """
    rows = []
    entry_offset = -pre_days  # e.g. -10 for 10 days before Day 0
    for lbl in labels:
        pre_vals = []
        for en in sel_ev:
            # Return from -pre_days to Day 0
            sv = poi_ret(lbl, en, entry_offset)
            fv = poi_ret(lbl, en, 0)
            if not any(np.isnan([sv, fv])):
                pre_vals.append(fv - sv)
        if len(pre_vals) < 2: continue
        med      = float(np.median(pre_vals))
        mn       = float(np.mean(pre_vals))
        sd       = float(np.std(pre_vals))
        hit_rate = float(np.mean([1 if (med>0 and v>0) or (med<0 and v<0)
                                  else 0 for v in pre_vals]))
        rows.append({
            "lbl":      lbl,
            "med":      med,
            "mean":     mn,
            "std":      sd,
            "hit_rate": hit_rate,
            "n":        len(pre_vals),
            "unit":     unit_label(lbl),
            "pre_vals": pre_vals,
        })
    rows.sort(key=lambda x: abs(x["med"]), reverse=True)
    return rows

def _prepos_fig(rows, sel_ev, pre_days):
    if not rows: return None
    show = rows[:30]
    lbls = [display_label(r["lbl"])[:18] for r in show]
    meds = [r["med"] for r in show]
    hits = [r["hit_rate"] for r in show]
    ns   = [r["n"] for r in show]
    colors = ["rgba(63,185,80,0.7)" if m>0 else "rgba(248,81,73,0.7)" for m in meds]
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=lbls, y=meds,
        text=[f"{v:+.1f}" for v in meds], textposition="outside",
        marker_color=colors,
        hovertemplate="<b>%{x}</b><br>Pre-event median: %{y:+.2f}<br>"
                      "Hit rate: %{customdata[0]:.0%}  N={%{customdata[1]}}<extra></extra>",
        customdata=list(zip(hits, ns)),
        name="Median pre-event return"
    ))
    fig.add_hline(y=0, line=dict(color=COL_ZERO, width=1, dash="dash"))
    fig.update_layout(**dark_layout(
        title=(f"<b>Pre-Positioning Playbook</b> — {pre_days}d before Day 0 | "
               f"{len(sel_ev)} analogues"),
        xaxis=dict(tickangle=45, tickfont=dict(color=COL_MUTED, size=9)),
        yaxis=dict(title="Return (% or bps)", gridcolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        width=CHART_W, height=460, margin=dict(b=140)))
    return fig

def _prepos_heatmap(labels, sel_ev, pre_days=10):
    """Heatmap: assets × analogues, pre-event return colour."""
    entry_offset = -pre_days
    mat = []
    txt = []
    shown_lbls = []
    for lbl in labels:
        row_v, row_t = [], []
        for en in sel_ev:
            sv = poi_ret(lbl, en, entry_offset)
            fv = poi_ret(lbl, en, 0)
            val = fv - sv if not any(np.isnan([sv,fv])) else np.nan
            row_v.append(val)
            row_t.append(f"{val:+.1f}" if not np.isnan(val) else "")
        if sum(1 for v in row_v if not np.isnan(v)) < 1: continue
        mat.append(row_v); txt.append(row_t)
        shown_lbls.append(display_label(lbl)[:16])
    if not mat: return None
    mat = np.array(mat)
    vmax = np.nanpercentile(np.abs(mat), 95) or 5
    fig = go.Figure(go.Heatmap(
        z=mat, x=sel_ev, y=shown_lbls, text=txt, texttemplate="%{text}",
        colorscale=[[0,"#da3633"],[0.5,"#21262d"],[1,"#238636"]],
        zmid=0, zmin=-vmax, zmax=vmax,
        colorbar=dict(title="Pre-event<br>return",
                      tickfont=dict(color=COL_MUTED,size=9),
                      titlefont=dict(color=COL_MUTED,size=10)),
        hovertemplate="<b>%{y}</b> | %{x}<br>Pre-event: %{z:+.2f}<extra></extra>",
    ))
    fig.update_layout(**dark_layout(
        title=f"<b>Pre-Event Heatmap</b> — {pre_days}d window | {len(sel_ev)} analogues",
        xaxis=dict(tickangle=35, tickfont=dict(color=COL_MUTED,size=9)),
        yaxis=dict(tickfont=dict(color=COL_MUTED,size=9)),
        width=CHART_W, height=max(360, len(shown_lbls)*22+120),
        margin=dict(b=120,l=160,r=80)))
    return fig


In [ ]:
# §5.11 — E2: PORTFOLIO STRESS TESTER
# ─────────────────────────────────────────────────────────────
# Given a portfolio of positions (from PORTFOLIO_SCENARIOS),
# compute what each selected analogue would have done to the book.
# Output: PnL distribution per scenario, per analogue, and combined.
# Notional in USD. Returns in % converted to $ PnL.

def _stress_portfolio(portfolio, sel_ev, dn, fwd_days):
    """Stress a portfolio dict {label: notional_usd} against analogues.
    Returns dict with per-analogue PnL and aggregate stats.
    """
    fo = dn + int(fwd_days)
    results = {}  # event_name → total_pnl
    asset_contribs = {}  # event_name → {label: pnl}

    for en in sel_ev:
        total_pnl = 0.0
        contribs  = {}
        for lbl, notional in portfolio.items():
            sv = poi_ret(lbl, en, dn)
            fv = poi_ret(lbl, en, fo)
            if any(np.isnan([sv, fv])): continue
            ret_pct = (fv - sv) / 100.0  # convert % to decimal
            pnl     = notional * ret_pct
            total_pnl += pnl
            contribs[lbl] = pnl
        results[en]         = total_pnl
        asset_contribs[en]  = contribs

    pnl_vals = [v for v in results.values() if not np.isnan(v)]
    if not pnl_vals:
        return {"pnl_by_event": results, "asset_contribs": asset_contribs,
                "med_pnl": np.nan, "worst_pnl": np.nan, "best_pnl": np.nan,
                "hit_rate": np.nan, "n": 0}
    med_pnl  = float(np.median(pnl_vals))
    return {
        "pnl_by_event":   results,
        "asset_contribs": asset_contribs,
        "med_pnl":        med_pnl,
        "worst_pnl":      float(np.min(pnl_vals)),
        "best_pnl":       float(np.max(pnl_vals)),
        "hit_rate":       float(np.mean([1 if p>0 else 0 for p in pnl_vals])),
        "n":              len(pnl_vals),
    }

def _stress_fig(scenarios, sel_ev, dn, fwd_days):
    """Bar chart: per-scenario PnL by analogue + summary stats table."""
    if not scenarios: return None, None
    # Compute results for all scenarios
    all_results = {}
    for name, portfolio in scenarios.items():
        all_results[name] = _stress_portfolio(portfolio, sel_ev, dn, fwd_days)

    # ── Chart: PnL per analogue per scenario ──────────────────
    fig1 = go.Figure()
    colors = ["#58a6ff","#3fb950","#e3b341","#f78166","#a371f7","#79c0ff"]
    for i, (name, res) in enumerate(all_results.items()):
        evts = list(res["pnl_by_event"].keys())
        pnls = [res["pnl_by_event"].get(e, 0)/1000 for e in evts]  # $k
        fig1.add_trace(go.Bar(
            name=name, x=evts, y=pnls,
            marker_color=colors[i % len(colors)],
            hovertemplate=f"<b>{name}</b><br>%{{x}}: $%{{y:.0f}}k<extra></extra>",
        ))
    fig1.add_hline(y=0, line=dict(color=COL_ZERO,width=1,dash="dash"))
    fig1.update_layout(**dark_layout(
        title=f"<b>Portfolio Stress</b> — {len(sel_ev)} analogues | Day+{dn}→+{dn+fwd_days}",
        xaxis=dict(tickangle=35,tickfont=dict(color=COL_MUTED,size=9)),
        yaxis=dict(title="PnL ($k)",gridcolor=COL_GRID,tickfont=dict(color=COL_MUTED)),
        barmode="group", width=CHART_W, height=460, margin=dict(b=140)))

    # ── Summary table ──────────────────────────────────────────
    rows_tbl = []
    for name, res in all_results.items():
        rows_tbl.append([name,
            f"${res['med_pnl']/1000:+.0f}k" if not np.isnan(res["med_pnl"]) else "—",
            f"${res['worst_pnl']/1000:+.0f}k" if not np.isnan(res["worst_pnl"]) else "—",
            f"${res['best_pnl']/1000:+.0f}k" if not np.isnan(res["best_pnl"]) else "—",
            f"{res['hit_rate']:.0%}" if not np.isnan(res["hit_rate"]) else "—",
            str(res["n"])])
    cols_tbl = ["<b>Scenario</b>","<b>Median PnL</b>","<b>Worst</b>",
                "<b>Best</b>","<b>Hit%</b>","<b>N</b>"]
    rc = [("rgba(63,185,80,0.3)" if float(r[1].replace("$","").replace("k","").replace("+","") or 0)>0
           else "rgba(248,81,73,0.25)") for r in rows_tbl]
    fig2 = go.Figure(go.Table(
        header=dict(values=cols_tbl,fill_color="#1c2128",
                    font=dict(color=COL_HEAD,size=11,family="monospace"),
                    align="center",height=30,line=dict(color=COL_GRID,width=1)),
        cells=dict(values=list(zip(*rows_tbl)) if rows_tbl else [[]]*len(cols_tbl),
                   fill_color=[rc]*len(cols_tbl),
                   font=dict(color=COL_HEAD,size=11,family="monospace"),
                   align="center",height=28,line=dict(color=COL_GRID,width=0.5))))
    fig2.update_layout(
        title=dict(text="Scenario Summary", font=dict(color=COL_HEAD,size=13)),
        paper_bgcolor=BG, font=dict(color=COL_TEXT,family="monospace"),
        width=CHART_W, height=max(300, len(rows_tbl)*32+120),
        margin=dict(t=60,b=20,l=10,r=10))
    return fig1, fig2


In [ ]:
# §5.12 — E3: SIGNAL DECAY TRACKER
# ─────────────────────────────────────────────────────────────
# Computes analogue ranking at every day offset 0..current_day_n.
# Shows how the top analogues shift over time — when the best-fit
# analogue changes, it signals the event is evolving differently.
# Uses path similarity (B1) at each day offset snapshot.

def _decay_scores_at(dn_target):
    """Run path similarity scoring at a specific day offset.
    Returns sorted list of {event, path_quant} same format as run_reverse_lookup.
    """
    if not _check_live(): return []
    sa = LIVE["sim_assets"]

    def _path_vec_at(returns_dict, assets, dn):
        if dn < 0: return np.zeros(len(assets))
        offsets = list(range(0, dn + 1))
        vecs = []
        for a in assets:
            if a not in returns_dict:
                vecs.append(np.zeros(len(offsets)))
                continue
            s = returns_dict[a]
            row = []
            for o in offsets:
                idx_arr = s.index[np.abs(np.array(s.index, dtype=float) - o) <= 1]
                if len(idx_arr):
                    below = idx_arr[idx_arr <= o]
                    best  = below[-1] if len(below) else idx_arr[0]
                    row.append(float(s.loc[best]))
                else:
                    row.append(0.0)
            vecs.append(row)
        return np.array(vecs).flatten()

    live_path = _path_vec_at(LIVE["returns"], sa, dn_target)
    scores = []
    for en, _ in EVENTS:
        hist_dict = {a: event_returns.get(a,{}).get(en, pd.Series(dtype=float)) for a in sa}
        hist_path = _path_vec_at(hist_dict, sa, dn_target)
        pq = (_cosine(live_path, hist_path) + 1) / 2
        scores.append({"event": en, "path_quant": pq})
    scores.sort(key=lambda x: x["path_quant"], reverse=True)
    return scores

def _decay_fig(max_dn=None, step=1):
    """Timeline of top-3 analogue ranks across Day 0 to max_dn.
    Shows when rank order shifts — signals event evolution.
    """
    if not _check_live(): return None
    real_dn = max_dn if max_dn is not None else LIVE.get("day_n", 0)
    if real_dn < 1: return None

    offsets   = list(range(0, real_dn + 1, max(step,1)))
    # Get all event names
    all_evts  = [e for e,_ in EVENTS]
    # score_timeline: {event: [score at each offset]}
    score_tl  = {e: [] for e in all_evts}
    top1_tl   = []  # top-1 analogue at each offset

    for dn_t in offsets:
        sc = _decay_scores_at(dn_t)
        sc_map = {s["event"]: s["path_quant"] for s in sc}
        for e in all_evts:
            score_tl[e].append(sc_map.get(e, 0.0))
        top1_tl.append(sc[0]["event"] if sc else "")

    # Only plot events that reached top-3 at any point
    top3_ever = set()
    for dn_t_idx in range(len(offsets)):
        sorted_at = sorted(all_evts, key=lambda e: score_tl[e][dn_t_idx], reverse=True)
        top3_ever.update(sorted_at[:3])

    fig = go.Figure()
    for j, en in enumerate(sorted(top3_ever)):
        fig.add_trace(go.Scatter(
            x=offsets, y=score_tl[en],
            name=en, mode="lines+markers",
            line=dict(color=EVENT_COLORS[j % len(EVENT_COLORS)], width=2),
            marker=dict(size=5),
            hovertemplate=f"<b>{en}</b><br>Day+%{{x}}: %{{y:.3f}}<extra></extra>",
        ))

    # Shade regions by top-1 analogue
    current_top = top1_tl[0] if top1_tl else ""
    span_start  = offsets[0]
    evt_idx_map = {e: i for i,(_,) in [(i,(e,)) for i,(e,_) in enumerate(EVENTS)]}
    for idx, dn_t in enumerate(offsets[1:], 1):
        if top1_tl[idx] != current_top or idx == len(offsets)-1:
            fig.add_vrect(
                x0=span_start, x1=dn_t,
                fillcolor=EVENT_COLORS[list(e for e,_ in EVENTS).index(current_top) % len(EVENT_COLORS)]
                          if current_top in [e for e,_ in EVENTS] else "#333",
                opacity=0.08, layer="below", line_width=0,
                annotation_text=current_top[:12] if current_top else "",
                annotation_font=dict(color=COL_MUTED, size=8),
                annotation_position="top left")
            current_top = top1_tl[idx]
            span_start  = dn_t

    fig.add_vline(x=real_dn, line=dict(color="#ffa657",width=2,dash="dot"),
                  annotation_text=f"Day+{real_dn}",
                  annotation_font=dict(color="#ffa657",size=10))
    fig.update_layout(**dark_layout(
        title=f"<b>Signal Decay Tracker</b> — analogue rank evolution Day 0 → Day+{real_dn}",
        xaxis=dict(title="Trading days from Day 0",
                   tickvals=offsets[::max(1,len(offsets)//15)],
                   gridcolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        yaxis=dict(title="Path Similarity Score", range=[0,1.05],
                   gridcolor=COL_GRID, tickfont=dict(color=COL_MUTED)),
        legend=dict(x=1.01,y=1,bgcolor=BG_PANEL,bordercolor=COL_GRID,
                    font=dict(color=COL_TEXT,size=9)),
        width=CHART_W, height=480, margin=dict(r=200)))
    return fig


---
## § 6 · DASHBOARD
> Tabbed ipywidgets interface. Run §6.0 first, then §6.1–§6.10 in order, or run all.  
> Each cell is one tab group — edit the relevant cell to change a specific tab's layout or behaviour.


In [ ]:
# §6.0 — DASHBOARD SHARED HELPERS
# ─────────────────────────────────────────────────────────────
# Widget factory functions, output helpers, CLASS_TO_ASSETS lookup,
# and a safe LIVE stub (overwritten when §5.1 runs).
# Must run before any other §6 cell.

# ── Safe LIVE default (overwritten by §5.1 pull_live_data()) ──
if 'LIVE' not in dir():
    LIVE = {"name": None, "day0": None, "tags": set(), "trigger": 0.0,
            "cpi": "low", "fed": "hold", "trigger_pctile": None, "sim_assets": [],
            "prices": None, "returns": None, "day_n": 0, "bday_index": []}

# Assembles all chart functions (§4) and live engine (§5) into a
# tabbed ipywidgets interface. Tab structure:
#
#   Events          — event selection, date overrides, custom event add
#   Overlay         — §4.1 overlay lines per asset
#   Heatmap         — §4.2 return heatmap per asset
#   Scatter         — §4.3 regime scatter (asset X vs Y)
#   VIX             — §4.5 VIX absolute level + IQR band
#   Box & Whisker   — §4.6 return distribution boxes
#   Summary Table   — §4.7 mean±std table
#   Step-In         — §4.8 forward return from step-in day
#   Live — Config   — §5.1 live event setup + data pull
#   Live — Analogues— §5.2 analogue scoring + cutoff
#   Live — Paths    — §5.3 live path vs analogues
#   Live — Quick    — §5.4 trade ideas table
#   Live — Detail   — §5.5 per-asset drill-down
import ipywidgets as W
from IPython.display import display, clear_output

TAB_STYLE = """
<style>
.widget-tab > .p-TabBar .p-TabBar-tab { 
    background: #161b22; color: #8b949e;
    border-color: #30363d; min-width: 100px;
    font-family: monospace; font-size: 11px;
}
.widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current { 
    background: #0d1117; color: #e6edf3;
    border-bottom: 2px solid #1f6feb;
}
.jupyter-widgets { font-family: monospace; }
</style>
"""
display(W.HTML(TAB_STYLE))

group_opts   = ["— All Assets —"] + sorted(CUSTOM_GROUPS.keys()) + ALL_CLASSES
poi_lbl_list = [l for l, _ in POIS]
fwd_poi_opts = [l for l, o in POIS if o > 0]


# ── UI/runtime guardrails ────────────────────────────────────
_UI_BUSY = {"flag": False}

def ui_safe(fn):
    """Prevent recursive callback loops and surface callback errors cleanly."""
    def _wrapped(*args, **kwargs):
        if _UI_BUSY["flag"]:
            return None
        _UI_BUSY["flag"] = True
        try:
            return fn(*args, **kwargs)
        except Exception as _ui_e:
            print(f"⚠️  UI callback error in {fn.__name__}: {_ui_e}")
            return None
        finally:
            _UI_BUSY["flag"] = False
    return _wrapped

def safe_numeric_df(df):
    """JSON-safe numeric frame for display/plotly serialization."""
    if df is None:
        return None
    out = df.replace([np.inf, -np.inf], np.nan)
    for c in out.columns:
        if pd.api.types.is_numeric_dtype(out[c]):
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def make_cls_dd(desc="Class:"):
    return W.Dropdown(options=ALL_CLASSES, description=desc,
                      layout=W.Layout(width="200px"), style={"description_width":"50px"})

def make_asset_dd(desc="Asset:"):
    return W.Dropdown(description=desc,
                      layout=W.Layout(width="240px"), style={"description_width":"50px"})

def make_anchor():
    return W.ToggleButtons(options=[("Day-0","day0"),("Step-in","stepin")],
                           value="day0", description="",
                           layout=W.Layout(width="280px"))

def make_slider(desc="Step-in:", lo=-21, hi=63, val=0):
    return W.IntSlider(value=val, min=lo, max=hi, step=1, description=desc,
                       continuous_update=False,
                       layout=W.Layout(width="500px"),
                       style={"description_width":"80px"})

def make_group_dd(desc="Group:", val="Risk Barometer"):
    return W.Dropdown(options=group_opts, value=val, description=desc,
                      layout=W.Layout(width="300px"), style={"description_width":"60px"})

def update_assets(cls_dd, asset_dd):
    if _UI_BUSY["flag"]:
        return
    _UI_BUSY["flag"] = True
    try:
        opts = [l for l in CLASS_TO_ASSETS.get(cls_dd.value, []) if l in event_returns]
        asset_dd.options = opts
        if opts:
            if asset_dd.value not in opts:
                asset_dd.value = opts[0]
        else:
            asset_dd.value = None
    finally:
        _UI_BUSY["flag"] = False

def get_group_labels(grp):
    if grp == "— All Assets —": return get_subset("all")
    if grp in CUSTOM_GROUPS:    return get_subset("group", group=grp)
    if grp in ALL_CLASSES:      return get_subset("class", cls=grp)
    return []

def out_fig(fig, out):
    with out:
        clear_output(wait=True)
        if not fig:
            return
        try:
            fig.show()
        except Exception as _fig_e:
            print(f"⚠️  Figure render failed: {_fig_e}")

def make_progress(desc="Working…"):
    bar = W.IntProgress(value=0, min=0, max=10, description=desc,
                        bar_style="info",
                        layout=W.Layout(width="400px", visibility="hidden"))
    lbl = W.Label(value="", layout=W.Layout(visibility="hidden"))
    return W.HBox([bar, lbl]), bar, lbl

def loading_start(bar, lbl, msg="Running…"):
    bar.value = 3; bar.bar_style = "info"
    lbl.value = msg
    bar.layout.visibility = "visible"
    lbl.layout.visibility = "visible"

def loading_done(bar, lbl, msg="✅ Done"):
    bar.value = 10; bar.bar_style = "success"; lbl.value = msg

def loading_error(bar, lbl, msg="❌ Error"):
    bar.value = 10; bar.bar_style = "danger"; lbl.value = msg


def make_refresh_state():
    return {"busy": False, "last": None, "ready": False}


def run_guarded_refresh(state, key, render_fn, label="refresh"):
    if not state.get("ready") or state.get("busy"):
        return None
    if key == state.get("last"):
        return None
    state["busy"] = True
    try:
        result = render_fn()
        state["last"] = key
        return result
    except Exception as _ui_e:
        print(f"⚠️  {label} failed: {_ui_e}")
        return None
    finally:
        state["busy"] = False


In [ ]:
# §6.1 — EVENTS TAB
# ─────────────────────────────────────────────────────────────
# Event selector checkboxes, date override inputs, and custom event add form.
# _event_rows : list of per-event widget rows (rebuilt on _apply_changes)
# _pull_single_event : incremental data pull for custom events (mirrors §3.1 logic)

# ══════════════════════════════════════════════════════════════
# EVENTS TAB — selector, date override, add custom
# ══════════════════════════════════════════════════════════════
def _make_event_info_label(en):
    """Compact one-line summary of tags + macro context for an event."""
    tags = EVENT_TAGS.get(en, set())
    mc   = MACRO_CONTEXT.get(en, {})
    trig_map = globals().get("TRIGGER_PERCENTILE_MAP", {}) or {}
    tag_str = ", ".join(sorted(tags)) if tags else "—"
    cpi_str = mc.get("cpi", "—")
    fed_str = mc.get("fed", "—")
    oil_val = mc.get("trigger", mc.get("oil", None))
    oil_str = f"{TRIGGER_ASSET.split()[0]}: ${oil_val:.0f}" if oil_val else "—"
    z_val   = trig_map.get(en)
    z_str   = f"z={z_val:+.2f}" if z_val is not None else "z=—"
    return (f"  [{tag_str}]  CPI: {cpi_str}  Fed: {fed_str}  "
            f"{oil_str} ({z_str})")

_event_rows = []
for en, ed in EVENTS:
    chk  = W.Checkbox(value=True, description="", indent=False,
                      layout=W.Layout(width="30px"))
    name = W.Label(value=en, layout=W.Layout(width="220px"))
    date = W.Text(value=str(ed.date()) if hasattr(ed, "date") else str(ed),
                  description="", placeholder="YYYY-MM-DD",
                  layout=W.Layout(width="120px"))
    info = W.Label(
        value=_make_event_info_label(en),
        layout=W.Layout(width="520px"),
        style={"font_family": "monospace", "font_size": "10px",
               "text_color": "#8b949e"})
    _event_rows.append({"event": en, "chk": chk, "name": name,
                        "date": date, "info": info})

_ev_sl = {"description_width": "90px"}
add_name  = W.Text(description="Name:", placeholder="e.g. 2026 Taiwan Crisis",
                   layout=W.Layout(width="340px"), style={"description_width":"50px"})
add_date  = W.Text(description="Date:", placeholder="YYYY-MM-DD",
                   layout=W.Layout(width="200px"), style={"description_width":"45px"})
add_tags  = W.SelectMultiple(options=ALL_TAGS,
                              description="Tags:",
                              layout=W.Layout(width="300px", height="115px"),
                              style={"description_width":"45px"})
add_oil   = W.FloatText(value=0.0, description=f"{TRIGGER_ASSET} at D0:",  # auto-labels from TRIGGER_ASSET
                         layout=W.Layout(width="220px"), style=_ev_sl)
add_oil_lbl = W.Label(value=f"Enter date → {TRIGGER_ASSET} auto-fetched",
                       layout=W.Layout(width="280px"))
add_cpi   = W.Dropdown(
    options=[("High >4%","high"),("Mid 2-4%","mid"),("Low <2%","low")],
    value="mid", description="CPI Regime:",
    layout=W.Layout(width="220px"), style=_ev_sl)
add_fed   = W.Dropdown(options=["hiking","cutting","hold"], value="hold",
                        description="Fed Stance:",
                        layout=W.Layout(width="200px"), style=_ev_sl)
add_btn   = W.Button(description="＋ Add Event", button_style="success",
                     layout=W.Layout(width="130px"))
add_status = W.Label(value="")

def _fetch_trigger_for_date(date_str):
    """Auto-fetch TRIGGER_ASSET close on or just after the given date.
    Tries prices first (already in memory); falls back to a live yf.download.
    """
    tkr_trigger = ASSET_META.get(TRIGGER_ASSET, {}).get("ticker", "")
    try:
        ts = pd.Timestamp(date_str.strip())
    except Exception:
        return  # not a valid date yet — ignore

    # ── Try prices frame first (covers any in-range date) ─────
    if TRIGGER_ASSET in prices.columns:
        col = prices[TRIGGER_ASSET].dropna()
        fwd = col[col.index >= ts]
        if not fwd.empty:
            price = float(fwd.iloc[0])
            add_oil.value = round(price, 2)
            add_oil_lbl.value = (f"{TRIGGER_ASSET} ({tkr_trigger}): {price:.2f} "
                                 f"@ {fwd.index[0].date()}  (from cached data)")
            return

    # ── Fall back to live yf.download (date outside prices range) ──
    try:
        raw = yf.download(tkr_trigger, start=ts - pd.Timedelta(days=5),
                          end=ts + pd.Timedelta(days=10),
                          auto_adjust=True, progress=False)["Close"]
        if raw.empty: raise ValueError("no data")
        raw = raw.dropna()
        fwd = raw[raw.index >= ts]
        price = float(fwd.iloc[0]) if not fwd.empty else float(raw.iloc[-1])
        add_oil.value = round(price, 2)
        add_oil_lbl.value = (f"{TRIGGER_ASSET} ({tkr_trigger}): {price:.2f} "
                             f"@ {(fwd.index[0] if not fwd.empty else raw.index[-1]).date()}")
    except Exception as e:
        add_oil_lbl.value = f"{TRIGGER_ASSET} fetch failed ({e}) — enter manually"

def _on_add_date_change(change):
    d = change["new"].strip()
    if len(d) == 10:  # only fetch when full YYYY-MM-DD entered
        _fetch_trigger_for_date(d)

add_date.observe(_on_add_date_change, names="value")

event_rows_box = W.VBox(
    [W.HBox([W.Label("", layout=W.Layout(width="30px")),
             W.Label("Event", layout=W.Layout(width="220px")),
             W.Label("Date", layout=W.Layout(width="120px")),
             W.Label("Tags · CPI · Fed · Trigger",
                     layout=W.Layout(width="520px"),
                     style={"font_size": "10px", "text_color": "#8b949e"})])]
    + [W.HBox([r["chk"], r["name"], r["date"], r["info"]])
       for r in _event_rows]
)

def _pull_single_event(en, ed):
    """Pull data for a new event, mirroring §3.1 construction exactly.
    Prints step-by-step progress so hangs are immediately visible.
    Events ending in † are MANUAL_DATA reconstructions — no pull needed.
    """
    if en.endswith("†"):
        print(
            f"ℹ️  '{en}' is a manually-reconstructed event (†).\n"
            "   Data is supplied via MANUAL_DATA in §1.3 — no yfinance pull required.\n"
            "   To update or correct values, edit the MANUAL_DATA dict directly."
        )
        return True, "manual data (†) — no pull"
    fetch_from = ed - timedelta(days=PRE_WINDOW_TD*2 + FETCH_BUFFER)
    fetch_to   = ed + timedelta(days=POST_WINDOW_TD*2 + FETCH_BUFFER)
    needs_pull = (prices.index.min() > fetch_from) or (prices.index.max() < fetch_to)
    print(f"  [{en}] Date range: {fetch_from.date()} → {fetch_to.date()}")
    print(f"  [{en}] prices cache: {prices.index.min().date()} → {prices.index.max().date()}")
    print(f"  [{en}] Needs pull: {needs_pull}")
    if needs_pull:
        # Determine gap boundaries (only pull what is not already in prices)
        need_left  = prices.index.min() > fetch_from
        need_right = prices.index.max() < fetch_to
        if need_left and need_right:
            pull_from, pull_to = fetch_from, fetch_to
        elif need_left:
            pull_from, pull_to = fetch_from, prices.index.min()
        else:
            pull_from, pull_to = prices.index.max(), fetch_to

        yf_labels  = [l for l in ASSET_ORDER if ASSET_META[l]["source"] == "yf"]
        yf_tickers = list(dict.fromkeys([ASSET_META[l]["ticker"] for l in yf_labels]))
        if VIX_TICKER not in yf_tickers: yf_tickers.append(VIX_TICKER)

        # ── yfinance pull ─────────────────────────────────────
        print(f"  [{en}] yfinance: pulling {len(yf_tickers)} tickers "
              f"{pull_from.date()} → {pull_to.date()} …")
        try:
            raw_yf = yf.download(yf_tickers, start=pull_from, end=pull_to,
                                 auto_adjust=True, progress=False)["Close"]
            if isinstance(raw_yf, pd.Series): raw_yf = raw_yf.to_frame(yf_tickers[0])
            raw_yf.index = pd.to_datetime(raw_yf.index)
            print(f"  [{en}] yfinance: {raw_yf.shape[1]} columns, {raw_yf.shape[0]} rows ✅")
        except Exception as e:
            print(f"  [{en}] yfinance: ❌ {e}")
            return False, f"yfinance error: {e}"

        # ── FRED pull ─────────────────────────────────────────
        fred_labels  = [l for l in ASSET_ORDER if ASSET_META[l]["source"] == "fred"]
        fred_tickers = list(dict.fromkeys([ASSET_META[l]["ticker"] for l in fred_labels]))
        raw_fred = pd.DataFrame()
        print(f"  [{en}] FRED: pulling {len(fred_tickers)} series …")
        _fred_ok, _fred_fail = [], []
        for tkr in fred_tickers:
            try:
                raw_fred[tkr] = _fred_fetch(tkr, pull_from, pull_to)
                _fred_ok.append(tkr)
            except Exception as _fe:
                _fred_fail.append(tkr)
        print(f"  [{en}] FRED: {len(_fred_ok)}/{len(fred_tickers)} pulled"
              + (f" | ⚠️ failed: {_fred_fail}" if _fred_fail else " ✅"))
        if not raw_fred.empty:
            raw_fred.index = pd.to_datetime(raw_fred.index)

        # ── Build new slice (mirrors Cell 5 prices construction) ─
        new_idx = raw_yf.index
        if not raw_fred.empty:
            new_idx = new_idx.union(raw_fred.index)
        # Only keep dates not already in prices
        new_idx = new_idx.difference(prices.index)
        if len(new_idx):
            new_slice = pd.DataFrame(index=new_idx, columns=prices.columns, dtype=float)
            for label in ASSET_ORDER:
                meta = ASSET_META[label]; tkr = meta["ticker"]
                if meta["source"] == "yf":
                    if tkr in raw_yf.columns:
                        new_slice[label] = raw_yf[tkr].reindex(new_idx)
                else:
                    if not raw_fred.empty and tkr in raw_fred.columns:
                        new_slice[label] = raw_fred[tkr].reindex(new_idx)
            if VIX_TICKER in raw_yf.columns:
                new_slice[VIX_LABEL] = raw_yf[VIX_TICKER].reindex(new_idx)
            # Append and re-ffill the whole frame (same as Cell 5)
            combined = pd.concat([prices, new_slice]).sort_index()
            combined.ffill(inplace=True)
            # Mutate prices in-place so all existing references stay valid
            prices.drop(prices.index, inplace=True)
            for col in combined.columns:
                prices[col] = combined[col].reindex(combined.index).values
            prices.index = combined.index
    # Compute event_returns from existing prices — no pull needed for in-range events
    for l in ALL_LABELS:
        s = compute_event_series(l, en, ed)
        if s is not None:
            if l not in event_returns: event_returns[l] = {}
            event_returns[l][en] = s
    n_ok = sum(1 for l in ALL_LABELS if en in event_returns.get(l, {}))
    print(f"  [{en}] event_returns: {n_ok}/{len(ALL_LABELS)} assets mapped ✅")
    return True, f"{n_ok}/{len(ALL_LABELS)} assets"

apply_btn    = W.Button(description="✔  Apply Changes", button_style="primary",
                        layout=W.Layout(width="160px"))
apply_status = W.Label(value="Edit then click Apply — re-run any tab to see changes")
ev_prog_box, ev_bar, ev_bar_lbl = make_progress("Pulling:")
ev_apply_state = make_refresh_state()
ev_add_state = make_refresh_state()
ev_apply_state["ready"] = True
ev_add_state["ready"] = True

def _apply_changes(_=None):
    if ev_apply_state["busy"]:
        return
    ev_apply_state["busy"] = True
    apply_btn.disabled = True
    current = {en: ed for en, ed in EVENTS}
    desired = {}
    for r in _event_rows:
        if not r["chk"].value: continue
        try:
            desired[r["event"]] = pd.Timestamp(r["date"].value.strip())
        except Exception:
            pass
    removed = [en for en in current if en not in desired]
    for en in removed:
        EVENTS[:] = [(n, d) for n, d in EVENTS if n != en]
        for l in event_returns:
            event_returns[l].pop(en, None)
    to_pull = {en: ed for en, ed in desired.items()
               if en not in current or current[en] != ed}
    errors = []
    for i, (en, ed) in enumerate(to_pull.items()):
        loading_start(ev_bar, ev_bar_lbl, f"Processing {i+1}/{len(to_pull)}: {en}…")
        ok, msg = _pull_single_event(en, ed)
        if ok:
            EVENTS[:] = [(n, d) for n, d in EVENTS if n != en]
            EVENTS.append((en, ed))
        else:
            errors.append(f"{en}: {msg}")
    # Re-sort EVENTS chronologically so new events slot in by date
    EVENTS.sort(key=lambda x: pd.Timestamp(x[1]))
    n_active = len(desired)
    if errors:
        loading_error(ev_bar, ev_bar_lbl, f"⚠️ errors: {'; '.join(errors)}")
    else:
        loading_done(ev_bar, ev_bar_lbl, f"✅ {n_active} events active — refreshing all tabs…")
    # Refresh all historical tabs so new/removed events show immediately
    try:
        t1_refresh(); t2_refresh(); t3_refresh()
        out_fig(build_vix_fig(), t5_out)
        t6_refresh(); t7_refresh(); t8_refresh()
        loading_done(ev_bar, ev_bar_lbl, f"✅ {n_active} events active — all tabs updated")
    except Exception as e:
        loading_error(ev_bar, ev_bar_lbl, f"⚠️ Applied but refresh failed: {e}")
    apply_btn.disabled = False
    ev_apply_state["busy"] = False

apply_btn.on_click(_apply_changes)

def _add_event(_=None):
    if ev_add_state["busy"]:
        return
    ev_add_state["busy"] = True
    try:
        n = add_name.value.strip(); d = add_date.value.strip()
        if not n or not d:
            add_status.value = "❌ Name and date required"; return
        try:
            ts = pd.Timestamp(d)
        except Exception:
            add_status.value = "❌ Invalid date — use YYYY-MM-DD"; return
        if any(r["event"] == n for r in _event_rows):
            add_status.value = f"❌ '{n}' already exists"; return
        # Register tags and macro context for analogue matching
        EVENT_TAGS[n]    = set(add_tags.value)
        MACRO_CONTEXT[n] = {"oil": add_oil.value, "cpi": add_cpi.value, "fed": add_fed.value}
        chk  = W.Checkbox(value=True, description="", indent=False,
                          layout=W.Layout(width="30px"))
        name = W.Label(value=f"★ {n}", layout=W.Layout(width="260px"))
        date = W.Text(value=str(ts.date()), description="", placeholder="YYYY-MM-DD",
                      layout=W.Layout(width="130px"))
        info = W.Label(
            value=_make_event_info_label(n),
            layout=W.Layout(width="520px"),
            style={"font_family": "monospace", "font_size": "10px",
                   "text_color": "#8b949e"})
        _event_rows.append({"event": n, "chk": chk, "name": name,
                            "date": date, "info": info})
        event_rows_box.children = list(event_rows_box.children) + [W.HBox([chk, name, date, info])]
        add_name.value = ""; add_date.value = ""
        add_tags.value = []
        add_status.value = f"✅ Added '{n}' — click Apply to pull data"
    finally:
        ev_add_state["busy"] = False

add_btn.on_click(_add_event)

tab_events = W.VBox([
    W.HTML("<h3 style='color:#e6edf3;font-family:monospace;margin:6px 0'>📅  Event Selection & Date Overrides</h3>"),
    W.Label("Uncheck to exclude. Edit dates to override Day-0. Click Apply when done."),
    W.HTML("<hr style='border-color:#30363d;margin:6px 0'>"),
    event_rows_box,
    W.HTML("<hr style='border-color:#30363d;margin:8px 0'>"),
    W.Label("➕ Add Custom Event  (Tags & macro context used in analogue matching)"),
    W.HBox([add_name, add_date]),
    W.HBox([add_tags, W.VBox([W.HBox([add_oil, add_oil_lbl]), add_cpi, add_fed])]),
    W.HBox([add_btn, add_status]),
    W.HTML("<hr style='border-color:#30363d;margin:8px 0'>"),
    W.HBox([apply_btn, apply_status]),
    ev_prog_box,
])


In [ ]:
# §6.2 — OVERLAY TAB
# ─────────────────────────────────────────────────────────────
# Asset/class dropdowns + anchor toggle + step-in slider.
# t1_refresh: rebuilds overlay fig, overlays live path if LIVE data loaded.

# ══════════════════════════════════════════════════════════════
# TAB 1 — OVERLAY
# ══════════════════════════════════════════════════════════════
t1_cls    = make_cls_dd()
t1_asset  = make_asset_dd()
t1_prev   = W.Button(description="◀", layout=W.Layout(width="40px"))
t1_next   = W.Button(description="▶", layout=W.Layout(width="40px"))
t1_anchor = make_anchor()
t1_slider = make_slider()
t1_srow   = W.HBox([t1_slider])
t1_srow.layout.visibility = "hidden"
t1_out    = W.Output()
t1_state  = make_refresh_state()

def t1_refresh(*_):
    asset = t1_asset.value
    if not asset:
        return
    step = t1_slider.value if t1_anchor.value == "stepin" else 0
    key = (t1_cls.value, asset, t1_anchor.value, int(step), LIVE.get("name"), LIVE.get("day_n"))

    def _render():
        fig  = build_overlay_fig(asset, t1_anchor.value, step)
        if fig and LIVE.get("returns") and asset:
            lr   = LIVE["returns"].get(asset)
            dn   = LIVE.get("day_n")
            unit = unit_label(asset)
            if lr is not None and dn is not None:
                lr = lr[lr.index <= dn]
                if t1_anchor.value == "stepin" and step != 0:
                    ai = lr.index[np.abs(np.array(lr.index) - step) <= 2]
                    anchor_val = float(lr.loc[ai[0]]) if len(ai) else 0.0
                    lr = lr - anchor_val
                fig.add_trace(go.Scatter(
                    x=list(lr.index), y=list(lr.values),
                    name=f"▶ {LIVE['name']}", mode="lines+markers",
                    line=dict(color="#ffa657", width=3),
                    marker=dict(size=5, color="#ffa657"),
                    hovertemplate=f"<b>{LIVE['name']}</b><br>Day %{{x}}: %{{y:.2f}} {unit}<extra></extra>",
                ))
            lname = LIVE.get("name", "Live")
            _is_ov = "wl_day_override_chk" in dir() and wl_day_override_chk.value
            _vline_lbl = f"Day+{dn} override ({lname})" if _is_ov else f"Today D+{dn} ({lname})"
            if dn is not None:
                fig.add_vline(x=dn,
                    line=dict(color="#ffa657" if not _is_ov else "#d29922",
                              width=2, dash="dot"),
                    annotation_text=_vline_lbl,
                    annotation_font=dict(color="#ffa657" if not _is_ov else "#d29922", size=10))
        out_fig(fig, t1_out)

    run_guarded_refresh(t1_state, key, _render, label="Dashboard overlay")


def t1_anchor_change(c):
    if c.get("name") != "value":
        return
    t1_srow.layout.visibility = "visible" if c["new"] == "stepin" else "hidden"
    t1_refresh()


def t1_prev_click(_):
    opts = list(t1_asset.options)
    if opts:
        t1_asset.value = opts[(opts.index(t1_asset.value)-1) % len(opts)]


def t1_next_click(_):
    opts = list(t1_asset.options)
    if opts:
        t1_asset.value = opts[(opts.index(t1_asset.value)+1) % len(opts)]


t1_cls.observe(lambda c: (update_assets(t1_cls, t1_asset), t1_refresh()) if c.get("name") == "value" else None, "value")
t1_asset.observe(lambda c: t1_refresh() if c.get("name") == "value" else None, "value")
t1_anchor.observe(t1_anchor_change, "value")
t1_slider.observe(lambda c: t1_refresh() if c.get("name") == "value" else None, "value")
t1_prev.on_click(t1_prev_click)
t1_next.on_click(t1_next_click)
update_assets(t1_cls, t1_asset)
t1_state["ready"] = True
t1_refresh()

tab1 = W.VBox([W.HBox([t1_cls, t1_asset, t1_prev, t1_next, t1_anchor]), t1_srow, t1_out])

# ══════════════════════════════════════════════════════════════


In [ ]:
# §6.3 — HEATMAP TAB
# ─────────────────────────────────────────────────────────────
# Group/class selector. t2_refresh appends live event row if available.

# TAB 2 — HEATMAP
# ══════════════════════════════════════════════════════════════
t2_cls    = make_cls_dd()
t2_asset  = make_asset_dd()
t2_anchor = make_anchor()
t2_slider = make_slider()
t2_srow   = W.HBox([t2_slider])
t2_srow.layout.visibility = "hidden"
t2_out    = W.Output()
t2_state  = make_refresh_state()

def t2_refresh(*_):
    asset = t2_asset.value
    if not asset:
        return
    step = t2_slider.value if t2_anchor.value == "stepin" else 0
    key = (t2_cls.value, asset, t2_anchor.value, int(step), LIVE.get("name"), LIVE.get("day_n"))

    def _render():
        fig  = build_heatmap_fig(asset, t2_anchor.value, step)
        if fig and LIVE.get("returns") and asset:
            lr    = LIVE["returns"].get(asset)
            dn    = LIVE.get("day_n", 0)
            _is_ov = "wl_day_override_chk" in dir() and wl_day_override_chk.value
            lname = (f"{LIVE.get('name', 'Live')} [D+{dn} OV]"
                     if _is_ov else LIVE.get("name", "Live"))
            if lr is not None:
                live_row = []
                live_txt = []
                for pl, po in POIS:
                    idx_arr = lr.index[np.abs(np.array(lr.index) - po) <= 3]
                    if len(idx_arr) and po <= dn:
                        val = float(lr.loc[idx_arr[-1]])
                        if step > 0 and t2_anchor.value == "stepin":
                            anch = lr.index[np.abs(np.array(lr.index) - step) <= 3]
                            val = val - float(lr.loc[anch[-1]]) if len(anch) else val
                        live_row.append(val)
                        live_txt.append(f"{val:+.1f}")
                    else:
                        live_row.append(None)
                        live_txt.append("—")
                hm   = fig.data[0]
                fig.data[0].z = list(hm.z) + [live_row]
                if getattr(hm, "text", None) is not None:
                    fig.data[0].text = list(hm.text) + [live_txt]
                fig.data[0].y = list(hm.y) + [f"▶ {lname} (D+{dn})"]
                fig.update_layout(height=fig.layout.height + 42)
        out_fig(fig, t2_out)

    run_guarded_refresh(t2_state, key, _render, label="Dashboard heatmap")


def t2_anchor_change(c):
    if c.get("name") != "value":
        return
    t2_srow.layout.visibility = "visible" if c["new"] == "stepin" else "hidden"
    t2_refresh()


t2_cls.observe(lambda c: (update_assets(t2_cls, t2_asset), t2_refresh()) if c.get("name") == "value" else None, "value")
t2_asset.observe(lambda c: t2_refresh() if c.get("name") == "value" else None, "value")
t2_anchor.observe(t2_anchor_change, "value")
t2_slider.observe(lambda c: t2_refresh() if c.get("name") == "value" else None, "value")
update_assets(t2_cls, t2_asset)
t2_state["ready"] = True
t2_refresh()

tab2 = W.VBox([W.HBox([t2_cls, t2_asset, t2_anchor]), t2_srow, t2_out])

# ══════════════════════════════════════════════════════════════


In [ ]:
# §6.4 — SCATTER TAB
# ─────────────────────────────────────────────────────────────
# Asset X / asset Y selectors + POI dropdown.

# TAB 3 — SCATTER
# ══════════════════════════════════════════════════════════════
t3_xcls   = make_cls_dd("X Class:")
t3_xasset = make_asset_dd("X:")
t3_ycls   = make_cls_dd("Y Class:")
t3_yasset = make_asset_dd("Y:")
t3_poi    = W.Dropdown(
    options=[(f"{l} (D{o:+d})", o) for l, o in POIS], value=0,
    description="Horizon:", layout=W.Layout(width="200px"),
    style={"description_width":"65px"})
t3_step   = W.ToggleButton(value=False, description="Step-forward",
                            layout=W.Layout(width="130px"))
t3_slider = make_slider("Step-in:", hi=58)
t3_srow   = W.HBox([t3_slider])
t3_srow.layout.visibility = "hidden"
t3_out    = W.Output()

# Guard against callback storms / re-entrant renders during widget init.
t3_state = {"busy": False, "last": None, "ready": False}


def t3_refresh(*_):
    if not t3_state["ready"] or t3_state["busy"]:
        return
    if not t3_xasset.value or not t3_yasset.value:
        return
    step = t3_slider.value if t3_step.value else None
    key = (t3_xcls.value, t3_xasset.value, t3_ycls.value, t3_yasset.value, t3_poi.value, step)
    if key == t3_state["last"]:
        return
    t3_state["busy"] = True
    try:
        fig = build_scatter_fig(t3_xasset.value, t3_yasset.value, t3_poi.value, step)
        out_fig(fig, t3_out)
        t3_state["last"] = key
    finally:
        t3_state["busy"] = False


def t3_on_xcls(change):
    if change.get("name") != "value":
        return
    update_assets(t3_xcls, t3_xasset)
    t3_refresh()


def t3_on_ycls(change):
    if change.get("name") != "value":
        return
    update_assets(t3_ycls, t3_yasset)
    t3_refresh()


def t3_on_asset(change):
    if change.get("name") == "value":
        t3_refresh()


def t3_on_poi(change):
    if change.get("name") == "value":
        t3_refresh()


def t3_on_slider(change):
    if change.get("name") == "value":
        t3_refresh()


def t3_on_step(change):
    if change.get("name") != "value":
        return
    t3_srow.layout.visibility = "visible" if change["new"] else "hidden"
    t3_refresh()


t3_xcls.observe(t3_on_xcls, "value")
t3_ycls.observe(t3_on_ycls, "value")
t3_xasset.observe(t3_on_asset, "value")
t3_yasset.observe(t3_on_asset, "value")
t3_poi.observe(t3_on_poi, "value")
t3_slider.observe(t3_on_slider, "value")
t3_step.observe(t3_on_step, "value")

t3_state["ready"] = False
t3_xcls.value = "Oil & Energy"; update_assets(t3_xcls, t3_xasset)
if "WTI Crude (spot)" in t3_xasset.options: t3_xasset.value = "WTI Crude (spot)"
t3_ycls.value = "Equities";     update_assets(t3_ycls, t3_yasset)
if "S&P 500" in t3_yasset.options: t3_yasset.value = "S&P 500"
t3_state["ready"] = True
t3_refresh()

tab3 = W.VBox([
    W.HBox([t3_xcls, t3_xasset, t3_ycls, t3_yasset]),
    W.HBox([t3_poi, t3_step]), t3_srow, t3_out
])

# ══════════════════════════════════════════════════════════════


In [ ]:
# §6.5 — VIX TAB
# ─────────────────────────────────────────────────────────────
# No controls — shows all selected events. Today vline if LIVE loaded.

# TAB 5 — VIX
# ══════════════════════════════════════════════════════════════
t5_out = W.Output()
out_fig(build_vix_fig(), t5_out)
tab5 = W.VBox([t5_out])

# ══════════════════════════════════════════════════════════════

In [ ]:
# §6.6 — BOX & WHISKER TAB
# ─────────────────────────────────────────────────────────────
# Group selector + POI multi-select.

# TAB 6 — BOX & WHISKER
# ══════════════════════════════════════════════════════════════
t6_group = make_group_dd("Group:", "Risk Barometer")
t6_pois  = W.SelectMultiple(
    options=poi_lbl_list,
    value=[l for l in poi_lbl_list if l.startswith("t+")],
    description="POIs:", layout=W.Layout(width="180px", height="160px"),
    style={"description_width":"40px"})
t6_band  = W.Checkbox(value=False, description="Group mean±1σ",
                       layout=W.Layout(width="180px"))
t6_out   = W.Output()
t6_state = make_refresh_state()

def t6_refresh(*_):
    lbls = get_group_labels(t6_group.value)
    pois = list(t6_pois.value)
    key = (t6_group.value, tuple(pois), bool(t6_band.value))
    run_guarded_refresh(
        t6_state,
        key,
        lambda: out_fig(build_box_fig(lbls, pois, t6_band.value) if lbls and pois else None, t6_out),
        label="Dashboard box",
    )


t6_group.observe(lambda c: t6_refresh() if c.get("name") == "value" else None, "value")
t6_pois.observe(lambda c: t6_refresh() if c.get("name") == "value" else None, "value")
t6_band.observe(lambda c: t6_refresh() if c.get("name") == "value" else None, "value")
t6_state["ready"] = True
t6_refresh()
tab6 = W.VBox([W.HBox([t6_group, t6_pois, t6_band]), t6_out])

# ══════════════════════════════════════════════════════════════


In [ ]:
# §6.7 — SUMMARY TABLE TAB
# ─────────────────────────────────────────────────────────────
# Group selector. Mean ± std across selected events at each POI.

# TAB 7 — SUMMARY TABLE
# ══════════════════════════════════════════════════════════════
t7_group = make_group_dd("Group:", "— All Assets —")
t7_out   = W.Output()
t7_state = make_refresh_state()

def t7_refresh(*_):
    lbls = get_group_labels(t7_group.value)
    key = (t7_group.value, len(lbls))
    run_guarded_refresh(
        t7_state,
        key,
        lambda: render_summary_output(lbls, t7_out),
        label="Dashboard summary",
    )


t7_group.observe(lambda c: t7_refresh() if c.get("name") == "value" else None, "value")
t7_state["ready"] = True
t7_refresh()
tab7 = W.VBox([t7_group, t7_out])

# ══════════════════════════════════════════════════════════════


In [ ]:
# §6.8 — STEP-IN ANALYSIS TAB
# ─────────────────────────────────────────────────────────────
# Step-in day slider + group selector. Forward return distributions.

# TAB 8 — STEP-IN ANALYSIS
# ══════════════════════════════════════════════════════════════
t8_group  = make_group_dd("Group:", "Risk Barometer")
t8_slider = W.IntSlider(value=STEP_IN_PRIMARY, min=-PRE_WINDOW_TD,
                         max=POST_WINDOW_TD-5, step=1,
                         description="Step-In Day:",
                         continuous_update=False,
                         layout=W.Layout(width="560px"),
                         style={"description_width":"100px"})
t8_pois   = W.SelectMultiple(options=fwd_poi_opts, value=fwd_poi_opts,
                               description="Fwd POIs:",
                               layout=W.Layout(width="180px", height="140px"),
                               style={"description_width":"65px"})
t8_out    = W.Output()
t8_state  = make_refresh_state()

def t8_refresh(*_):
    lbls = get_group_labels(t8_group.value)
    pois = list(t8_pois.value)
    key = (t8_group.value, int(t8_slider.value), tuple(pois))

    def _render():
        if not lbls or not pois:
            with t8_out:
                clear_output(wait=True)
            return
        fb, fr, fh = build_stepin_figs(t8_slider.value, lbls, pois)
        with t8_out:
            clear_output(wait=True)
            if fb: fb.show()
            if fr: fr.show()
            if fh: fh.show()

    run_guarded_refresh(t8_state, key, _render, label="Dashboard step-in")


t8_group.observe(lambda c: t8_refresh() if c.get("name") == "value" else None, "value")
t8_slider.observe(lambda c: t8_refresh() if c.get("name") == "value" else None, "value")
t8_pois.observe(lambda c: t8_refresh() if c.get("name") == "value" else None, "value")
t8_state["ready"] = True
t8_refresh()

tab8 = W.VBox([
    W.HBox([t8_group, t8_pois]),
    W.Label("Drag slider — all 3 charts update:"),
    t8_slider, t8_out
])

# ══════════════════════════════════════════════════════════════


In [ ]:
# §6.9 — LIVE TABS (L1–L5)
# ─────────────────────────────────────────────────────────────
# L1: event name/date/tags/trigger-price/cpi/fed inputs + pull button
# L2: analogue scoring + cutoff slider
# L3: live path vs analogues overlay + forward heatmap
# L4: trade ideas quick table
# L5: per-asset detailed analysis

# TAB L1 — LIVE CONFIG
# Safe inline defaults so this cell still works in pruned/headless execution.
_default_live_name = globals().get("_default_live_name", LIVE.get("name") or "Live Event")
_default_live_date = globals().get("_default_live_date", str(pd.Timestamp.today().date()))
_default_live_tags = tuple(t for t in globals().get("_default_live_tags", tuple()) if t in ALL_TAGS)
if not _default_live_tags:
    _default_live_tags = tuple(ALL_TAGS[:2]) if len(ALL_TAGS) >= 2 else tuple(ALL_TAGS)
# ══════════════════════════════════════════════════════════════
sl = {"description_width":"120px"}
wl_name = W.Text(value=_default_live_name, description="Event Name:",
    layout=W.Layout(width="340px"), style=sl)
wl_date = W.Text(value=_default_live_date, description="Day 0 Date:",
    layout=W.Layout(width="240px"), style=sl)
wl_tags = W.SelectMultiple(options=ALL_TAGS,
    value=_default_live_tags, description="Event Tags:",
    layout=W.Layout(width="320px", height="130px"), style=sl)
wl_oil_lbl = W.Label(value=f"{TRIGGER_ASSET} at Day 0:  fetching…",
    layout=W.Layout(width="300px"))
wl_oil = type("_OilObj", (), {"value": 70.0})()  # stub with .value for LIVE dict compat

def _fetch_trigger_price():
    """Auto-fetch current TRIGGER_ASSET price for the Live Config widget."""
    tkr = ASSET_META.get(TRIGGER_ASSET, {}).get("ticker", "")
    if not tkr:
        wl_oil_lbl.value = f"{TRIGGER_ASSET}: ticker not found in ASSET_META"
        return
    try:
        raw = yf.download(tkr, period="5d", auto_adjust=True, progress=False)["Close"]
        if raw.empty: raise ValueError("empty")
        price = float(raw.dropna().iloc[-1])
        wl_oil.value = round(price, 2)
        wl_oil_lbl.value = f"{TRIGGER_ASSET} ({tkr}):  {price:.2f}  ({raw.dropna().index[-1].date()})"
    except Exception as e:
        wl_oil_lbl.value = f"{TRIGGER_ASSET} fetch failed: {e} — enter manually"
        wl_oil.value = 0.0

wti_refresh_btn = W.Button(description=f"↻ {TRIGGER_ASSET.split()[0]}",  # first word of TRIGGER_ASSET
    layout=W.Layout(width="80px"))
wti_refresh_btn.on_click(lambda _: _fetch_trigger_price())
_fetch_trigger_price()  # auto-fetch on load
wl_cpi  = W.Dropdown(
    options=[("High >4%","high"),("Mid 2-4%","mid"),("Low <2%","low")],
    value="mid", description="CPI Regime:",
    layout=W.Layout(width="220px"), style=sl)
wl_fed  = W.Dropdown(options=["hiking","cutting","hold"], value="hold",
    description="Fed Stance:", layout=W.Layout(width="200px"), style=sl)
wl_sim_assets = W.SelectMultiple(options=[(display_label(a), a) for a in SIMILARITY_ASSET_POOL],
    value=tuple(SIMILARITY_ASSET_POOL[:5]),  # first 5 from SIMILARITY_ASSET_POOL (TRIGGER_ASSET is slot 0)
    description="Sim Assets:",
    layout=W.Layout(width="340px", height="200px"), style=sl)

# Cutoff lives in L2 (wl2_cut). Min Coverage lives in L4 (wl4_min_cov).

# ── Day-N override ──────────────────────────────────────────
wl_day_override_chk = W.Checkbox(value=False,
    description="Override current day:",
    style={"description_width": "145px"},
    layout=W.Layout(width="200px"))
wl_day_override_n = W.BoundedIntText(value=0, min=0, max=500, step=1,
    description="Day+",
    layout=W.Layout(width="130px"), style={"description_width": "38px"})
wl_day_override_lbl = W.Label(
    value="(unchecked = use today)",
    layout=W.Layout(width="170px"))

# Stored real day_n from last pull — restored when override unchecked
_live_real_day_n = None

def _apply_override_and_refresh():
    """Apply day_n override to LIVE and re-render all affected charts.
    Does NOT re-pull price data — operates on what is already in LIVE.
    """
    global _live_real_day_n
    if LIVE.get("returns") is None:
        return  # nothing pulled yet

    if wl_day_override_chk.value:
        # Store real day_n once on first activation
        if _live_real_day_n is None:
            _live_real_day_n = LIVE.get("day_n")
        _max_off = max(
            (max(s.index) for s in LIVE["returns"].values() if len(s)),
            default=_live_real_day_n or 0)
        _ov = max(0, min(int(wl_day_override_n.value), _max_off))
        LIVE["day_n"] = _ov
        wl_day_override_lbl.value = f"trading days from Day 0  (real = Day+{_live_real_day_n})"
        # Update status label to show override is active
        _base = wl_status.value.split("  |  Day+")[0] if "  |  Day+" in wl_status.value else wl_status.value
        wl_status_lbl.value = f"{_base}  |  Day+{_ov} [OVERRIDE — real Day+{_live_real_day_n}]"
    else:
        if _live_real_day_n is not None:
            LIVE["day_n"] = _live_real_day_n
        wl_day_override_lbl.value = "(unchecked = use today)"
        # Restore real status
        wl_status_lbl.value = wl_status.value

    # Re-render every chart/tab that reads LIVE["day_n"]
    t1_refresh()
    t2_refresh()
    out_fig(build_vix_fig(), t5_out)
    if _scores_cache:
        wl2_run_btn_click()
    if LIVE.get("returns") and wl3_ast.value:
        wl3_refresh()
    if _shared_rows:
        wl4_refresh()
    if _shared_rows and wl5_ast.value and wl5_ast.value not in ("(no data)", ""):
        wl5_update_detail()

def _on_override_toggle(change):
    """Enable/disable int input, then apply override immediately."""
    wl_day_override_n.disabled = not change["new"]
    _apply_override_and_refresh()

def _on_override_day_change(change):
    """Re-apply whenever the day number changes (only if checkbox is on)."""
    if wl_day_override_chk.value:
        _apply_override_and_refresh()

wl_day_override_chk.observe(_on_override_toggle, "value")
wl_day_override_n.observe(_on_override_day_change, "value")
wl_day_override_n.disabled = True  # start disabled until checkbox ticked

wl_refresh_btn = W.Button(description="⟳  Refresh Live Data",
    button_style="primary", layout=W.Layout(width="200px", height="36px"))
wl_status_lbl  = W.Label(value="⬤  Not loaded")
wl_setup_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Setup checks: waiting</span>"
)
wl_fp_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Live fingerprint: waiting</span>"
)
wl_out_l1      = W.Output()
l1_prog_box, l1_bar, l1_bar_lbl = make_progress("Pulling:")

class _StatusObj:
    value = "⬤  Not loaded"
wl_status = _StatusObj()

def _live_fingerprint_text():
    if LIVE.get("returns") is None:
        return "Live fingerprint: not loaded"
    tags = ", ".join(list(LIVE.get("tags") or [])[:6]) or "none"
    sim_assets = list(LIVE.get("sim_assets") or [])
    usable = 0
    for a in sim_assets:
        s = LIVE.get("returns", {}).get(a)
        if s is not None and len(s.dropna()):
            usable += 1
    z = LIVE.get("trigger_pctile")
    z_txt = f"{z:+.2f}" if isinstance(z, (int, float)) else "n/a"
    trigger = LIVE.get("trigger")
    trig_txt = f"{trigger:.2f}" if isinstance(trigger, (int, float)) else "n/a"
    return (
        f"Live fingerprint: {LIVE.get('name')} | day0 {LIVE.get('date')} | "
        f"{TRIGGER_ASSET} {trig_txt} | z {z_txt} | CPI {LIVE.get('cpi')} | Fed {LIVE.get('fed')} | "
        f"tags [{tags}] | sim assets {usable}/{len(sim_assets)}"
    )

def _update_live_fingerprint():
    wl_fp_lbl.value = (
        "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
        + _live_fingerprint_text()
        + "</span>"
    )

def _l1_setup_check_text():
    issues = []
    try:
        pd.Timestamp(wl_date.value)
    except Exception:
        issues.append("bad date")
    tag_n = len(tuple(wl_tags.value or ()))
    sim_n = len(tuple(wl_sim_assets.value or ()))
    if tag_n == 0:
        issues.append("no tags")
    elif tag_n == 1:
        issues.append("only 1 tag")
    if sim_n < 3:
        issues.append("few sim assets")
    trig_ok = isinstance(getattr(wl_oil, "value", None), (int, float)) and float(getattr(wl_oil, "value", 0.0)) > 0
    if not trig_ok:
        issues.append("trigger missing")
    status = "ready" if not issues else "watch"
    issue_txt = ", ".join(issues) if issues else "none"
    return (
        f"Setup checks: {status} | tags {tag_n} | sim assets {sim_n} | "
        f"trigger {'ok' if trig_ok else 'missing'} | warnings: {issue_txt}"
    )

def _update_l1_setup_check(*_):
    wl_setup_lbl.value = (
        "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
        + _l1_setup_check_text()
        + "</span>"
    )

def _l1_btn_click(_=None):
    _update_l1_setup_check()
    loading_start(l1_bar, l1_bar_lbl, "Pulling live data…")
    wl_refresh_btn.disabled = True
    try:
        pull_live_data()
        _update_l1_setup_check()
        _update_live_fingerprint()
        _wl0_refresh()
        # Reset real day_n tracker to what was just pulled
        global _live_real_day_n
        _live_real_day_n = LIVE.get("day_n")

        # Apply override immediately if checkbox is active,
        # so all subsequent chart renders use the correct day_n
        if wl_day_override_chk.value:
            _max_off = max(
                (max(s.index) for s in LIVE["returns"].values() if len(s)),
                default=_live_real_day_n or 0)
            _ov = max(0, min(int(wl_day_override_n.value), _max_off))
            LIVE["day_n"] = _ov
            wl_day_override_lbl.value = f"trading days from Day 0  (real = Day+{_live_real_day_n})"
            _base = wl_status.value.split("  |  Day+")[0] if "  |  Day+" in wl_status.value else wl_status.value
            wl_status_lbl.value = f"{_base}  |  Day+{_ov} [OVERRIDE — real Day+{_live_real_day_n}]"
        else:
            wl_status_lbl.value = wl_status.value

        # ── Main dashboard charts that embed LIVE data ───────────
        loading_done(l1_bar, l1_bar_lbl, "↻ Overlay chart (T1)…")
        t1_refresh()

        loading_done(l1_bar, l1_bar_lbl, "↻ Heatmap (T2)…")
        t2_refresh()

        loading_done(l1_bar, l1_bar_lbl, "↻ VIX chart (T5)…")
        out_fig(build_vix_fig(), t5_out)

        # ── Live analogue tabs: re-run if previously computed ────
        if _scores_cache:
            loading_done(l1_bar, l1_bar_lbl, "↻ Re-scoring analogues (L2)…")
            wl2_run_btn_click()

        if LIVE.get("returns") and wl3_ast.value:
            loading_done(l1_bar, l1_bar_lbl, "↻ Live Path (L3)…")
            wl3_refresh()

        if _shared_rows:
            loading_done(l1_bar, l1_bar_lbl, "↻ Trade Ideas (L4)…")
            wl4_refresh()

        if _shared_rows and wl5_ast.value and wl5_ast.value not in ("(no data)", ""):
            loading_done(l1_bar, l1_bar_lbl, "↻ Detail (L5)…")
            wl5_update_detail()

        loading_done(l1_bar, l1_bar_lbl, "✅ All tabs refreshed")
    except Exception as e:
        loading_error(l1_bar, l1_bar_lbl, f"❌ {e}")
    finally:
        wl_refresh_btn.disabled = False

wl_refresh_btn.on_click(_l1_btn_click)
for _w in [wl_name, wl_date, wl_tags, wl_sim_assets, wl_cpi, wl_fed]:
    _w.observe(_update_l1_setup_check, names="value")
_update_l1_setup_check()

tab_l1 = W.VBox([
    W.HTML("<h3 style='color:#e6edf3;margin:6px 0'>⬤  Live Event Configuration</h3>"),
    W.HBox([
        W.VBox([wl_name, wl_date, W.HBox([wl_oil_lbl, wti_refresh_btn]), wl_cpi, wl_fed]),
        W.VBox([wl_tags, wl_sim_assets])
    ]),
    W.HTML("<hr style='border-color:#30363d;margin:6px 0'>"),
    W.HBox([wl_day_override_chk, wl_day_override_n, wl_day_override_lbl]),
    W.HBox([wl_refresh_btn, wl_status_lbl]),
    wl_setup_lbl,
    wl_fp_lbl,
    l1_prog_box, wl_out_l1
])

# ══════════════════════════════════════════════════════════════


In [ ]:
# TAB L2 — ANALOGUE MATCHING
# ══════════════════════════════════════════════════════════════
# wl2_cut, _wl2_wq/wt/wm, wl2_run, wl2_out, _scores_cache defined in §5.2.
# This tab wraps them into a layout with progress bar.
l2_prog_box, l2_bar, l2_bar_lbl = make_progress("Scoring:")
wl2_top_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Top analogues: waiting</span>"
)
wl2_fit_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Match fit: waiting</span>"
)
wl2_set_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Selected set: waiting</span>"
)
wl2_diag_out = W.Output()

def _fmt_top_analogues(sc, n=5):
    if not sc:
        return "Top analogues: none"
    bits = []
    for row in list(sc)[:n]:
        nm = str(row.get("event", ""))
        score = row.get("composite", row.get("score", 0.0))
        bits.append(f"{nm} ({score:.2f})")
    return "Top analogues: " + " | ".join(bits)

def _wl2_driver(row):
    dims = {
        "Path": float(row.get("quant", 0.0)),
        "Tag": float(row.get("tag", 0.0)),
        "Macro": float(row.get("macro", 0.0)),
    }
    return max(dims.items(), key=lambda kv: kv[1])[0]

def _wl2_match_summary(sc):
    if not sc:
        return "Match fit: none"
    top = sc[0]
    cutoff = float(wl2_cut.value)
    selected = sum(1 for r in sc if float(r.get("composite", 0.0)) >= cutoff)
    avg_top = np.mean([float(r.get("composite", 0.0)) for r in sc[: min(5, len(sc))]])
    driver = _wl2_driver(top)
    return (
        f"Match fit: best {top.get('event')} | score {float(top.get('composite', 0.0)):.2f} | "
        f"driver {driver} | selected {selected}/{len(sc)} | avg top5 {avg_top:.2f}"
    )

def _wl2_diag_df(sc, n=8):
    rows = []
    zmap = globals().get("TRIGGER_PERCENTILE_MAP", {}) or {}
    for rank, row in enumerate(list(sc)[:n], 1):
        en = row.get("event")
        mc = MACRO_CONTEXT.get(en, {})
        tags = ", ".join(list(EVENT_TAGS.get(en, set()))[:4]) or "—"
        z = zmap.get(en)
        ztxt = f"{float(z):+.2f}" if isinstance(z, (int, float, np.floating)) else "—"
        rows.append({
            "Rank": rank,
            "Event": en,
            "Score": round(float(row.get("composite", 0.0)), 3),
            "Driver": _wl2_driver(row),
            "PathQ": round(float(row.get("quant", 0.0)), 3),
            "Tag": round(float(row.get("tag", 0.0)), 3),
            "Macro": round(float(row.get("macro", 0.0)), 3),
            "CPI": mc.get("cpi", "—"),
            "Fed": mc.get("fed", "—"),
            "Trigger z": ztxt,
            "Tags": tags,
        })
    if not rows:
        return None
    return pd.DataFrame(rows, columns=["Rank", "Event", "Score", "Driver", "PathQ", "Tag", "Macro", "CPI", "Fed", "Trigger z", "Tags"])

def _wl2_selected_set_summary(sc, cutoff):
    if not sc:
        return "Selected set: none"
    emap = {name: dt for name, dt in EVENTS}
    selected = [row for row in sc if float(row.get("composite", 0.0)) >= float(cutoff)]
    if not selected:
        selected = list(sc[: min(3, len(sc))])
    years = []
    cpi_counts = {}
    fed_counts = {}
    tag_counts = {}
    avg = {"Path": [], "Tag": [], "Macro": []}
    for row in selected:
        en = row.get("event")
        dt = emap.get(en)
        if dt is not None:
            try:
                years.append(pd.Timestamp(dt).year)
            except Exception:
                pass
        mc = MACRO_CONTEXT.get(en, {})
        cpi = mc.get("cpi", "—")
        fed = mc.get("fed", "—")
        cpi_counts[cpi] = cpi_counts.get(cpi, 0) + 1
        fed_counts[fed] = fed_counts.get(fed, 0) + 1
        for t in EVENT_TAGS.get(en, set()):
            tag_counts[t] = tag_counts.get(t, 0) + 1
        avg["Path"].append(float(row.get("quant", 0.0)))
        avg["Tag"].append(float(row.get("tag", 0.0)))
        avg["Macro"].append(float(row.get("macro", 0.0)))
    year_span = f"{min(years)}-{max(years)}" if years else "n/a"
    top_tag = ", ".join(k for k, _ in sorted(tag_counts.items(), key=lambda kv: (-kv[1], kv[0]))[:3]) or "none"
    top_cpi = ", ".join(f"{k} {v}" for k, v in sorted(cpi_counts.items(), key=lambda kv: (-kv[1], kv[0]))[:2])
    top_fed = ", ".join(f"{k} {v}" for k, v in sorted(fed_counts.items(), key=lambda kv: (-kv[1], kv[0]))[:2])
    avg_bits = " | ".join(f"{k} {np.mean(v):.2f}" for k, v in avg.items() if v)
    return (
        f"Selected set: {len(selected)} events | years {year_span} | tags {top_tag} | "
        f"CPI {top_cpi} | Fed {top_fed} | avg {avg_bits}"
    )

def wl2_run_btn_click(_=None):
    loading_start(l2_bar, l2_bar_lbl, "Scoring analogues…")
    wl2_run.disabled = True
    try:
        sc = run_analogue_match()
        if sc is None:
            loading_error(l2_bar, l2_bar_lbl, "❌ Run L1 first"); return
        if len(sc) == 0:
            loading_error(l2_bar, l2_bar_lbl,
                "❌ No sim assets at this day offset — try a smaller override day"); return
        _scores_cache.clear(); _scores_cache.extend(sc)
        _wl0_refresh()
        wl2_top_lbl.value = (
            "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
            + _fmt_top_analogues(sc, 5)
            + "</span>"
        )
        wl2_fit_lbl.value = (
            "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
            + _wl2_match_summary(sc)
            + "</span>"
        )
        wl2_set_lbl.value = (
            "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
            + _wl2_selected_set_summary(sc, wl2_cut.value)
            + "</span>"
        )
        with wl2_diag_out:
            clear_output(wait=True)
            _diag_df = _wl2_diag_df(sc, 8)
            if _diag_df is not None and len(_diag_df):
                display(W.HTML("<b style='color:#8b949e;font-family:monospace'>Match Diagnostics</b>"))
                display(_diag_df)
        with wl2_out:
            clear_output(wait=True)
            _rank_fig(sc).show()
            _score_tbl(sc, wl2_cut.value).show()
        loading_done(l2_bar, l2_bar_lbl, f"✅ {len(sc)} events scored")
    except Exception as e:
        loading_error(l2_bar, l2_bar_lbl, f"❌ {e}")
    finally:
        wl2_run.disabled = False

# wl2_cut.observe already wired in §5.2 (_wl2_cut)
wl2_run.on_click(wl2_run_btn_click)

tab_l2 = W.VBox([
    W.HTML("<h3 style='color:#e6edf3;margin:6px 0'>🔍  Analogue Matching Engine</h3>"),
    W.HTML("<span style='color:#8b949e;font-size:10px;font-family:monospace'>"
           "Factor weights auto-renormalised. Re-run after changing weights or cutoff.</span>"),
    W.HBox([_wl2_wq, _wl2_wt]),
    W.HBox([_wl2_wm, _wl2_wtot_lbl]),
    W.HTML("<hr style='border-color:#30363d;margin:4px 0'>"),
    W.HBox([wl2_run, wl2_cut]),
    wl2_top_lbl,
    wl2_fit_lbl,
    wl2_set_lbl,
    wl2_diag_out,
    l2_prog_box, wl2_out
])

# ══════════════════════════════════════════════════════════════


In [ ]:
# TAB L3 — LIVE PATH vs ANALOGUES
# ══════════════════════════════════════════════════════════════
wl3_cls  = make_cls_dd()
wl3_ast  = make_asset_dd()
wl3_prev = W.Button(description="◀", layout=W.Layout(width="40px"))
wl3_next = W.Button(description="▶", layout=W.Layout(width="40px"))
# wl3_cut mirrors wl2_cut — changing either syncs the other
wl3_cut  = W.FloatSlider(value=wl2_cut.value, min=0.0, max=1.0, step=0.01,
    description="Score cutoff:", continuous_update=False,
    layout=W.Layout(width="300px"), style={"description_width":"95px"})

def _wl3_cut_sync(change):
    if abs(wl2_cut.value - change["new"]) > 1e-9:
        wl2_cut.value = change["new"]

wl3_cut.observe(_wl3_cut_sync, "value")
wl3_grp  = W.Dropdown(options=group_opts, value="Risk Barometer",
    description="Fwd Group:",
    layout=W.Layout(width="280px"), style={"description_width":"80px"})
wl3_run_btn = W.Button(description="▶  Run", button_style="primary",
    layout=W.Layout(width="100px"))
wl3_gap_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Live gap monitor: waiting</span>"
)
wl3_gap_out = W.Output()
wl3_out  = W.Output()
l3_prog_box, l3_bar, l3_bar_lbl = make_progress("Loading:")
update_assets(wl3_cls, wl3_ast)
wl3_state = make_refresh_state()

def _wl3_live_gap_df(labels, cutoff, topn=8):
    if LIVE.get("returns") is None:
        return None
    dn = LIVE.get("day_n") or 0
    sel = _sel_ev(cutoff)
    rows = []
    for lbl in labels:
        lr = LIVE["returns"].get(lbl)
        if lr is None or len(lr.dropna()) == 0:
            continue
        try:
            live_v = poi_live_ret(lbl, dn)
        except Exception:
            continue
        hist_vals = [poi_ret(lbl, en, dn) for en in sel]
        hist_vals = [v for v in hist_vals if pd.notna(v)]
        if not hist_vals:
            continue
        med = float(np.median(hist_vals))
        gap = float(live_v - med)
        same = 0.0
        if live_v != 0:
            same = sum(1 for v in hist_vals if np.sign(v) == np.sign(live_v)) / len(hist_vals)
        rows.append({
            "Asset": display_label(lbl),
            "Live": round(float(live_v), 2),
            "Analogue Med": round(med, 2),
            "Gap": round(gap, 2),
            "Same Sign %": round(100.0 * same, 0),
            "_abs_gap": abs(gap),
        })
    if not rows:
        return None
    rows = sorted(rows, key=lambda r: (-r["_abs_gap"], r["Asset"]))[:topn]
    for r in rows:
        r.pop("_abs_gap", None)
    return pd.DataFrame(rows, columns=["Asset", "Live", "Analogue Med", "Gap", "Same Sign %"])

def _wl3_gap_summary(df):
    if df is None or len(df) == 0:
        return "Live gap monitor: none"
    top = df.iloc[0].to_dict()
    leaders = []
    for _, rec in df.head(3).iterrows():
        sign = "+" if float(rec["Gap"]) >= 0 else ""
        leaders.append(f"{rec['Asset']} {sign}{float(rec['Gap']):.2f}")
    return (
        f"Live gap monitor: largest divergence {top['Asset']} ({float(top['Gap']):+.2f}) | "
        f"top gaps " + " | ".join(leaders)
    )

def wl3_refresh(_=None):
    if LIVE["returns"] is None:
        with wl3_out: clear_output(wait=True); print("❌ Run L1 first")
        return
    key = (wl3_cls.value, wl3_ast.value, wl3_grp.value, float(wl3_cut.value), LIVE.get("name"), LIVE.get("day_n"))

    def _render():
        loading_start(l3_bar, l3_bar_lbl, "Building charts…")
        wl3_run_btn.disabled = True
        try:
            lb = get_group_labels(wl3_grp.value)
            with wl3_out:
                clear_output(wait=True)
                if wl3_ast.value:
                    _overlay_fig(wl3_ast.value, wl3_cut.value).show()
                fh = _fwd_heatmap(lb, wl3_cut.value)
                if fh:
                    fh.show()
            _gap_df = _wl3_live_gap_df(lb, wl3_cut.value, topn=8)
            wl3_gap_lbl.value = (
                "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
                + _wl3_gap_summary(_gap_df)
                + "</span>"
            )
            with wl3_gap_out:
                clear_output(wait=True)
                if _gap_df is not None and len(_gap_df):
                    display(W.HTML("<b style='color:#8b949e;font-family:monospace'>Live vs Analogue Gap Table</b>"))
                    display(_gap_df)
            _wl0_refresh()
            loading_done(l3_bar, l3_bar_lbl, "✅ Done")
        except Exception as e:
            loading_error(l3_bar, l3_bar_lbl, f"❌ {e}")
        finally:
            wl3_run_btn.disabled = False

    run_guarded_refresh(wl3_state, key, _render, label="Live paths")


def wl3_prev_click(_):
    opts = list(wl3_ast.options)
    if opts:
        wl3_ast.value = opts[(opts.index(wl3_ast.value)-1) % len(opts)]


def wl3_next_click(_):
    opts = list(wl3_ast.options)
    if opts:
        wl3_ast.value = opts[(opts.index(wl3_ast.value)+1) % len(opts)]


def wl3_run_click(_):
    wl3_state["last"] = None
    wl3_refresh()


wl3_cls.observe(lambda c: update_assets(wl3_cls, wl3_ast) if c.get("name") == "value" else None, "value")
wl3_run_btn.on_click(wl3_run_click)
wl3_prev.on_click(wl3_prev_click)
wl3_next.on_click(wl3_next_click)
wl3_state["ready"] = True

tab_l3 = W.VBox([
    W.HTML("<h3 style='color:#e6edf3;margin:6px 0'>📈  Live Path vs Analogues</h3>"),
    W.HBox([wl3_cls, wl3_ast, wl3_prev, wl3_next]),
    W.HBox([wl3_grp, wl3_run_btn, wl3_cut]),
    wl3_gap_lbl,
    wl3_gap_out,
    l3_prog_box, wl3_out
])

# ══════════════════════════════════════════════════════════════


In [ ]:
# TAB L4 — TRADE IDEAS
# ══════════════════════════════════════════════════════════════
wl4_grp     = make_group_dd("Group:", "— All Assets —")
# Horizon: trading days forward from current live day (dn) — not from Day 0
wl4_hor_days = W.BoundedIntText(value=21, min=1, max=252, step=1,
    description="Horizon (days from now):",
    layout=W.Layout(width="260px"), style={"description_width":"165px"})
wl4_hor_lbl = W.Label(value="", layout=W.Layout(width="200px"))
wl4_hor_1w = W.Button(description="1W", layout=W.Layout(width="50px"))
wl4_hor_1m = W.Button(description="1M", layout=W.Layout(width="50px"))
wl4_hor_3m = W.Button(description="3M", layout=W.Layout(width="50px"))
wl4_hor_now = W.Label(value="presets:", layout=W.Layout(width="52px"))
# Min coverage — canonical here, mirrored to L5
wl4_min_cov = W.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.05,
    description="Min Coverage:", continuous_update=False,
    layout=W.Layout(width="340px"), style={"description_width":"105px"})
# Cutoff: read from wl2_cut (canonical L2/L3 widget)
wl4_dir = W.Dropdown(
    options=[("All", "all"), ("Longs", "LONG"), ("Shorts", "SHORT")],
    value="all", description="Direction:",
    layout=W.Layout(width="200px"), style={"description_width":"70px"})
wl4_topn = W.BoundedIntText(
    value=10, min=3, max=30, step=1, description="Top N:",
    layout=W.Layout(width="130px"), style={"description_width":"45px"})
wl4_run_btn = W.Button(description="▶  Run", button_style="primary",
    layout=W.Layout(width="100px"))
wl4_cache_lbl = W.Label(
    value="cache: waiting",
    layout=W.Layout(width="260px"))
wl4_quick_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Quick take: waiting</span>"
)
wl4_conf_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Confidence mix: waiting</span>"
)
wl4_class_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Class mix: waiting</span>"
)
wl4_radar_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Trade radar: waiting</span>"
)
wl4_consensus_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Consensus shortlist: waiting</span>"
)
wl4_save_btn = W.Button(description="Save Snapshot", layout=W.Layout(width="140px"))
wl4_save_lbl = W.Label(value="snapshot: not saved", layout=W.Layout(width="430px"))
wl4_matrix_out = W.Output()
wl4_board_out = W.Output()
wl4_memo_out = W.Output()
wl4_out     = W.Output()
l4_prog_box, l4_bar, l4_bar_lbl = make_progress("Computing:")
wl4_state = make_refresh_state()

wl4_select_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
    "Select an asset to pre-load in Detail tab (L5):</span>"
)
wl4_select = W.Select(options=[], rows=8, layout=W.Layout(width="480px"))

def _update_hor_label():
    dn = LIVE.get("day_n") or 0
    wl4_hor_lbl.value = f"→ Day+{dn + wl4_hor_days.value} abs from Day 0"

def _wl4_quick_take(rows):
    if not rows:
        return "Quick take: no ideas"
    top = rows[0]
    hit = top.get("hit_rate")
    hit_s = f"{hit*100:.0f}%" if isinstance(hit, (int, float)) else "n/a"
    med = top.get("med")
    med_s = f"{med:+.1f}{top.get('unit','')}" if isinstance(med, (int, float)) else "n/a"
    return (
        f"Quick take: {display_label(top.get('lbl'))} | {top.get('dir')} | "
        f"med {med_s} | hit {hit_s} | conf {top.get('data_conf','low')}"
    )

def _wl4_conf_mix(rows):
    if not rows:
        return "Confidence mix: none"
    counts = {"high": 0, "medium": 0, "low": 0}
    for r in rows:
        counts[r.get("data_conf", "low")] = counts.get(r.get("data_conf", "low"), 0) + 1
    return f"Confidence mix: high {counts.get('high',0)} | med {counts.get('medium',0)} | low {counts.get('low',0)}"

def _wl4_class_mix(rows, top_n=4):
    if not rows:
        return "Class mix: none"
    counts = {}
    for r in rows:
        cls = r.get("cls") or "Other"
        counts[cls] = counts.get(cls, 0) + 1
    ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    parts = [f"{k} {v}" for k, v in ranked[:top_n]]
    return "Class mix: " + " | ".join(parts)

def _wl4_horizon_rows(labels, cutoff, min_cov, direction):
    pack = {}
    for _tag, _days in [("1W", 5), ("1M", 21), ("3M", 63)]:
        _rows = _compute_rows(labels, cutoff, _days)
        if direction != "all":
            _rows = [r for r in _rows if r.get("dir") == direction]
        _main = [r for r in _rows if r["n_total"] == 0 or r["n"]/r["n_total"] >= min_cov]
        pack[_tag] = _main if _main else _rows
    return pack

def _wl4_trade_radar(labels, cutoff, min_cov, direction):
    pack = _wl4_horizon_rows(labels, cutoff, min_cov, direction)
    head = []
    seen = {}
    for _tag in ["1W", "1M", "3M"]:
        rows = pack.get(_tag, [])
        if not rows:
            head.append(f"{_tag}: none")
            continue
        top = rows[0]
        nm = display_label(top["lbl"])
        seen[nm] = seen.get(nm, 0) + 1
        head.append(f"{_tag}: {nm} {top['dir']} ({top.get('data_conf','low')})")
    repeaters = [nm for nm, cnt in sorted(seen.items(), key=lambda kv: (-kv[1], kv[0])) if cnt >= 2]
    tail = "Repeaters: " + (" | ".join(repeaters[:5]) if repeaters else "none")
    return "Trade radar: " + " || ".join(head) + " || " + tail

def _wl4_consensus_shortlist(labels, cutoff, min_cov, direction):
    pack = _wl4_horizon_rows(labels, cutoff, min_cov, direction)
    hits = {}
    for _tag, rows in pack.items():
        for rank, row in enumerate(rows[:5], 1):
            key = (display_label(row["lbl"]), row["dir"])
            item = hits.setdefault(key, {"count": 0, "best_rank": 99, "tags": []})
            item["count"] += 1
            item["best_rank"] = min(item["best_rank"], rank)
            item["tags"].append(_tag)
    ranked = sorted(
        hits.items(),
        key=lambda kv: (-kv[1]["count"], kv[1]["best_rank"], kv[0][0], kv[0][1])
    )
    keep = []
    for (name, dirn), meta in ranked:
        if meta["count"] < 2:
            continue
        keep.append(f"{name} {dirn} ({'/'.join(meta['tags'])})")
        if len(keep) >= 5:
            break
    return "Consensus shortlist: " + (" | ".join(keep) if keep else "none")

def _wl4_snapshot_rows():
    if not _shared_rows:
        return []
    min_cov = wl4_min_cov.value
    main_rows = [r for r in _shared_rows if r["n_total"] == 0 or r["n"] / r["n_total"] >= min_cov]
    weak_rows = [r for r in _shared_rows if r["n_total"] > 0 and r["n"] / r["n_total"] < min_cov]
    topn = int(wl4_topn.value)
    return (main_rows[:topn] + weak_rows[:topn])

def _wl4_consensus_matrix(labels, cutoff, min_cov, direction, topn):
    pack = _wl4_horizon_rows(labels, cutoff, min_cov, direction)
    book = {}
    for _tag in ["1W", "1M", "3M"]:
        rows = (pack.get(_tag, []) or [])[:topn]
        for rank, row in enumerate(rows, 1):
            key = (display_label(row["lbl"]), row["dir"])
            item = book.setdefault(
                key,
                {
                    "Asset": display_label(row["lbl"]),
                    "Dir": row["dir"],
                    "Class": row.get("cls") or "Other",
                    "1W": "",
                    "1M": "",
                    "3M": "",
                    "Hits": 0,
                    "BestRank": 99,
                },
            )
            item[_tag] = f"r{rank} {row.get('data_conf','low')}"
            item["Hits"] += 1
            item["BestRank"] = min(item["BestRank"], rank)
    rows = sorted(book.values(), key=lambda r: (-r["Hits"], r["BestRank"], r["Asset"], r["Dir"]))
    rows = [r for r in rows if r["Hits"] >= 2][:12]
    if not rows:
        return None
    return pd.DataFrame(rows, columns=["Asset", "Dir", "Class", "1W", "1M", "3M", "Hits"])

def _wl4_action_board(rows, labels, cutoff, min_cov, direction, topn):
    longs = [r for r in rows if r.get("dir") == "LONG"]
    shorts = [r for r in rows if r.get("dir") == "SHORT"]
    matrix = _wl4_consensus_matrix(labels, cutoff, min_cov, direction, topn)
    board = []
    if longs:
        r = longs[0]
        board.append({
            "Slot": "Best Long",
            "Asset": display_label(r["lbl"]),
            "Dir": r["dir"],
            "Class": r.get("cls") or "Other",
            "Score": round(float(r.get("score", 0.0)), 3),
            "Conf": r.get("data_conf", "low"),
        })
    if shorts:
        r = shorts[0]
        board.append({
            "Slot": "Best Short",
            "Asset": display_label(r["lbl"]),
            "Dir": r["dir"],
            "Class": r.get("cls") or "Other",
            "Score": round(float(r.get("score", 0.0)), 3),
            "Conf": r.get("data_conf", "low"),
        })
    if matrix is not None and len(matrix):
        for idx, rec in matrix.head(2).iterrows():
            board.append({
                "Slot": f"Consensus {len(board)-1}",
                "Asset": rec["Asset"],
                "Dir": rec["Dir"],
                "Class": rec["Class"],
                "Score": rec["Hits"],
                "Conf": "multi-horizon",
            })
    if not board:
        return None
    return pd.DataFrame(board, columns=["Slot", "Asset", "Dir", "Class", "Score", "Conf"])

def _wl4_action_memo(rows, labels, cutoff, min_cov, direction, topn):
    if not rows:
        return "No trade ideas available for the current setup."
    longs = [r for r in rows if r.get("dir") == "LONG"]
    shorts = [r for r in rows if r.get("dir") == "SHORT"]
    best_long = longs[0] if longs else None
    best_short = shorts[0] if shorts else None
    consensus = _wl4_consensus_shortlist(labels, cutoff, min_cov, direction).replace("Consensus shortlist: ", "")
    conf_mix = _wl4_conf_mix(rows).replace("Confidence mix: ", "")
    class_mix = _wl4_class_mix(rows).replace("Class mix: ", "")
    lines = [
        f"Event: {LIVE.get('name')} | Day 0: {LIVE.get('date')} | Eval Day+: {LIVE.get('day_n')}",
        f"Mode: group={wl4_grp.value} | dir={wl4_dir.value} | horizon={int(wl4_hor_days.value)}d | top_n={int(wl4_topn.value)}",
    ]
    if best_long:
        lines.append(
            f"Best long: {display_label(best_long['lbl'])} | {best_long.get('data_conf','low')} | "
            f"med {best_long.get('med', 0):+.1f}{best_long.get('unit','')} | hit {best_long.get('hit_rate',0)*100:.0f}%"
        )
    if best_short:
        lines.append(
            f"Best short: {display_label(best_short['lbl'])} | {best_short.get('data_conf','low')} | "
            f"med {best_short.get('med', 0):+.1f}{best_short.get('unit','')} | hit {best_short.get('hit_rate',0)*100:.0f}%"
        )
    lines.append(f"Consensus: {consensus}")
    lines.append(f"Confidence mix: {conf_mix}")
    lines.append(f"Class mix: {class_mix}")
    if any(r.get("data_conf") == "low" for r in rows[: max(3, min(10, len(rows)))]):
        lines.append("Caveat: top set includes low-confidence names, so favour repeaters and high-confidence classes first.")
    else:
        lines.append("Caveat: idea set is relatively well-supported, but still compare with path/context before acting.")
    return "\n".join(lines)

def _wl4_save_snapshot(_=None):
    try:
        rows = _wl4_snapshot_rows()
        if not rows:
            wl4_save_lbl.value = "snapshot: nothing to save"
            return
        out_dir = Path.cwd() / "_runtime" / "live_snapshots"
        out_dir.mkdir(parents=True, exist_ok=True)
        safe_name = str(LIVE.get("name") or "live_event").replace(" ", "_").replace("/", "-")
        stamp = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")
        out_path = out_dir / f"{safe_name}_{stamp}.csv"
        df = pd.DataFrame([{
            "event_name": LIVE.get("name"),
            "event_date": LIVE.get("date"),
            "day_n": LIVE.get("day_n"),
            "group": wl4_grp.value,
            "direction_filter": wl4_dir.value,
            "top_n": int(wl4_topn.value),
            "horizon_days": int(wl4_hor_days.value),
            "cutoff": float(wl2_cut.value),
            "min_cov": float(wl4_min_cov.value),
            "asset": r.get("lbl"),
            "display": display_label(r.get("lbl")),
            "class": r.get("cls"),
            "direction": r.get("dir"),
            "median": r.get("med"),
            "hit_rate": r.get("hit_rate"),
            "score": r.get("score"),
            "data_conf": r.get("data_conf"),
            "n": r.get("n"),
            "n_total": r.get("n_total"),
        } for r in rows])
        df.to_csv(out_path, index=False)
        wl4_save_lbl.value = f"snapshot: {out_path.name}"
    except Exception as e:
        wl4_save_lbl.value = f"snapshot: failed ({e})"

def _set_wl4_horizon(days, rerun=True):
    wl4_hor_days.value = int(days)
    _update_hor_label()
    if rerun and LIVE["returns"] is not None:
        wl4_state["last"] = None
        wl4_refresh()


def wl4_refresh(_=None):
    if LIVE["returns"] is None:
        with wl4_out: clear_output(wait=True); print("❌ Run L1 first")
        return
    key = (wl4_grp.value, wl4_dir.value, int(wl4_topn.value), int(wl4_hor_days.value), float(wl4_min_cov.value), float(wl2_cut.value), LIVE.get("name"), LIVE.get("day_n"))

    def _render():
        _update_hor_label()
        loading_start(l4_bar, l4_bar_lbl, "Computing trade ideas…")
        wl4_run_btn.disabled = True
        try:
            lb   = get_group_labels(wl4_grp.value)
            rows = _compute_rows(lb, wl2_cut.value, wl4_hor_days.value)
            if wl4_dir.value != "all":
                rows = [r for r in rows if r.get("dir") == wl4_dir.value]
            _shared_rows.clear(); _shared_rows.extend(rows)
            _cache_meta = globals().get("_TRADE_ROWS_CACHE_LAST", {})
            _cache_state = "hit" if _cache_meta.get("hit") else "build"
            wl4_cache_lbl.value = f"cache: {_cache_state} | rows: {len(rows)}"
            main_fig, weak_fig = _summary_table_fig(
                rows, wl4_hor_days.value, wl2_cut.value, min_cov=wl4_min_cov.value)
            with wl4_out:
                clear_output(wait=True)
                if main_fig: main_fig.show()
                if weak_fig:
                    display(W.HTML(
                        "<h4 style='color:#e3b341;margin:12px 0 4px 0;font-family:monospace'>"
                        "⚠  Weak Signal — below coverage threshold</h4>"))
                    weak_fig.show()
            min_cov = wl4_min_cov.value
            main_rows = [r for r in rows if r["n_total"] == 0 or r["n"]/r["n_total"] >= min_cov]
            weak_rows = [r for r in rows if r["n_total"] > 0 and r["n"]/r["n_total"] < min_cov]
            topn = int(wl4_topn.value)
            main_rows = main_rows[:topn]
            weak_rows = weak_rows[:topn]
            opts = (
                [f"{i+1:>2}. {display_label(r['lbl'])} — {r['dir']}  ({r['hit_rate']*100:.0f}% hit | med {r['med']:+.1f}{r['unit']})"
                 for i, r in enumerate(main_rows)]
                + (["── Weak Signal ──"] if weak_rows else [])
                + [f"{len(main_rows)+i+1:>2}. {display_label(r['lbl'])} — {r['dir']}  [{r['n']}/{r['n_total']} analogues]"
                   for i, r in enumerate(weak_rows)]
            )
            wl4_select.options = opts if opts else ["(no results)"]
            if opts:
                wl4_select.value = opts[0]
            wl4_quick_lbl.value = (
                "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
                + _wl4_quick_take(main_rows if main_rows else rows)
                + "</span>"
            )
            wl4_conf_lbl.value = (
                "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
                + _wl4_conf_mix(main_rows if main_rows else rows)
                + "</span>"
            )
            wl4_class_lbl.value = (
                "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
                + _wl4_class_mix(main_rows if main_rows else rows)
                + "</span>"
            )
            wl4_radar_lbl.value = (
                "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
                + _wl4_trade_radar(lb, wl2_cut.value, wl4_min_cov.value, wl4_dir.value)
                + "</span>"
            )
            wl4_consensus_lbl.value = (
                "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
                + _wl4_consensus_shortlist(lb, wl2_cut.value, wl4_min_cov.value, wl4_dir.value)
                + "</span>"
            )
            _matrix_df = _wl4_consensus_matrix(lb, wl2_cut.value, wl4_min_cov.value, wl4_dir.value, int(wl4_topn.value))
            _board_df = _wl4_action_board(main_rows if main_rows else rows, lb, wl2_cut.value, wl4_min_cov.value, wl4_dir.value, int(wl4_topn.value))
            with wl4_board_out:
                clear_output(wait=True)
                if _board_df is not None and len(_board_df):
                    display(W.HTML("<b style='color:#8b949e;font-family:monospace'>Action Board</b>"))
                    display(_board_df)
            with wl4_memo_out:
                clear_output(wait=True)
                display(W.HTML("<b style='color:#8b949e;font-family:monospace'>Action Memo</b>"))
                print(_wl4_action_memo(main_rows if main_rows else rows, lb, wl2_cut.value, wl4_min_cov.value, wl4_dir.value, int(wl4_topn.value)))
            with wl4_matrix_out:
                clear_output(wait=True)
                if _matrix_df is not None and len(_matrix_df):
                    display(W.HTML("<b style='color:#8b949e;font-family:monospace'>Consensus Matrix</b>"))
                    display(_matrix_df)
            _wl0_refresh()
            loading_done(
                l4_bar,
                l4_bar_lbl,
                f"✅ showing {len(main_rows) + len(weak_rows)} ideas | cache: {'hit' if globals().get('_TRADE_ROWS_CACHE_LAST', {}).get('hit') else 'build'}"
            )
        except Exception as e:
            loading_error(l4_bar, l4_bar_lbl, f"❌ {e}")
        finally:
            wl4_run_btn.disabled = False

    run_guarded_refresh(wl4_state, key, _render, label="Live trade ideas")


def _wl4_on_select(change):
    val = change.get("new", "")
    if not val or val == "(no results)" or val.startswith("──"):
        return
    try:
        rank = int(val.strip().split(".")[0].strip()) - 1
    except (ValueError, IndexError):
        return
    if rank < 0 or rank >= len(_shared_rows):
        return
    row = _shared_rows[rank]
    lbl = row["lbl"]
    opts_l5 = [display_label(r["lbl"]) + " — " + r["dir"] for r in _shared_rows]
    wl5_ast.options = opts_l5 if opts_l5 else ["(no data)"]
    target = display_label(lbl) + " — " + row["dir"]
    if target in wl5_ast.options:
        wl5_ast.value = target


def wl4_run_click(_):
    wl4_state["last"] = None
    wl4_refresh()


wl4_select.observe(_wl4_on_select, names="value")
wl4_run_btn.on_click(wl4_run_click)
wl4_save_btn.on_click(_wl4_save_snapshot)
wl4_hor_1w.on_click(lambda _: _set_wl4_horizon(5))
wl4_hor_1m.on_click(lambda _: _set_wl4_horizon(21))
wl4_hor_3m.on_click(lambda _: _set_wl4_horizon(63))
wl4_hor_days.observe(lambda c: _update_hor_label() if c.get("name") == "value" else None, "value")
_update_hor_label()
wl4_state["ready"] = True

tab_l4 = W.VBox([
    W.HTML("<h3 style='color:#e6edf3;margin:6px 0'>💡  Trade Ideas</h3>"),
    W.HTML("<span style='color:#8b949e;font-size:10px;font-family:monospace'>"
           "Cutoff: set in L2. Horizon = trading days forward from today (current live day).</span>"),
    W.HBox([wl4_grp, wl4_hor_days, wl4_hor_lbl]),
    W.HBox([wl4_hor_now, wl4_hor_1w, wl4_hor_1m, wl4_hor_3m]),
    W.HBox([wl4_dir, wl4_topn, wl4_min_cov, wl4_run_btn, wl4_cache_lbl]),
    wl4_quick_lbl,
    wl4_conf_lbl,
    wl4_class_lbl,
    wl4_radar_lbl,
    wl4_consensus_lbl,
    W.HBox([wl4_save_btn, wl4_save_lbl]),
    wl4_board_out,
    wl4_memo_out,
    wl4_matrix_out,
    l4_prog_box,
    wl4_out,
    W.HTML("<hr style='border-color:#30363d;margin:8px 0'>"),
    wl4_select_lbl,
    wl4_select,
])

# ══════════════════════════════════════════════════════════════


In [ ]:
# TAB L0 — LIVE BRIEF
# ══════════════════════════════════════════════════════════════
wl0_hdr = W.HTML("<h3 style='color:#e6edf3;margin:6px 0'>🧭  Live Brief</h3>")
wl0_sub = W.HTML(
    "<span style='color:#8b949e;font-size:10px;font-family:monospace'>"
    "One-page event brief that reuses live config, analogue fit, path divergence, and top trade context."
    "</span>"
)
wl0_summary = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Live brief: waiting</span>"
)
wl0_verdict = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Regime verdict: waiting</span>"
)
wl0_focus_mode_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Focus mode: waiting</span>"
)
wl0_priority_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Priority signals: waiting</span>"
)
wl0_mode = W.ToggleButtons(
    options=[("Compact", "compact"), ("Detail", "detail")],
    value="compact",
    description="Brief:",
    layout=W.Layout(width="240px"),
    style={"description_width":"45px"},
)
wl0_next_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Next best view: waiting</span>"
)
wl0_runbook_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Runbook: waiting</span>"
)
wl0_progress_lbl = W.HTML(
    "<span style='color:#8b949e;font-size:11px;font-family:monospace'>Checklist: waiting</span>"
)
wl0_state = {"opened_trade": False, "opened_gap": False}
wl0_trade_btn = W.Button(description="Open Top Trade", layout=W.Layout(width="140px"))
wl0_gap_btn = W.Button(description="Open Top Gap", layout=W.Layout(width="140px"))
wl0_focus_lbl = W.Label(value="focus: waiting", layout=W.Layout(width="520px"))
wl0_out = W.Output()

def _wl0_find_label(display_name):
    for _lbl in ALL_LABELS:
        if display_label(_lbl) == display_name or _lbl == display_name:
            return _lbl
    return None

def _wl0_find_class(lbl):
    for _cls, _assets in CLASS_TO_ASSETS.items():
        if lbl in _assets:
            return _cls
    return None

def _wl0_focus_trade(_=None):
    if not _shared_rows:
        wl0_focus_lbl.value = "focus: run L4 first"
        return
    min_cov = wl4_min_cov.value if "wl4_min_cov" in globals() else 0.0
    rows = [r for r in _shared_rows if r["n_total"] == 0 or r["n"] / r["n_total"] >= min_cov]
    rows = rows if rows else list(_shared_rows)
    row = rows[0]
    target = display_label(row["lbl"]) + " — " + row["dir"]
    wl5_update_table()
    opts = list(wl5_ast.options or [])
    if target in opts:
        wl5_ast.value = target
        wl5_update_detail()
    if "dashboard" in globals():
        dashboard.selected_index = 13
    wl0_state["opened_trade"] = True
    _wl0_refresh()
    wl0_focus_lbl.value = f"focus: detail -> {display_label(row['lbl'])} {row['dir']}"

def _wl0_focus_gap(_=None):
    if LIVE.get("returns") is None:
        wl0_focus_lbl.value = "focus: run L1 first"
        return
    lb = get_group_labels(wl3_grp.value)
    gap_df = _wl3_live_gap_df(lb, wl3_cut.value, topn=8)
    if gap_df is None or len(gap_df) == 0:
        lb = get_group_labels("— All Assets —")
        gap_df = _wl3_live_gap_df(lb, wl3_cut.value, topn=8)
    top = None
    focus_note = "gap"
    if gap_df is not None and len(gap_df):
        top = gap_df.iloc[0].to_dict()
    else:
        dn = LIVE.get("day_n") or 0
        fallback = []
        for _lbl in list(LIVE.get("sim_assets") or []):
            try:
                _v = poi_live_ret(_lbl, dn)
            except Exception:
                continue
            if pd.notna(_v):
                fallback.append({"Asset": display_label(_lbl), "Gap": float(_v), "_lbl": _lbl})
        if not fallback:
            wl0_focus_lbl.value = "focus: no gap candidates"
            return
        fallback = sorted(fallback, key=lambda r: (-abs(r["Gap"]), r["Asset"]))
        top = fallback[0]
        focus_note = "live move"
    lbl = top.get("_lbl") or _wl0_find_label(top["Asset"])
    if lbl is None:
        wl0_focus_lbl.value = f"focus: could not resolve {top['Asset']}"
        return
    cls = _wl0_find_class(lbl)
    if cls in wl3_cls.options:
        wl3_cls.value = cls
        update_assets(wl3_cls, wl3_ast)
    if lbl in list(wl3_ast.options or []):
        wl3_ast.value = lbl
    wl3_refresh()
    if "dashboard" in globals():
        dashboard.selected_index = 11
    wl0_state["opened_gap"] = True
    _wl0_refresh()
    wl0_focus_lbl.value = f"focus: paths -> {top['Asset']} {focus_note} {float(top['Gap']):+.2f}"

def _wl0_verdict_text():
    if LIVE.get("returns") is None or not _scores_cache:
        return "Regime verdict: waiting for live pull and analogue match"
    top_score = float(_scores_cache[0].get("composite", 0.0))
    lb = get_group_labels(wl3_grp.value) if "wl3_grp" in globals() else []
    gap_df = _wl3_live_gap_df(lb, wl3_cut.value, topn=5) if lb else None
    avg_gap = None
    avg_sign = None
    if gap_df is not None and len(gap_df):
        avg_gap = float(np.mean(np.abs(gap_df["Gap"].astype(float))))
        avg_sign = float(np.mean(gap_df["Same Sign %"].astype(float)))
    if top_score >= 0.75 and (avg_gap is None or avg_gap <= 1.5):
        verdict = "aligned"
    elif top_score < 0.55 or (avg_gap is not None and avg_gap >= 4.0):
        verdict = "breaking away"
    else:
        verdict = "mixed"
    bits = [f"Regime verdict: {verdict}", f"top analogue {top_score:.2f}"]
    if avg_gap is not None:
        bits.append(f"avg gap {avg_gap:.2f}")
    if avg_sign is not None:
        bits.append(f"same-sign {avg_sign:.0f}%")
    return " | ".join(bits)

def _wl0_focus_mode_text():
    tags = set(LIVE.get("tags") or [])
    name = str(LIVE.get("name") or "").lower()
    if not tags and not name:
        return "Focus mode: generic | inspect analogues, divergence, and shortlist evenly"
    macro_hits = {"cpi", "pce", "nfp", "fomc", "inflation", "jobs", "payrolls", "fed"}
    election_hits = {"election", "midterm", "presidential", "vote", "senate", "house"}
    geo_hits = {"war", "conflict", "military_conflict", "energy_shock", "sanctions", "oil", "shipping_disruption"}
    macro_score = sum(1 for t in tags if t in macro_hits) + sum(k in name for k in ["cpi", "pce", "nfp", "fomc", "inflation", "payroll"])
    election_score = sum(1 for t in tags if t in election_hits) + sum(k in name for k in ["election", "midterm", "presidential", "vote"])
    geo_score = sum(1 for t in tags if t in geo_hits) + sum(k in name for k in ["war", "conflict", "iran", "israel", "oil", "embargo", "shipping"])
    if geo_score >= max(macro_score, election_score) and geo_score > 0:
        return "Focus mode: geopolitical | prioritize Paths and cross-asset divergence, then confirm trade breadth"
    if election_score >= max(macro_score, geo_score) and election_score > 0:
        return "Focus mode: election | prioritize analogues and cross-market agreement before drilling into one trade"
    if macro_score > 0:
        return "Focus mode: macro release | prioritize analogue quality and trigger-sensitive trades before path detail"
    return "Focus mode: generic | inspect analogues, divergence, and shortlist evenly"

def _wl0_priority_text():
    mode = _wl0_focus_mode_text().split(" | ")[0].replace("Focus mode: ", "")
    bits = []
    if LIVE.get("returns") is not None:
        bits.append(_wl0_verdict_text().replace("Regime verdict: ", "verdict "))
    if _scores_cache:
        bits.append(_wl0_next_best_text().replace("Next best view: ", "next "))
    if mode == "geopolitical":
        lb = get_group_labels(wl3_grp.value) if "wl3_grp" in globals() else []
        gap_df = _wl3_live_gap_df(lb, wl3_cut.value, topn=3) if lb else None
        if gap_df is not None and len(gap_df):
            top = gap_df.iloc[0].to_dict()
            bits.append(f"gap {top['Asset']} {float(top['Gap']):+.2f}")
        if _shared_rows:
            bits.append(_wl4_consensus_shortlist(get_group_labels(wl4_grp.value), wl2_cut.value, wl4_min_cov.value, wl4_dir.value).replace("Consensus shortlist: ", "consensus "))
    elif mode == "macro release":
        if _scores_cache:
            bits.append(_wl2_match_summary(_scores_cache).replace("Match fit: ", "fit "))
            bits.append(_wl2_selected_set_summary(_scores_cache, wl2_cut.value).replace("Selected set: ", "set "))
        if _shared_rows:
            bits.append(_wl4_quick_take(_shared_rows).replace("Quick take: ", "trade "))
    elif mode == "election":
        if _scores_cache:
            bits.append(_wl2_selected_set_summary(_scores_cache, wl2_cut.value).replace("Selected set: ", "set "))
        if _shared_rows:
            bits.append(_wl4_class_mix(_shared_rows).replace("Class mix: ", "class "))
            bits.append(_wl4_consensus_shortlist(get_group_labels(wl4_grp.value), wl2_cut.value, wl4_min_cov.value, wl4_dir.value).replace("Consensus shortlist: ", "consensus "))
    else:
        if _scores_cache:
            bits.append(_wl2_match_summary(_scores_cache).replace("Match fit: ", "fit "))
        if _shared_rows:
            bits.append(_wl4_quick_take(_shared_rows).replace("Quick take: ", "trade "))
    bits = [b for b in bits if b][:3]
    return "Priority signals: " + (" || ".join(bits) if bits else "waiting")

def _wl0_next_best_text():
    if LIVE.get("returns") is None:
        return "Next best view: run Config first"
    if not _scores_cache:
        return "Next best view: run Analogues to establish the historical map"
    top_score = float(_scores_cache[0].get("composite", 0.0))
    gap_df = None
    if "wl3_grp" in globals():
        lb = get_group_labels(wl3_grp.value)
        gap_df = _wl3_live_gap_df(lb, wl3_cut.value, topn=5)
    avg_gap = None
    if gap_df is not None and len(gap_df):
        avg_gap = float(np.mean(np.abs(gap_df["Gap"].astype(float))))
    if not _shared_rows:
        return "Next best view: Trade Ideas — analogues exist, but the opportunity set has not been built yet"
    min_cov = wl4_min_cov.value if "wl4_min_cov" in globals() else 0.0
    main_rows = [r for r in _shared_rows if r["n_total"] == 0 or r["n"] / r["n_total"] >= min_cov]
    use_rows = main_rows if main_rows else list(_shared_rows)
    top_conf = use_rows[0].get("data_conf", "low") if use_rows else "low"
    if avg_gap is not None and avg_gap >= 4.0:
        return "Next best view: Paths — live behaviour is diverging materially from the analogue basket"
    if top_score < 0.55:
        return "Next best view: Analogues — fit is weak, so inspect the match set before acting"
    if top_conf == "high":
        return "Next best view: Detail — fit is solid and top trade quality is strong"
    return "Next best view: Trade Ideas — fit is usable, but compare the shortlist before drilling into one asset"

def _wl0_runbook_text():
    if LIVE.get("returns") is None:
        return "Runbook: 1. Load live data | 2. Score analogues | 3. Build trade shortlist"
    steps = []
    if not _scores_cache:
        steps.append("1. Run Analogues to establish the historical map")
        steps.append("2. Review match diagnostics before trusting any trade output")
        steps.append("3. Build trade ideas only after analogue quality looks sensible")
        return "Runbook: " + " || ".join(steps)
    top_score = float(_scores_cache[0].get("composite", 0.0))
    gap_df = None
    avg_gap = None
    if "wl3_grp" in globals():
        lb = get_group_labels(wl3_grp.value)
        gap_df = _wl3_live_gap_df(lb, wl3_cut.value, topn=5)
    if gap_df is not None and len(gap_df):
        avg_gap = float(np.mean(np.abs(gap_df["Gap"].astype(float))))
    if top_score < 0.55:
        steps.append("1. Check Analogues — fit is weak, so historical mapping is the main risk")
    elif avg_gap is not None and avg_gap >= 4.0:
        steps.append("1. Check Paths — live event is moving away from the analogue basket")
    else:
        steps.append("1. Check Trade Ideas — the historical map is coherent enough to rank opportunities")
    if not _shared_rows:
        steps.append("2. Build the trade shortlist to see whether the event expresses cleanly")
        steps.append("3. Use Detail only after a strong candidate emerges")
        return "Runbook: " + " || ".join(steps)
    min_cov = wl4_min_cov.value if "wl4_min_cov" in globals() else 0.0
    main_rows = [r for r in _shared_rows if r["n_total"] == 0 or r["n"] / r["n_total"] >= min_cov]
    use_rows = main_rows if main_rows else list(_shared_rows)
    top_conf = use_rows[0].get("data_conf", "low") if use_rows else "low"
    if top_conf == "high":
        steps.append("2. Open Detail on the best trade to inspect path, decay, and deviation")
    else:
        steps.append("2. Compare shortlist breadth and confidence before drilling into one asset")
    steps.append("3. Save a snapshot if the setup looks decision-ready")
    return "Runbook: " + " || ".join(steps)

def _wl0_progress_text():
    bits = []
    bits.append(("Live", LIVE.get("returns") is not None))
    bits.append(("Analogues", bool(_scores_cache)))
    bits.append(("Paths", bool(wl0_state.get("opened_gap")) or ("wl3_gap_lbl" in globals() and "largest divergence" in getattr(wl3_gap_lbl, "value", ""))))
    bits.append(("Trades", bool(_shared_rows)))
    bits.append(("Detail", bool(wl0_state.get("opened_trade")) or ("wl5_ast" in globals() and bool(getattr(wl5_ast, "value", None)) and getattr(wl5_ast, "value", None) != "(no data)")))
    parts = [f"{name} {'done' if ok else 'todo'}" for name, ok in bits]
    return "Checklist: " + " | ".join(parts)

def _wl0_refresh_visibility():
    compact = wl0_mode.value == "compact"
    wl0_summary.layout.display = "" if compact else "none"
    wl0_next_lbl.layout.display = "" if compact else "none"
    wl0_progress_lbl.layout.display = "none" if compact else ""
    wl0_runbook_lbl.layout.display = "none" if compact else ""
    wl0_focus_mode_lbl.layout.display = "none" if compact else ""

def _wl0_refresh():
    parts = []
    if LIVE.get("returns") is None:
        parts.append("Run L1 to populate the live brief.")
    else:
        parts.append(f"Event {LIVE.get('name')} | day0 {LIVE.get('date')} | eval Day+{LIVE.get('day_n')}")
        parts.append(_live_fingerprint_text())
    if _scores_cache:
        parts.append(_wl2_match_summary(_scores_cache))
        if "wl2_set_lbl" in globals():
            parts.append(_wl2_selected_set_summary(_scores_cache, wl2_cut.value))
    if "wl3_gap_lbl" in globals() and _scores_cache and LIVE.get("returns") is not None:
        lb = get_group_labels(wl3_grp.value)
        _gap_df = _wl3_live_gap_df(lb, wl3_cut.value, topn=3)
        parts.append(_wl3_gap_summary(_gap_df))
    if _shared_rows:
        min_cov = wl4_min_cov.value if "wl4_min_cov" in globals() else 0.0
        main_rows = [r for r in _shared_rows if r["n_total"] == 0 or r["n"] / r["n_total"] >= min_cov]
        use_rows = main_rows if main_rows else _shared_rows
        parts.append(_wl4_quick_take(use_rows))
        parts.append(_wl4_consensus_shortlist(get_group_labels(wl4_grp.value), wl2_cut.value, min_cov, wl4_dir.value))
    wl0_summary.value = (
        "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
        + " || ".join(parts[:6])
        + "</span>"
    )
    wl0_verdict.value = (
        "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
        + _wl0_verdict_text()
        + "</span>"
    )
    wl0_focus_mode_lbl.value = (
        "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
        + _wl0_focus_mode_text()
        + "</span>"
    )
    wl0_priority_lbl.value = (
        "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
        + _wl0_priority_text()
        + "</span>"
    )
    wl0_next_lbl.value = (
        "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
        + _wl0_next_best_text()
        + "</span>"
    )
    wl0_runbook_lbl.value = (
        "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
        + _wl0_runbook_text()
        + "</span>"
    )
    wl0_progress_lbl.value = (
        "<span style='color:#8b949e;font-size:11px;font-family:monospace'>"
        + _wl0_progress_text()
        + "</span>"
    )
    with wl0_out:
        clear_output(wait=True)
        if _shared_rows:
            min_cov = wl4_min_cov.value if "wl4_min_cov" in globals() else 0.0
            main_rows = [r for r in _shared_rows if r["n_total"] == 0 or r["n"] / r["n_total"] >= min_cov]
            use_rows = (main_rows if main_rows else _shared_rows)[:5]
            brief_df = pd.DataFrame([
                {
                    "Asset": display_label(r["lbl"]),
                    "Dir": r["dir"],
                    "Class": r.get("cls") or "Other",
                    "Hit %": round(float(r.get("hit_rate", 0.0)) * 100.0, 0),
                    "Median": round(float(r.get("med", 0.0)), 2),
                    "Conf": r.get("data_conf", "low"),
                }
                for r in use_rows
            ], columns=["Asset", "Dir", "Class", "Hit %", "Median", "Conf"])
            display(W.HTML("<b style='color:#8b949e;font-family:monospace'>Current shortlist</b>"))
            display(brief_df)

wl0_trade_btn.on_click(_wl0_focus_trade)
wl0_gap_btn.on_click(_wl0_focus_gap)

wl0_mode.observe(lambda c: _wl0_refresh_visibility() if c.get("name") == "value" else None, names="value")

tab_l0 = W.VBox([
    wl0_hdr,
    wl0_sub,
    W.HBox([wl0_verdict, wl0_mode]),
    wl0_focus_mode_lbl,
    wl0_priority_lbl,
    wl0_next_lbl,
    wl0_progress_lbl,
    wl0_runbook_lbl,
    wl0_summary,
    W.HBox([wl0_trade_btn, wl0_gap_btn, wl0_focus_lbl]),
    wl0_out,
])
_wl0_refresh_visibility()

# ══════════════════════════════════════════════════════════════


In [ ]:
# TAB L5 — DETAILED ANALYSIS
# ══════════════════════════════════════════════════════════════
wl5_ast     = W.Dropdown(description="Asset:",
    layout=W.Layout(width="420px"), style={"description_width":"55px"})
wl5_run_btn = W.Button(description="▶  Run", button_style="primary",
    layout=W.Layout(width="100px"))
# wl5_min_cov mirrors wl4_min_cov — changing either syncs the other
wl5_min_cov = W.FloatSlider(value=wl4_min_cov.value, min=0.0, max=1.0, step=0.05,
    description="Min Coverage:", continuous_update=False,
    layout=W.Layout(width="340px"), style={"description_width":"105px"})
wl5_tbl_state = make_refresh_state()
wl5_det_state = make_refresh_state()

def _wl5_mc_sync(change):
    if abs(wl4_min_cov.value - change["new"]) > 1e-9:
        wl4_min_cov.value = change["new"]


def _wl4_mc_sync(change):
    if abs(wl5_min_cov.value - change["new"]) > 1e-9:
        wl5_min_cov.value = change["new"]


wl5_min_cov.observe(_wl5_mc_sync, "value")
wl4_min_cov.observe(_wl4_mc_sync, "value")

wl5_out_writeup = W.Output()
wl5_out_tbl     = W.Output()
wl5_out_det     = W.Output()
l5_prog_box, l5_bar, l5_bar_lbl = make_progress("Loading:")

def wl5_update_table(_=None):
    if LIVE["returns"] is None:
        with wl5_out_tbl: clear_output(wait=True); print("❌ Run L1 first")
        return
    if not _shared_rows:
        with wl5_out_tbl: clear_output(wait=True); print("⚠️  Run Live — Quick first")
        return
    key = (len(_shared_rows), int(wl4_hor_days.value), float(wl2_cut.value), float(wl4_min_cov.value))

    def _render():
        loading_start(l5_bar, l5_bar_lbl, "Loading table…")
        wl5_run_btn.disabled = True
        try:
            min_cov = wl4_min_cov.value
            cov_pass = [r for r in _shared_rows if r["n_total"] == 0 or r["n"] / r["n_total"] >= min_cov]
            cov_weak = [r for r in _shared_rows if r["n_total"] > 0 and r["n"] / r["n_total"] < min_cov]
            ordered  = cov_pass + cov_weak
            opts = [display_label(r["lbl"]) + " — " + r["dir"] for r in ordered]
            if not wl5_ast.options or list(wl5_ast.options) != opts:
                wl5_ast.options = opts if opts else ["(no data)"]
                if opts:
                    wl5_ast.value = opts[0]
            _cache_meta = globals().get("_TRADE_ROWS_CACHE_LAST", {})
            wl4_cache_lbl.value = f"cache: {'hit' if _cache_meta.get('hit') else 'build'} | rows: {len(_shared_rows)}"
            main_fig, weak_fig = _summary_table_fig(
                _shared_rows, wl4_hor_days.value, wl2_cut.value, min_cov=wl4_min_cov.value)
            with wl5_out_tbl:
                clear_output(wait=True)
                if main_fig: main_fig.show()
                if weak_fig:
                    display(W.HTML(
                        "<h4 style='color:#e3b341;margin:12px 0 4px 0;font-family:monospace'>"
                        "⚠  Weak Signal — below coverage threshold</h4>"))
                    weak_fig.show()
            loading_done(l5_bar, l5_bar_lbl, f"✅ {len(_shared_rows)} ideas")
        except Exception as e:
            loading_error(l5_bar, l5_bar_lbl, f"❌ {e}")
        finally:
            wl5_run_btn.disabled = False

    run_guarded_refresh(wl5_tbl_state, key, _render, label="Live detail table")


def wl5_update_detail(*_):
    if not _shared_rows or not wl5_ast.value or wl5_ast.value == "(no data)":
        return
    sel_display = wl5_ast.value.split(" — ")[0].strip()
    row = next((r for r in _shared_rows if display_label(r["lbl"]) == sel_display), None)
    if row is None:
        return
    lbl = row["lbl"]
    sel = _sel_ev(wl2_cut.value)
    key = (wl5_ast.value, len(_shared_rows), int(wl4_hor_days.value), float(wl2_cut.value))

    def _render():
        with wl5_out_writeup:
            clear_output(wait=True)
            display(W.HTML(
                f"<h4 style='color:#ffa657;margin:6px 0'>"
                f"▶ {display_label(lbl)}  |  {row['dir']}  |  "
                f"{row['status']}  |  {row['stars']}</h4>"
            ))
            wf = _writeup_fig(lbl, row, sel, wl2_cut.value)
            if wf:
                wf.show()

        with wl5_out_det:
            clear_output(wait=True)
            for fig in [
                _decay_fig(lbl, sel, wl2_cut.value),
                _dot_plot_fig(lbl, sel, wl4_hor_days.value, wl2_cut.value),
                _live_deviation_fig(lbl, sel, wl2_cut.value),
            ]:
                if fig:
                    fig.show()

    run_guarded_refresh(wl5_det_state, key, _render, label="Live detail charts")


def wl5_run_click(_):
    wl5_tbl_state["last"] = None
    wl5_update_table()


wl5_run_btn.on_click(wl5_run_click)
wl5_ast.observe(lambda c: wl5_update_detail() if c.get("name") == "value" else None, "value")
wl5_tbl_state["ready"] = True
wl5_det_state["ready"] = True

tab_l5 = W.VBox([
    W.HTML("<h3 style='color:#e6edf3;margin:6px 0'>🔬  Detailed Trade Analysis</h3>"),
    W.Label("Run L4 first. Select asset in L4 selector or use dropdown below."),
    W.HTML("<span style='color:#8b949e;font-size:10px;font-family:monospace'>"
           "Cutoff: set in L2. Min Coverage synced with L4.</span>"),
    W.HBox([wl5_run_btn, wl5_ast]),
    wl5_min_cov,
    l5_prog_box,
    W.HTML("<b style='color:#c9d1d9;font-family:monospace'>📊 Asset Detail Charts</b>"),
    wl5_out_det,
    W.HTML("<hr style='border-color:#30363d;margin:8px 0'>"),
    wl5_out_writeup,
    W.HTML("<hr style='border-color:#30363d;margin:8px 0'>"),
    W.HTML("<b style='color:#8b949e;font-family:monospace'>📋 Full Trade Ideas Table</b>"),
    wl5_out_tbl,
])

# ══════════════════════════════════════════════════════════════


In [ ]:
# §6.11 — L6: SIGNAL SCREENER (A1 Conviction + A3 Bimodal + A4 Redundancy)
# ─────────────────────────────────────────────────────────────
import ipywidgets as W6

wl6_out     = W6.Output()
wl6_fwd     = W6.BoundedIntText(value=21, min=1, max=252, step=1,
    description="Horizon (days):", layout=W6.Layout(width="230px"),
    style={"description_width":"110px"})
wl6_min_hit = W6.FloatSlider(value=0.60, min=0.0, max=1.0, step=0.05,
    description="Min Hit%:", continuous_update=False,
    layout=W6.Layout(width="360px"), style={"description_width":"80px"})
wl6_min_cov = W6.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.05,
    description="Min Cov:", continuous_update=False,
    layout=W6.Layout(width="360px"), style={"description_width":"80px"})
wl6_min_rr  = W6.FloatSlider(value=0.80, min=0.0, max=5.0, step=0.10,
    description="Min R/R:", continuous_update=False,
    layout=W6.Layout(width="360px"), style={"description_width":"80px"})
wl6_grp     = W6.Dropdown(
    options=["All Assets"] + sorted(CUSTOM_GROUPS.keys()),
    value="All Assets", description="Asset group:",
    layout=W6.Layout(width="340px"), style={"description_width":"100px"})
wl6_run     = W6.Button(description="▶  Run Screener", button_style="primary",
    layout=W6.Layout(width="180px"))
_screener_cache = []

def _l6_run(*_):
    global _screener_cache
    if wl6_state["busy"]:
        return
    wl6_state["busy"] = True
    wl6_run.disabled = True
    try:
        if not _check_live():
            with wl6_out:
                clear_output(wait=True)
                print("⚠️  Run L1 first to pull live data.")
            return
        grp = wl6_grp.value
        lbls = (CUSTOM_GROUPS.get(grp, ALL_LABELS)
                if grp != "All Assets" else ALL_LABELS)
        lbls = [l for l in lbls if l in ALL_LABELS]
        rows = _screener_rows(lbls, wl2_cut.value, wl6_fwd.value,
                              min_hit=wl6_min_hit.value,
                              min_cov=wl6_min_cov.value,
                              min_rr=wl6_min_rr.value)
        _screener_cache = rows
        with wl6_out:
            clear_output(wait=True)
            fig = _screener_fig(rows, wl6_fwd.value, wl2_cut.value)
            if fig: fig.show()
            else: print("No assets passed minimum coverage threshold.")
    finally:
        wl6_run.disabled = False
        wl6_state["busy"] = False

wl6_run.on_click(_l6_run)

tab_l6 = W6.VBox([
    W6.HTML("<h3 style='color:#e6edf3;margin:6px 0 2px'>Signal Screener"
            " <span style='font-size:12px;color:#8b949e'>"
            " A1 conviction · A3 bimodal · A4 redundancy</span></h3>"),
    W6.HBox([wl6_grp, wl6_fwd, wl6_run]),
    W6.HBox([wl6_min_hit, wl6_min_cov, wl6_min_rr]),
    wl6_out,
])


In [ ]:
# §6.12 — L7: LEAD-LAG & TIMING (C3)
# ─────────────────────────────────────────────────────────────
import ipywidgets as W7

wl7_out    = W7.Output()
wl7_grp    = W7.Dropdown(
    options=["Risk Barometer","Safe Havens","Oil & Energy",
             "FX G10","FX EM","Sector ETFs","All Assets"]
            + sorted(CUSTOM_GROUPS.keys()),
    value="Risk Barometer", description="Asset group:",
    layout=W7.Layout(width="340px"), style={"description_width":"100px"})
wl7_daily  = W7.Checkbox(value=False, description="Daily offsets (slow)",
    layout=W7.Layout(width="220px"))
wl7_run    = W7.Button(description="▶  Build Matrix", button_style="primary",
    layout=W7.Layout(width="180px"))

def _l7_run(*_):
    if wl7_state["busy"]:
        return
    wl7_state["busy"] = True
    wl7_run.disabled = True
    try:
        if not _check_live():
            with wl7_out:
                clear_output(wait=True)
                print("⚠️  Run L1 first.")
            return
        _check_scores()
        sel  = _sel_ev(wl2_cut.value)
        grp  = wl7_grp.value
        lbls = (CUSTOM_GROUPS.get(grp, ALL_LABELS)
                if grp != "All Assets" else ALL_LABELS)
        lbls = [l for l in lbls if l in ALL_LABELS][:25]
        offsets = None
        if wl7_daily.value:
            dn = LIVE.get("day_n", 0)
            offsets = list(range(0, min(dn+1, POST_WINDOW_TD+1)))
        with wl7_out:
            clear_output(wait=True)
            if len(lbls) < 2:
                print("Need at least 2 assets. Choose a group with more assets.")
                return
            suffix = " (daily)" if wl7_daily.value else " (POIS)"
            fig = _leadlag_fig(lbls, sel, offsets, title_suffix=f" · {grp}{suffix}")
            if fig: fig.show()
    finally:
        wl7_run.disabled = False
        wl7_state["busy"] = False

wl7_run.on_click(_l7_run)

tab_l7 = W7.VBox([
    W7.HTML("<h3 style='color:#e6edf3;margin:6px 0 2px'>Lead-Lag & Timing"
            " <span style='font-size:12px;color:#8b949e'>"
            " Which asset moves first?</span></h3>"),
    W7.HBox([wl7_grp, wl7_daily, wl7_run]),
    wl7_out,
])


In [ ]:
# §6.13 — L8: REVERSE ANALOGUE LOOKUP (B3)
# ─────────────────────────────────────────────────────────────
import ipywidgets as W8

wl8_out   = W8.Output()
wl8_topn  = W8.BoundedIntText(value=10, min=3, max=len(EVENTS),
    description="Show top N:", layout=W8.Layout(width="200px"),
    style={"description_width":"90px"})
wl8_run   = W8.Button(description="▶  Run Reverse Lookup",
    button_style="primary", layout=W8.Layout(width="220px"))

def _l8_run(*_):
    if wl8_state["busy"]:
        return
    wl8_state["busy"] = True
    wl8_run.disabled = True
    try:
        with wl8_out:
            clear_output(wait=True)
            scores = run_reverse_lookup()
            if not scores:
                print("⚠️  Run L1 first."); return
            fig = _reverse_fig(scores, top_n=wl8_topn.value)
            if fig: fig.show()
            print(f"\nTop {wl8_topn.value} by pure path similarity (no event label used):")
            print(f"{'Event':<40} {'PathQ':>7} {'CPI':>6} {'Fed':>8} {'Tag sim':>8}")
            print("-"*75)
            for s in scores[:wl8_topn.value]:
                t = _tag_sim(LIVE.get("tags",[]), EVENT_TAGS.get(s["event"],set()))
                print(f"{s['event']:<40} {s['path_quant']:>7.3f}" f" {s['macro_cpi']:>6} {s['macro_fed']:>8} {t:>8.3f}")
    finally:
        wl8_run.disabled = False
        wl8_state["busy"] = False

wl8_run.on_click(_l8_run)

tab_l8 = W8.VBox([
    W8.HTML("<h3 style='color:#e6edf3;margin:6px 0 2px'>Reverse Pattern Lookup"
            " <span style='font-size:12px;color:#8b949e'>"
            " Pattern-first · no event label · pure cosine path match</span></h3>"),
    W8.HBox([wl8_topn, wl8_run]),
    wl8_out,
])


In [ ]:
# §6.14 — L9: PRE-POSITIONING PLAYBOOK (D1)
# ─────────────────────────────────────────────────────────────
import ipywidgets as W9

wl9_out    = W9.Output()
wl9_pre    = W9.BoundedIntText(value=10, min=1, max=PRE_WINDOW_TD,
    description="Pre-event days:", layout=W9.Layout(width="230px"),
    style={"description_width":"120px"})
wl9_grp    = W9.Dropdown(
    options=["All Assets"] + sorted(CUSTOM_GROUPS.keys()),
    value="Risk Barometer", description="Asset group:",
    layout=W9.Layout(width="340px"), style={"description_width":"100px"})
wl9_view   = W9.ToggleButtons(options=["Bar Chart","Heatmap"],
    value="Bar Chart",
    layout=W9.Layout(width="280px"))
wl9_run    = W9.Button(description="▶  Run Pre-Positioning",
    button_style="primary", layout=W9.Layout(width="220px"))

def _l9_run(*_):
    if wl9_state["busy"]:
        return
    wl9_state["busy"] = True
    wl9_run.disabled = True
    try:
        if not _check_live():
            with wl9_out:
                clear_output(wait=True)
                print("⚠️  Run L1 first.")
            return
        _check_scores()
        sel  = _sel_ev(wl2_cut.value)
        grp  = wl9_grp.value
        lbls = (CUSTOM_GROUPS.get(grp, ALL_LABELS) if grp != "All Assets" else ALL_LABELS)
        lbls = [l for l in lbls if l in ALL_LABELS]
        pre  = wl9_pre.value
        with wl9_out:
            clear_output(wait=True)
            if wl9_view.value == "Bar Chart":
                rows = _prepos_rows(lbls, sel, pre)
                fig  = _prepos_fig(rows, sel, pre)
                if fig: fig.show()
                else: print("No data for selected group/window.")
            else:
                fig = _prepos_heatmap(lbls, sel, pre)
                if fig: fig.show()
                else: print("No data for selected group/window.")
    finally:
        wl9_run.disabled = False
        wl9_state["busy"] = False

wl9_run.on_click(_l9_run)

tab_l9 = W9.VBox([
    W9.HTML("<h3 style='color:#e6edf3;margin:6px 0 2px'>Pre-Positioning Playbook"
            " <span style='font-size:12px;color:#8b949e'>"
            " How markets moved BEFORE the shock · no overfit · pure history</span></h3>"),
    W9.HBox([wl9_grp, wl9_pre, wl9_view, wl9_run]),
    wl9_out,
])


In [ ]:
# §6.15 — L10: SECTOR ROTATION (D4)
# ─────────────────────────────────────────────────────────────
# Shows sector ETF performance ranked by analogue-weighted median return.
# Rotation view: which sectors lead, which lag, across pre/post event windows.
# Includes bond ETFs and thematic ETFs for a complete cross-sector picture.

import ipywidgets as W10

_SECTOR_GROUPS = {
    "SPDR Sectors":    ["Comm Services","Materials","Financials","Industrials",
                        "Technology","Cons Staples","Real Estate","Utilities",
                        "Healthcare","Cons Discret"],
    "Spec Sectors":    ["Gold Miners","Oil Services","Regional Banks",
                        "Homebuilders","Innovation (ARKK)"],
    "Bond ETFs":       ["20Y Treasury (TLT)","7-10Y Treasury (IEF)",
                        "1-3Y Treasury (SHY)","TIPS ETF (TIP)","Gold ETF (IAU)"],
    "Thematic ETFs":   ["Airlines (JETS)","Agri (MOO)","Defense (ITA)",
                        "Cyber Security (BUG)","Clean Energy (ICLN)","Nuclear (NLR)"],
    "All Sectors":     ["Comm Services","Materials","Financials","Industrials",
                        "Technology","Cons Staples","Real Estate","Utilities",
                        "Healthcare","Cons Discret","Gold Miners","Oil Services",
                        "Regional Banks","Homebuilders","Innovation (ARKK)",
                        "20Y Treasury (TLT)","7-10Y Treasury (IEF)",
                        "1-3Y Treasury (SHY)","TIPS ETF (TIP)","Gold ETF (IAU)",
                        "Airlines (JETS)","Agri (MOO)","Defense (ITA)",
                        "Cyber Security (BUG)","Clean Energy (ICLN)","Nuclear (NLR)"],
}

wl10_out    = W10.Output()
wl10_grp    = W10.Dropdown(
    options=list(_SECTOR_GROUPS.keys()),
    value="All Sectors", description="Sector group:",
    layout=W10.Layout(width="300px"), style={"description_width":"110px"})
wl10_fwd    = W10.BoundedIntText(value=21, min=1, max=252, step=1,
    description="Horizon (days):", layout=W10.Layout(width="220px"),
    style={"description_width":"110px"})
wl10_view   = W10.ToggleButtons(
    options=["Ranked Bar","Rotation Heatmap","Waterfall"],
    value="Ranked Bar", layout=W10.Layout(width="420px"))
wl10_run    = W10.Button(description="▶  Run Rotation",
    button_style="primary", layout=W10.Layout(width="180px"))
wl10_state  = make_refresh_state()

def _rotation_bar(rows, fwd_days, sel_ev):
    """Ranked horizontal bar chart: sectors sorted by analogue-weighted median."""
    if not rows:
        return None
    rows_s = sorted(rows, key=lambda r: r["med"])
    lbls   = [display_label(r["lbl"])[:22] for r in rows_s]
    meds   = [r["med"] for r in rows_s]
    hits   = [r["hit_rate"] for r in rows_s]
    ns     = [r["n_cov"] for r in rows_s]
    colors = ["rgba(63,185,80,0.75)" if m > 0 else "rgba(248,81,73,0.75)"
              for m in meds]
    fig = go.Figure(go.Bar(
        x=meds, y=lbls, orientation="h",
        text=[f"{v:+.1f}%" for v in meds], textposition="outside",
        marker_color=colors,
        hovertemplate="<b>%{y}</b><br>Median: %{x:+.2f}%"
                      "<br>Hit rate: %{customdata[0]:.0%}  N=%{customdata[1]}<extra></extra>",
        customdata=list(zip(hits, ns)),
    ))
    fig.add_vline(x=0, line=dict(color=COL_ZERO, width=1, dash="dash"))
    fig.update_layout(**dark_layout(
        title=(f"<b>Sector Rotation — Ranked</b> | +{fwd_days}d horizon"
               f" | {len(sel_ev)} analogues"),
        xaxis=dict(title="Median forward return (%)", gridcolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        yaxis=dict(tickfont=dict(color=COL_MUTED, size=10)),
        width=CHART_W, height=max(400, len(rows_s) * 28 + 120),
        margin=dict(l=200, r=120, b=60)))
    return fig

def _rotation_heatmap(lbls_show, sel_ev, poi_offsets=None):
    """Heatmap: sectors × POIS — shows rotation timeline across event lifecycle."""
    if poi_offsets is None:
        poi_offsets = [(lbl, off) for lbl, off in POIS if off >= 0]
    mat, txt, shown = [], [], []
    for lbl in lbls_show:
        if lbl not in ALL_LABELS:
            continue
        row_v, row_t = [], []
        for poi_lbl, off in poi_offsets:
            vals = []
            for en in sel_ev:
                v0 = poi_ret(lbl, en, 0)
                vt = poi_ret(lbl, en, off)
                if not any(np.isnan([v0, vt])):
                    vals.append(vt - v0)
            med = float(np.median(vals)) if vals else np.nan
            row_v.append(med)
            row_t.append(f"{med:+.1f}" if not np.isnan(med) else "")
        mat.append(row_v)
        txt.append(row_t)
        shown.append(display_label(lbl)[:20])
    if not mat:
        return None
    mat = np.array(mat)
    vmax = np.nanpercentile(np.abs(mat), 95) or 3
    x_lbl = [f"{pl} (D{off:+d})" for pl, off in poi_offsets]
    fig = go.Figure(go.Heatmap(
        z=mat, x=x_lbl, y=shown, text=txt, texttemplate="%{text}",
        colorscale=[[0, "#da3633"], [0.5, "#21262d"], [1, "#238636"]],
        zmid=0, zmin=-vmax, zmax=vmax,
        colorbar=dict(title="Return (%)", tickfont=dict(color=COL_MUTED, size=9),
                      titlefont=dict(color=COL_MUTED, size=10)),
        hovertemplate="<b>%{y}</b> at %{x}<br>Median: %{z:+.2f}%<extra></extra>",
    ))
    fig.update_layout(**dark_layout(
        title=f"<b>Sector Rotation Heatmap</b> — {len(sel_ev)} analogues",
        xaxis=dict(tickfont=dict(color=COL_MUTED, size=10)),
        yaxis=dict(tickfont=dict(color=COL_MUTED, size=10)),
        width=CHART_W, height=max(360, len(shown) * 26 + 120),
        margin=dict(l=200, r=80, b=80)))
    return fig

def _rotation_waterfall(rows, fwd_days, sel_ev):
    """Waterfall: each sector's contribution sorted from biggest winner to loser."""
    if not rows:
        return None
    rows_s = sorted(rows, key=lambda r: r["med"], reverse=True)
    lbls = [display_label(r["lbl"])[:22] for r in rows_s]
    meds = [r["med"] for r in rows_s]
    fig = go.Figure(go.Waterfall(
        x=lbls, y=meds, measure=["relative"] * len(meds),
        text=[f"{v:+.1f}%" for v in meds], textposition="outside",
        connector=dict(line=dict(color=COL_GRID, width=0.8)),
        increasing=dict(marker_color="rgba(63,185,80,0.8)"),
        decreasing=dict(marker_color="rgba(248,81,73,0.8)"),
    ))
    fig.update_layout(**dark_layout(
        title=(f"<b>Sector Rotation Waterfall</b> | +{fwd_days}d | "
               f"{len(sel_ev)} analogues"),
        xaxis=dict(tickangle=35, tickfont=dict(color=COL_MUTED, size=9)),
        yaxis=dict(title="Median return (%)", gridcolor=COL_GRID,
                   tickfont=dict(color=COL_MUTED)),
        width=CHART_W, height=460, margin=dict(b=140)))
    return fig

def _l10_forward_rows(lbls, sel, dn, fo):
    rows = []
    for lbl in lbls:
        fwd_vals = []
        for en in sel:
            sv = poi_ret(lbl, en, dn)
            fv = poi_ret(lbl, en, fo)
            if not any(np.isnan([sv, fv])):
                fwd_vals.append(fv - sv)
        if len(fwd_vals) < 2:
            continue
        med = float(np.median(fwd_vals))
        rows.append({
            "lbl": lbl,
            "med": med,
            "hit_rate": float(np.mean([
                1 if (med > 0 and v > 0) or (med < 0 and v < 0) else 0
                for v in fwd_vals
            ])),
            "n_cov": len(fwd_vals),
            "unit": unit_label(lbl),
        })
    return rows

def _l10_render():
    if not _check_live():
        with wl10_out:
            clear_output(wait=True)
            print("⚠️  Run L1 first.")
        return
    _check_scores()
    sel = _sel_ev(wl2_cut.value)
    grp = wl10_grp.value
    lbls = [l for l in _SECTOR_GROUPS.get(grp, []) if l in ALL_LABELS]
    fwd = wl10_fwd.value
    dn = LIVE.get("day_n", 0)
    fo = dn + fwd
    view = wl10_view.value

    with wl10_out:
        clear_output(wait=True)
        if not lbls:
            print("No sector ETFs loaded. Check ASSETS list.")
            return
        if view == "Ranked Bar":
            rows = _l10_forward_rows(lbls, sel, dn, fo)
            fig = _rotation_bar(rows, fwd, sel)
            if fig:
                out_fig(fig)
            else:
                print("No ranked-bar data for selected group/window.")
        elif view == "Rotation Heatmap":
            fig = _rotation_heatmap(lbls, sel)
            if fig:
                out_fig(fig)
            else:
                print("No heatmap data for selected group/window.")
        else:
            rows = _l10_forward_rows(lbls, sel, dn, fo)
            fig = _rotation_waterfall(rows, fwd, sel)
            if fig:
                out_fig(fig)
            else:
                print("No waterfall data for selected group/window.")

def _l10_run(*_):
    run_guarded_refresh(
        wl10_state,
        wl10_out,
        _l10_render,
        button=wl10_run,
        key_fn=lambda: (
            wl10_grp.value,
            wl10_fwd.value,
            wl10_view.value,
            LIVE.get("name"),
            LIVE.get("day_n"),
            len(ALL_LABELS),
            len(_sel_ev(wl2_cut.value)),
        ),
    )

wl10_run.on_click(_l10_run)

tab_l10 = W10.VBox([
    W10.HTML("<h3 style='color:#e6edf3;margin:6px 0 2px'>Sector Rotation"
             " <span style='font-size:12px;color:#8b949e'>"
             " SPDR · Bond ETFs · Thematic — ranked by analogue-weighted return</span></h3>"),
    W10.HBox([wl10_grp, wl10_fwd, wl10_view, wl10_run]),
    wl10_out,
])


In [ ]:
# §6.16 — L11: PORTFOLIO STRESS TESTER (E2)
# ─────────────────────────────────────────────────────────────
# Runs PORTFOLIO_SCENARIOS from §1.6 against selected analogues.
# Each scenario is a separate dict — copy/edit freely in §1.6.
# All scenarios run simultaneously, shown side by side.
# Copy-to-clipboard button exports summary as plain text.

import ipywidgets as W11

wl11_out  = W11.Output()
wl11_fwd  = W11.BoundedIntText(value=21, min=1, max=252, step=1,
    description="Horizon (days):", layout=W11.Layout(width="220px"),
    style={"description_width":"110px"})
wl11_run  = W11.Button(description="▶  Stress All Portfolios",
    button_style="primary", layout=W11.Layout(width="240px"))
wl11_copy = W11.Button(description="📋  Copy Summary",
    button_style="", layout=W11.Layout(width="160px"))
_stress_summary_text = [""]  # mutable container for clipboard

def _l11_run(*_):
    if wl11_state["busy"]:
        return
    wl11_state["busy"] = True
    wl11_run.disabled = True
    try:
        if not _check_live():
            with wl11_out:
                clear_output(wait=True); print("⚠️  Run L1 first."); return
        _check_scores()
        sel  = _sel_ev(wl2_cut.value)
        dn   = LIVE.get("day_n",0)
        fwd  = wl11_fwd.value
        with wl11_out:
            clear_output(wait=True)
            figs = _stress_fig(PORTFOLIO_SCENARIOS, sel, dn, fwd)
            if figs[0]: figs[0].show()
            if figs[1]: figs[1].show()
            lines = [f"Portfolio Stress — Day+{dn} → +{dn+fwd} | {len(sel)} analogues", "="*60]
            for name, portfolio in PORTFOLIO_SCENARIOS.items():
                res = _stress_portfolio(portfolio, sel, dn, fwd)
                lines.append(f"\n{name}")
                lines.append(f"  Median PnL : ${res['med_pnl']/1000:+.1f}k")
                lines.append(f"  Worst case : ${res['worst_pnl']/1000:+.1f}k")
                lines.append(f"  Best case  : ${res['best_pnl']/1000:+.1f}k")
                lines.append(f"  Hit rate   : {res['hit_rate']:.0%}  (N={res['n']})")
            _stress_summary_text[0] = "\n".join(lines)
    finally:
        wl11_run.disabled = False
        wl11_state["busy"] = False

def _l11_copy(*_):
    if not _stress_summary_text[0]:
        return
    # Use JavaScript clipboard API via display
    from IPython.display import display as idisplay, Javascript
    safe = _stress_summary_text[0].replace("`","'").replace("\\","\\\\")
    idisplay(Javascript(
        f"navigator.clipboard.writeText(`{safe}`)"
        ".then(()=>console.log('Copied'))"
        ".catch(e=>console.error('Copy failed',e))"))

wl11_run.on_click(_l11_run)
wl11_copy.on_click(_l11_copy)

tab_l11 = W11.VBox([
    W11.HTML("<h3 style='color:#e6edf3;margin:6px 0 2px'>Portfolio Stress Tester"
             " <span style='font-size:12px;color:#8b949e'>"
             " Edit PORTFOLIO_SCENARIOS in §1.6 to add/modify books</span></h3>"),
    W11.HTML("<div style='color:#8b949e;font-size:11px;margin-bottom:4px'>"
             "Scenarios currently loaded: "
             + ", ".join(f"<b style='color:#e6edf3'>{k}</b>"
                         for k in PORTFOLIO_SCENARIOS) + "</div>"),
    W11.HBox([wl11_fwd, wl11_run, wl11_copy]),
    wl11_out,
])


In [ ]:
# §6.17 — L12: SIGNAL DECAY TRACKER (E3)
# ─────────────────────────────────────────────────────────────
# Shows how the top-3 analogues evolve in rank from Day 0 to Day+N.
# Shaded regions mark dominant analogue at each day offset.
# When the top analogue changes: the event is tracking a different historical path.

import ipywidgets as W12

wl12_out   = W12.Output()
wl12_step  = W12.BoundedIntText(value=1, min=1, max=5, step=1,
    description="Step (days):", layout=W12.Layout(width="200px"),
    style={"description_width":"100px"})
wl12_run   = W12.Button(description="▶  Build Decay Chart",
    button_style="primary", layout=W12.Layout(width="220px"))
wl12_note  = W12.HTML(
    "<div style='color:#8b949e;font-size:11px;margin-top:2px'>"
    "⚠️  Step=1 runs a score at every day offset — may be slow for long events."
    " Use Step=2 or 3 for faster builds.</div>")

def _l12_run(*_):
    if wl12_state["busy"]:
        return
    wl12_state["busy"] = True
    wl12_run.disabled = True
    try:
        with wl12_out:
            clear_output(wait=True)
            if not _check_live():
                print("⚠️  Run L1 first."); return
            dn  = LIVE.get("day_n",0)
            fig = _decay_fig(max_dn=dn, step=wl12_step.value)
            if fig:
                fig.show()
                sc_now = _decay_scores_at(dn)
                print(f"\nRank at Day+{dn}:")
                for rank, s in enumerate(sc_now[:5], 1):
                    print(f"  #{rank}  {s['event']:<40}  {s['path_quant']:.3f}")
            else:
                print("Need at least Day+1 data.")
    finally:
        wl12_run.disabled = False
        wl12_state["busy"] = False

wl12_run.on_click(_l12_run)

tab_l12 = W12.VBox([
    W12.HTML("<h3 style='color:#e6edf3;margin:6px 0 2px'>Signal Decay Tracker"
             " <span style='font-size:12px;color:#8b949e'>"
             " Top analogue rank evolution — Day 0 → today</span></h3>"),
    W12.HBox([wl12_step, wl12_run]),
    wl12_note,
    wl12_out,
])


In [ ]:
# §6.18 — L13: TRADE MEMO (E1)
# ─────────────────────────────────────────────────────────────
# Auto-generates a plain-language trade memo from screener + analogue results.
# Collects: event name, day_n, top analogues, top signals, sector rotation,
# and portfolio stress summary. Formats as copy-pasteable text.

import ipywidgets as W13

wl13_out   = W13.Output()
wl13_fwd   = W13.BoundedIntText(value=21, min=1, max=252, step=1,
    description="Horizon (days):", layout=W13.Layout(width="220px"),
    style={"description_width":"110px"})
wl13_run   = W13.Button(description="▶  Generate Memo",
    button_style="primary", layout=W13.Layout(width="200px"))
wl13_copy  = W13.Button(description="📋  Copy Memo",
    button_style="", layout=W13.Layout(width="160px"))
_memo_text = [""]

def _build_memo(fwd_days):
    if not _check_live(): return "⚠️  Run L1 first."
    dn      = LIVE.get("day_n",0)
    ev_name = LIVE.get("name","Unknown Event")
    sel     = _sel_ev(wl2_cut.value)
    n_sel   = len(sel)
    fo      = dn + fwd_days

    lines = []
    lines.append("=" * 64)
    lines.append(f"TRADE MEMO — {ev_name}")
    lines.append(f"Generated: Day+{dn}  |  Horizon: +{fwd_days}d (Day+{fo})")
    lines.append(f"Analogues used: {n_sel}  |  Score cutoff: {wl2_cut.value:.2f}")
    lines.append("=" * 64)

    # Top analogues
    if _scores_cache:
        lines.append("\nTOP ANALOGUES (composite score)")
        lines.append("-" * 40)
        for i, s in enumerate(_scores_cache[:5], 1):
            ctx = MACRO_CONTEXT.get(s["event"],{})
            lines.append(
                f"  #{i}  {s['event']:<38}  "
                f"Composite={s['composite']:.3f}  "
                f"PathQ={s['quant']:.3f}  PtQ={s.get('quant_pt',s['quant']):.3f}  "
                f"Tag={s['tag']:.3f}  Macro={s['macro']:.3f}"
            )
            lines.append(
                f"       CPI: {ctx.get('cpi','?')}  "
                f"Fed: {ctx.get('fed','?')}  "
                f"Oil: {ctx.get('oil','?')} (z-score)"
            )

    # Top signals from screener (if cached)
    if _screener_cache:
        act = [r for r in _screener_cache if r["conviction"] == "🟢 Act"]
        mon = [r for r in _screener_cache if r["conviction"] == "🟡 Monitor"]
        lines.append("\nTOP SIGNALS (screener)")
        lines.append("-" * 40)
        lines.append(f"  🟢 Act ({len(act)}):  "
                     + "  |  ".join(f"{display_label(r['lbl'])} "
                                    f"{r['direction']} {r['med']:+.1f}{r['unit']} "
                                    f"({r['hit_rate']:.0%})"
                                    for r in act[:6]))
        if mon:
            lines.append(f"  🟡 Monitor ({len(mon)}): "
                         + "  |  ".join(display_label(r["lbl"]) for r in mon[:6]))
        bimodal = [r for r in _screener_cache if r["conviction"]=="⚠️ Bimodal"]
        if bimodal:
            lines.append(f"  ⚠️  Bimodal/skip: "
                         + "  |  ".join(display_label(r["lbl"]) for r in bimodal[:4]))

    # Portfolio stress summary (if available)
    if PORTFOLIO_SCENARIOS:
        lines.append("\nPORTFOLIO STRESS SUMMARY")
        lines.append("-" * 40)
        for name, portfolio in PORTFOLIO_SCENARIOS.items():
            res = _stress_portfolio(portfolio, sel, dn, fwd_days)
            lines.append(
                f"  {name:<30}  "
                f"Med=${res['med_pnl']/1000:+.1f}k  "
                f"Worst=${res['worst_pnl']/1000:+.1f}k  "
                f"Hit={res['hit_rate']:.0%}"
                if not np.isnan(res["med_pnl"]) else f"  {name:<30}  No data"
            )

    lines.append("\n" + "=" * 64)
    lines.append("END OF MEMO")
    return "\n".join(lines)

def _l13_run(*_):
    if wl13_state["busy"]:
        return
    wl13_state["busy"] = True
    wl13_run.disabled = True
    try:
        with wl13_out:
            clear_output(wait=True)
            memo = _build_memo(wl13_fwd.value)
            _memo_text[0] = memo
            print(memo)
    finally:
        wl13_run.disabled = False
        wl13_state["busy"] = False

def _l13_copy(*_):
    if not _memo_text[0]: return
    from IPython.display import display as idisplay, Javascript
    safe = _memo_text[0].replace("`","'").replace("\\","\\\\")
    idisplay(Javascript(
        f"navigator.clipboard.writeText(`{safe}`)"
        ".then(()=>console.log('Memo copied'))"
        ".catch(e=>console.error(e))"))

wl13_run.on_click(_l13_run)
wl13_copy.on_click(_l13_copy)

tab_l13 = W13.VBox([
    W13.HTML("<h3 style='color:#e6edf3;margin:6px 0 2px'>Trade Memo"
             " <span style='font-size:12px;color:#8b949e'>"
             " Auto-generated · runs all available signals · copy to clipboard</span></h3>"),
    W13.HTML("<div style='color:#8b949e;font-size:11px;margin-bottom:4px'>"
             "Run L2 (Analogues), L6 (Screener), L11 (Stress) first for full memo. "
             "Partial results shown if not all tabs have been run.</div>"),
    W13.HBox([wl13_fwd, wl13_run, wl13_copy]),
    wl13_out,
])


In [ ]:
# §6.10 — TAB ASSEMBLY & DISPLAY
# ─────────────────────────────────────────────────────────────
# Assembles all tab_N VBoxes into the W.Tab dashboard and renders it.
# Also fires initial refresh calls so charts are populated on load.

# ASSEMBLE TABS & RENDER
# ══════════════════════════════════════════════════════════════
tab_names = [
    "Events",
    "Overlay", "Heatmap", "Scatter", "VIX",
    "Box & Whisker", "Summary Table", "Step-In",
    "Live — Brief",
    "Live — Config",
    "Live — Analogues",
    "Live — Paths",
    "Live — Quick",
    "Live — Detail",
    "L6 Screener",
    "L7 Lead-Lag",
    "L8 Reverse",
    "L9 Pre-Position",
    "L10 Rotation",
    "L11 Portfolio",
    "L12 Decay",
    "L13 Memo",
]
tab_contents = [
    tab_events,
    tab1, tab2, tab3, tab5, tab6, tab7, tab8,
    tab_l0, tab_l1, tab_l2, tab_l3, tab_l4, tab_l5,
    tab_l6, tab_l7, tab_l8, tab_l9,
    tab_l10, tab_l11, tab_l12, tab_l13,
]

dashboard = W.Tab(children=tab_contents)
for i, name in enumerate(tab_names):
    dashboard.set_title(i, name)

t1_refresh(); t2_refresh()
t6_refresh(); t7_refresh(); t8_refresh()
if "tab_l0" in globals():
    _wl0_refresh()

display(dashboard)
